In [ ]:
import pandas as pd

# 1. JSON 파일 불러오기 (용량이 커서 약간의 시간이 걸릴 수 있습니다)
try:
    df = pd.read_json('CR_meta_data_SI.json', orient='index')
    print("🎉 데이터 불러오기 대성공!")
    print(f"총 확보된 원본 MOF 개수: {len(df)}개")
    display(df.head()) # 표 형태로 예쁘게 앞의 5줄만 보여주기

except FileNotFoundError:
    print("앗! 'CR_meta_data_SI.json' 파일을 찾지 못했습니다. 파일이 현재 폴더에 들어있는지 목록을 다시 확인해 주세요.")

In [ ]:
import pprint

# 1. 원본 데이터(df)에서 첫 번째(0번째) MOF 데이터 하나만 통째로 끄집어내어 사전(Dictionary)으로 만듭니다.
# (다른 걸 보고 싶다면 iloc[0] 대신 iloc[10], iloc[100] 등으로 숫자를 바꿔보세요!)
sample_mof = df.iloc[0].to_dict()

print(f"🔬 === [{sample_mof.get('id', '이름 없음')}] MOF 정밀 해부 결과 === \n")

# 2. pprint를 이용해 짤림 없이, 들여쓰기를 맞춰서 예쁘게 출력합니다.
# width=100은 한 줄의 길이를 넉넉하게 주어 가독성을 높입니다.
pprint.pprint(sample_mof, width=100, sort_dicts=False)


In [ ]:
import pandas as pd

# ---------------------------------------------------------
# 1. 안전하게 데이터 추출하기 (데이터가 없으면 'unknown' 이나 0으로 처리)
# ---------------------------------------------------------

# ① 기공 크기(PLD) 추출 (Zeopp 열이 있는지 먼저 확인)
if 'Zeopp' in df.columns:
    df['PLD_value'] = df['Zeopp'].apply(lambda x: x.get('PLD', 0) if isinstance(x, dict) else 0)
else:
    df['PLD_value'] = 0
    print("경고: 'Zeopp' 열을 찾을 수 없습니다.")

# ② 뼈대 안정성 추출 (대소문자 상관없이 찾기)
node_col = 'Crystalnets' if 'Crystalnets' in df.columns else 'crystalnets'

if node_col in df.columns:
    df['node_stability'] = df[node_col].apply(lambda x: x.get('all_nodes', 'unknown') if isinstance(x, dict) else 'unknown')
else:
    df['node_stability'] = 'unknown'
    print(f"경고: 뼈대 정보 열('{node_col}')을 찾을 수 없어 모두 'unknown'으로 처리합니다.")

# ③ 수분 저항성 판정 추출 (water 열이 있는지 확인)
if 'water' in df.columns:
    df['water_class'] = df['water'].apply(lambda x: x.get('water_classification', 'unknown') if isinstance(x, dict) else 'unknown')
else:
    df['water_class'] = 'unknown'
    print("경고: 'water' 열을 찾을 수 없습니다.")


# ---------------------------------------------------------
# 2. 대망의 3중 스크리닝 (가장 중요한 순서대로 필터링)
# ---------------------------------------------------------
final_candidates = df[
    (df['node_stability'] == 'stable') &   # 1순위: 관절이 무조건 튼튼할 것
    (df['PLD_value'] >= 3.3) &             # 2순위: 입구가 이산화탄소보다 넓을 것
    (df['water_class'] == 'weak')          # 3순위: 수분 흡착력이 약할(소수성) 것
]

print(f"\n🎯 3중 필터링 완료!")
print(f"혹독한 조건을 모두 통과한 최정예 MOF 개수: {len(final_candidates)}개")

# 합격자들의 성적표 확인하기 (합격자가 있을 경우에만 표 보여주기)
if len(final_candidates) > 0:
    display(final_candidates[['id', 'metal', 'node_stability', 'PLD_value', 'water_class']].head())
else:
    print("아쉽게도 모든 조건을 만족하는 MOF가 없습니다.")

In [ ]:
# === 범인 찾기: 각 조건별로 생존자가 몇 명인지 따로따로 세어보기 ===

print("🔍 [ 스크리닝 병목 현상 진단 리포트 ]\n")

# 1. 개별 조건 생존자 확인
stable_count = len(df[df['node_stability'] == 'stable'])
pld_count = len(df[df['PLD_value'] >= 3.3])
weak_count = len(df[df['water_class'] == 'weak'])

print(f"1. [안정성] 관절이 튼튼한(stable) MOF: {stable_count}개")
print(f"2. [크  기] 입구가 충분히 넓은(PLD >= 3.3) MOF: {pld_count}개")
print(f"3. [소수성] 물을 잘 튕겨내는(weak) MOF: {weak_count}개\n")

# 2. 혹시 데이터가 없어서 억울하게 탈락한 녀석들은?
unknown_node = len(df[df['node_stability'] == 'unknown'])
unknown_water = len(df[df['water_class'] == 'unknown'])

print(f"⚠️ [주의] 뼈대 정보가 누락된(unknown) MOF: {unknown_node}개")
print(f"⚠️ [주의] 수분 저항성 정보가 누락된(unknown) MOF: {unknown_water}개")

In [ ]:
import pandas as pd

# ---------------------------------------------------------
# 1. 완벽한 대소문자 타겟팅으로 데이터 추출하기
# ---------------------------------------------------------

# ① 기공 크기(PLD) 추출
df['PLD_value'] = df['Zeopp'].apply(lambda x: x.get('PLD', 0) if isinstance(x, dict) else 0)

# ② 뼈대 안정성 추출 (CrystalNets 대문자 N 정확히 타겟팅!)
if 'CrystalNets' in df.columns:
    df['node_stability'] = df['CrystalNets'].apply(lambda x: x.get('all_nodes', 'unknown') if isinstance(x, dict) else 'unknown')
else:
    # 혹시 몰라 기존 소문자 n도 대비해 두기
    df['node_stability'] = df.get('Crystalnets', pd.Series()).apply(lambda x: x.get('all_nodes', 'unknown') if isinstance(x, dict) else 'unknown')

# ③ 수분 저항성 판정 추출
df['water_class'] = df['water'].apply(lambda x: x.get('water_classification', 'unknown') if isinstance(x, dict) else 'unknown')


# ---------------------------------------------------------
# 2. 대망의 3중 스크리닝 (가장 중요한 순서대로 필터링)
# ---------------------------------------------------------
final_candidates = df[
    (df['node_stability'] == 'stable') &   # 1순위: 관절이 무조건 튼튼할 것
    (df['PLD_value'] >= 3.3) &             # 2순위: 입구가 이산화탄소보다 넓을 것
    (df['water_class'] == 'weak')          # 3순위: 수분 흡착력이 약할(소수성) 것
]

print(f"\n 3중 필터링을 완수했습니다")
print(f"전체 2,737개 중, 조건을 모두 통과한 최정예 MOF 개수: {len(final_candidates)}개")

# 합격자들의 성적표 확인하기 (합격자가 있을 경우)
if len(final_candidates) > 0:
    display(final_candidates[['id', 'metal', 'node_stability', 'PLD_value', 'water_class']].head())
else:
    print("\n아쉽게도 이 2,737개의 SI 데이터 안에는 조건을 모두 만족하는 유니콘 물질이 없습니다. 세상이 만만치 않죠?")
    print("다음 스텝으로 넘어가기 위해 조건을 살짝 완화해 보거나, 본편 데이터(CSD)를 가져와야 합니다.")

In [ ]:
# === 🔍 0개 사태 원인 규명: 스크리닝 부검 리포트 ===

print(" [ 충족 조건별 재스크리닝 ] \n")

# 1. 각 조건별 단독 생존자 수 확인
stable_count = len(df[df['node_stability'] == 'stable'])
pld_count = len(df[df['PLD_value'] >= 3.3])
weak_count = len(df[df['water_class'] == 'weak'])

print(f"1. [안정성] 뼈대가 튼튼한(stable) MOF: {stable_count}개")
print(f"2. [크  기] 입구가 넓은(PLD >= 3.3) MOF: {pld_count}개")
print(f"3. [소수성] 물을 잘 튕겨내는(weak) MOF: {weak_count}개\n")

# 2. 단계별로 썰려나간 과정 확인
step1_df = df[df['node_stability'] == 'stable']
print(f"👉 1단계 통과 (튼튼함): {len(step1_df)}개 생존")

step2_df = step1_df[step1_df['PLD_value'] >= 3.3]
print(f"👉 2단계 통과 (튼튼함 + 넓은 입구): {len(step2_df)}개 생존")

step3_df = step2_df[step2_df['water_class'] == 'weak']
print(f"👉 3단계 통과 (튼튼함 + 넓은 입구 + 방수): {len(step3_df)}개 (최종)")

In [ ]:
# === 1. CrystalNets 데이터의 민낯 파헤치기 ===

# 딕셔너리에서 'all_nodes' 값을 안전하게 꺼내오는 함수
def check_node_status(x):
    if isinstance(x, dict):
        # 딕셔너리는 맞는데 'all_nodes'라는 키워드가 없으면 '키 누락' 반환
        return x.get('all_nodes', '키 누락(No Key)')
    else:
        # 아예 딕셔너리 형태가 아니거나 비어있으면(NaN) '데이터 없음' 반환
        return '데이터 없음(Not Dict)'

# 데이터 추출
if 'CrystalNets' in df.columns:
    df['raw_node_status'] = df['CrystalNets'].apply(check_node_status)
else:
    df['raw_node_status'] = df.get('Crystalnets', pd.Series()).apply(check_node_status)

print("🔬 [ 뼈대 안정성 데이터(all_nodes) 종류별 개수 확인 ]\n")

# value_counts()를 쓰면 어떤 값들이 몇 개씩 있는지 한눈에 표로 정리해 줍니다.
print(df['raw_node_status'].value_counts())

In [ ]:
import pandas as pd

# 1. 추출 (안정성 추출은 포기하고, 크기와 수분 저항성만 뺍니다)
df['PLD_value'] = df['Zeopp'].apply(lambda x: x.get('PLD', 0) if isinstance(x, dict) else 0)
df['water_class'] = df['water'].apply(lambda x: x.get('water_classification', 'unknown') if isinstance(x, dict) else 'unknown')

# 2. 2중 스크리닝 (안정성 조건 임시 해제!)
final_candidates = df[
    (df['PLD_value'] >= 3.3) &             # 입구가 넓은 것
    (df['water_class'] == 'weak')          # 수분 저항성이 강한 것(소수성)
]

print("🚨 [ 안정성 조건 임시 해제 스크리닝 ] 🚨\n")
print(f"입구(PLD>=3.3)와 소수성(weak)만으로 살아남은 MOF 개수: {len(final_candidates)}개")

if len(final_candidates) > 0:
    display(final_candidates[['id', 'metal', 'PLD_value', 'water_class']].head(10))
else:
    print("\n아... 여전히 0개라면, 이 부록(SI) 데이터에는 '방수(weak)'가 되는 녀석 자체가 거의 없는 겁니다.")

In [ ]:
# === 🚀 질문자님의 아이디어: '매직 윈도우' 스크리닝 ===

# 아까 795개가 살아남았던 final_candidates 명단에서 한 번 더 걸러냅니다.
# N2(3.64 Å)의 입체 장애를 유도하기 위해 PLD 상한선을 3.6 이하로 깎아냅니다.

magic_window_candidates = final_candidates[
    (final_candidates['PLD_value'] >= 3.3) & 
    (final_candidates['PLD_value'] <= 3.6)
]

print("🚨 [ 매직 윈도우 (3.3 ≤ PLD ≤ 3.6) 도입 결과 ] 🚨\n")
print(f"질소(N2)를 완벽히 차단할 수 있는 구조적 크기를 가진 MOF: {len(magic_window_candidates)}개 생존!")

if len(magic_window_candidates) > 0:
    display(magic_window_candidates[['id', 'metal', 'PLD_value', 'water_class']].head(10))
else:
    print("너무 가혹한 조건이었나 봅니다. 상한선을 3.8이나 4.0 정도로 살짝 여유를 두어야 할 것 같습니다.")

In [ ]:
import pandas as pd

# (어제까지의 코드가 실행되어 'magic_window_candidates' 명단이 63개로 잘 남아있다고 가정합니다)

print(" [ 최정예 63개 MOF 중심 금속(Metal) 분포 확인 ]\n")

# 1. 63개 MOF가 어떤 금속을 쓰고 있는지 랭킹 매기기
metal_distribution = magic_window_candidates['metal'].value_counts()
print(metal_distribution)
print("-" * 50)

# 2. 분석하기 편하게 전체 명단을 CSV(엑셀) 파일로 뽑아내기
export_cols = ['id', 'metal', 'PLD_value', 'water_class', 'reference']
export_df = magic_window_candidates[export_cols]

file_name = 'MAGIC_WINDOW_63_MOFS.csv'
export_df.to_csv(file_name, index=False)

print(f" 최종 63개의 명단이 '{file_name}' 파일로 저장되었습니다.")

In [ ]:
import pandas as pd

# === 🚀 질문자님의 아이디어: '매직 윈도우' 스크리닝 ===

magic_window_candidates = final_candidates[
    (final_candidates['PLD_value'] >= 3.3) & 
    (final_candidates['PLD_value'] <= 3.6)
]

print("🚨 [ 매직 윈도우 (3.3 ≤ PLD ≤ 3.6) 도입 결과 ] 🚨\n")
print(f"질소를 완벽히 차단할 수 있는 구조적 크기를 가진 MOF: {len(magic_window_candidates)}개 생존!")

if len(magic_window_candidates) > 0:
    
    # 1. 주피터 노트북 화면에 63개를 중간 생략 없이 전부 띄우기 위한 마법의 설정
    pd.set_option('display.max_rows', None) # 최대 출력 행 수를 '무제한(None)'으로 설정
    
    # 63개 전체 표 출력 (head를 빼버립니다)
    display(magic_window_candidates[['id', 'metal', 'PLD_value', 'water_class']])
    
    # 출력이 끝난 후에는 노트북이 너무 길어지는 걸 방지하기 위해 설정을 원상복구 해줍니다.
    pd.reset_option('display.max_rows')

    # ---------------------------------------------------------
    # 2. 이 63개의 전체 데이터를 엑셀로 열 수 있는 CSV 파일로 내보내기
    # ---------------------------------------------------------
    file_name = 'Magic_Window_63_MOFS.csv'
    
    # 엑셀에서 열 때 특수문자(Å 등)가 깨지지 않도록 encoding='utf-8-sig'를 꼭 넣어줍니다.
    magic_window_candidates.to_csv(file_name, index=False, encoding='utf-8-sig')
    
    print(f"\n✅ 데이터 추출 성공! 폴더에 '{file_name}' 파일이 생성되었습니다.")
    print("이 파일을 더블클릭해서 엑셀로 여시면 63개 전체 데이터를 편하게 보실 수 있습니다.")

else:
    print("조건을 만족하는 MOF가 없습니다.")

In [ ]:
import pandas as pd

# 1. 기존에 합쳐둔 통합 데이터(df)에서 LCD(내부 동굴 크기) 데이터 추가 추출
df['PLD_value'] = df['Zeopp'].apply(lambda x: x.get('PLD', 0) if isinstance(x, dict) else 0)
df['LCD_value'] = df['Zeopp'].apply(lambda x: x.get('LCD', 0) if isinstance(x, dict) else 0)
df['water_class'] = df['water'].apply(lambda x: x.get('water_classification', 'unknown') if isinstance(x, dict) else 'unknown')

# 2. 호리병(Bottle-neck) 구조 스크리닝
# 조건 1: 입구는 속도를 위해 살짝 널널하게 (3.4 ~ 3.9 Å)
# 조건 2: 내부는 대용량 저장을 위해 아주 넓게 (5.0 Å 이상)
# 조건 3: 습윤 환경을 버티기 위한 소수성 (weak)
bottleneck_candidates = df[
    (df['PLD_value'] >= 3.4) & 
    (df['PLD_value'] <= 3.9) & 
    (df['LCD_value'] >= 5.0) & 
    (df['water_class'] == 'weak')
]

print(" [ 플랜 B: 호리병(Bottle-neck) 구조 스크리닝 결과 ] \n")
print(f"입구는 좁고 속은 뻥 뚫린 대용량 소수성 MOF: {len(bottleneck_candidates)}개 생존!")

if len(bottleneck_candidates) > 0:
    display(bottleneck_candidates[['id', 'metal', 'PLD_value', 'LCD_value', 'water_class']].head(10))
    
    # 원하신다면 이 결과도 바로 엑셀로 저장!
    bottleneck_candidates.to_csv('Bottleneck_MOFS.csv', index=False, encoding='utf-8-sig')
else:
    print("조건이 너무 빡빡한가 봅니다. LCD 기준을 4.5 정도로 낮춰보세요.")

In [ ]:
# === 🔬 GEMC 시뮬레이션 알맹이 데이터 끝까지 까보기 ===

# (주의: 저 데이터가 'CO2'라는 이름의 열에 들어있다고 가정했습니다. 
# 만약 열 이름이 다르다면 combined_df['실제열이름'] 으로 바꿔주세요!)

sample_data = combined_df['CO2'].iloc[0] 

print("🚨 [ GEMC 시뮬레이션 전체 로그 출력 ] 🚨\n")

if isinstance(sample_data, dict) and 'GEMC' in sample_data:
    # 리스트 안의 문장들을 하나씩 줄바꿈해서 출력합니다.
    for line in sample_data['GEMC']:
        print(line.strip()) # 텍스트 끝의 \n 기호를 지워줘서 깔끔하게 나옵니다.
else:
    print("앗, 데이터 구조가 예상과 다릅니다. 이 샘플에는 GEMC 데이터가 없네요.")

In [ ]:
import pandas as pd

# 1. G열(index 6) 및 필요한 물리량 데이터 추출
# G열에 GEMC 시뮬레이션 결과가 있다고 하셨으므로 iloc를 사용하여 접근합니다.
df['GEMC_data'] = df.iloc[:, 6] 

# Zeo++ 및 수분 정보 추출
df['PLD_value'] = df['Zeopp'].apply(lambda x: x.get('PLD', 0) if isinstance(x, dict) else 0)
df['LCD_value'] = df['Zeopp'].apply(lambda x: x.get('LCD', 0) if isinstance(x, dict) else 0)
df['ASA_value'] = df['Zeopp'].apply(lambda x: x.get('ASA', 0) if isinstance(x, dict) else 0)
df['water_class'] = df['water'].apply(lambda x: x.get('water_classification', 'unknown') if isinstance(x, dict) else 'unknown')

# 2. 두 가지 핵심 타겟 그룹 스크리닝
# [그룹 1] 매직 윈도우 (고순도형): 질소 완벽 차단
magic_window = df[
    (df['PLD_value'] >= 3.3) & (df['PLD_value'] <= 3.6) & 
    (df['water_class'] == 'weak')
]

# [그룹 2] 고용량 호리병 구조 (고속/대용량형): 입구는 살짝 넓게, 속은 뻥 뚫리게
# 입구(PLD)를 3.8~4.2로 넓혀 속도를 확보하고, 내부(LCD)를 5.5 이상으로 잡아 용량을 극대화합니다.
high_capacity_bottleneck = df[
    (df['PLD_value'] >= 3.7) & (df['PLD_value'] <= 4.2) & 
    (df['LCD_value'] >= 5.5) & 
    (df['water_class'] == 'weak')
]

print(f"✅ G열 데이터 로드 완료")
print(f"🎯 [그룹 1] 매직 윈도우(고순도) 후보: {len(magic_window)}개")
print(f"🚀 [그룹 2] 고용량 호리병(고속/대용량) 후보: {len(high_capacity_bottleneck)}개")

# 결과 저장
magic_window.to_csv('Target_Group_1_MagicWindow.csv', index=False, encoding='utf-8-sig')
high_capacity_bottleneck.to_csv('Target_Group_2_HighCapacity.csv', index=False, encoding='utf-8-sig')

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# 1. 질문자님이 저장하신 두 개의 파일 불러오기
try:
    df_magic = pd.read_csv('MAGIC_WINDOW_63_MOFS.csv')
    df_bottle = pd.read_csv('Bottleneck_MOFS.csv')
    
    print("✅ 두 파일 모두 성공적으로 불러왔습니다!\n")

    # 2. 각 그룹의 금속(metal) 개수 세기
    # 상위 10개 금속만 추려서 보기 좋게 만듭니다.
    magic_metals = df_magic['metal'].value_counts().head(10).rename('Magic Window (Narrow)')
    bottle_metals = df_bottle['metal'].value_counts().head(10).rename('Bottleneck (Wide)')

    # 3. 두 데이터를 하나의 표로 합치기 (빈칸은 0으로 채움)
    comparison_df = pd.concat([magic_metals, bottle_metals], axis=1).fillna(0)

    # 4. 비교표 출력
    print("📊 [ 금속 분포 비교표 ]")
    display(comparison_df.astype(int))

    # 5. 한눈에 들어오는 비교 막대그래프 그리기
    # 폰트 깨짐 방지를 위해 영문 라벨 사용
    ax = comparison_df.plot(kind='bar', figsize=(12, 6), color=['#1f77b4', '#ff7f0e'], edgecolor='black')
    
    plt.title('Metal Distribution Comparison: Magic Window vs Bottleneck', fontsize=16, fontweight='bold')
    plt.xlabel('Metal Type', fontsize=12)
    plt.ylabel('Number of MOFs', fontsize=12)
    plt.xticks(rotation=45)
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    plt.legend(fontsize=12)
    
    plt.tight_layout()
    plt.show()

except FileNotFoundError as e:
    print(f"앗! 파일 이름이 다르거나 같은 폴더에 없습니다: {e}")

In [ ]:
# === 📊 GEMC 시뮬레이션 성적표 구조 파악하기 ===

# 호리병 그룹(52개) 중에서 첫 번째로 살아남은 녀석의 G열 데이터 하나만 쏙 뽑아봅니다.
sample_gemc = high_capacity_bottleneck['GEMC_data'].iloc[0]

print("🔍 [ GEMC 데이터 내부 구조(Key) 확인 ]\n")

if isinstance(sample_gemc, dict):
    # 딕셔너리가 맞다면, 어떤 항목(Key)들이 있는지 예쁘게 출력
    for key, value in sample_gemc.items():
        print(f"▶ {key}: {value}")
else:
    print("딕셔너리 형태가 아닙니다. 실제 데이터 형태를 확인해 주세요:")
    print(sample_gemc)

In [ ]:
import pandas as pd

# ---------------------------------------------------------
# 1. J열(index 9)에서 "진짜" GEMC 시뮬레이션 데이터 구출하기
# ---------------------------------------------------------
# A열이 0이므로, J열은 인덱스 9입니다.
df['GEMC_data'] = df.iloc[:, 9] 

# 물리량 및 수분 정보 추출
df['PLD_value'] = df['Zeopp'].apply(lambda x: x.get('PLD', 0) if isinstance(x, dict) else 0)
df['LCD_value'] = df['Zeopp'].apply(lambda x: x.get('LCD', 0) if isinstance(x, dict) else 0)
df['water_class'] = df['water'].apply(lambda x: x.get('water_classification', 'unknown') if isinstance(x, dict) else 'unknown')

# ---------------------------------------------------------
# 2. 그룹 다시 세팅 (매직 윈도우 63개 & 호리병 52개)
# ---------------------------------------------------------
magic_window = df[
    (df['PLD_value'] >= 3.3) & (df['PLD_value'] <= 3.6) & 
    (df['water_class'] == 'weak')
]

high_capacity_bottleneck = df[
    (df['PLD_value'] >= 3.7) & (df['PLD_value'] <= 4.2) & 
    (df['LCD_value'] >= 5.5) & 
    (df['water_class'] == 'weak')
]

print("✅ J열 타겟팅 완료! 그룹 재설정 완료!")

# ---------------------------------------------------------
# 3. 드디어 까보는 진짜 GEMC 시뮬레이션 성적표 (J열)
# ---------------------------------------------------------
# 호리병 그룹의 첫 번째 MOF 성적표를 까봅니다.
sample_gemc = high_capacity_bottleneck['GEMC_data'].iloc[0]

print("\n🔍 [ 찐 GEMC 데이터 내부 구조(Key) 확인 ]\n")

if isinstance(sample_gemc, dict):
    for key, value in sample_gemc.items():
        print(f"▶ {key}: {value}")
else:
    print("딕셔너리 형태가 아닙니다. 실제 데이터 형태를 확인해 주세요:")
    print(sample_gemc)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import ast

# 1. 저장된 후보군 파일 불러오기
try:
    df_magic = pd.read_csv('MAGIC_WINDOW_63_MOFS.csv')
    df_bottle = pd.read_csv('Bottleneck_MOFS.csv')
    print("✅ 데이터 로드 완료!")
except FileNotFoundError:
    print("앗! 파일이 없습니다. 파일 이름을 확인해 주세요.")

# 2. J열(GEMC_data)에서 Widom 성적표를 추출하는 함수
def extract_performance(data_str):
    try:
        # 문자열로 저장된 딕셔너리를 실제 파이썬 객체로 변환
        if isinstance(data_str, str):
            data = ast.literal_eval(data_str)
        else:
            data = data_str
            
        widom = data.get('Widom', [])
        if len(widom) >= 2:
            co2_affinity = widom[0]
            n2_affinity = widom[1]
            # 선택도 계산: CO2 / N2
            selectivity = co2_affinity / n2_affinity if n2_affinity > 0 else 0
            return pd.Series([co2_affinity, selectivity])
    except:
        return pd.Series([np.nan, np.nan])
    return pd.Series([np.nan, np.nan])

# 3. 각 그룹별 성적 계산
df_magic[['CO2_Affinity', 'Selectivity']] = df_magic['GEMC_data'].apply(extract_performance)
df_bottle[['CO2_Affinity', 'Selectivity']] = df_bottle['GEMC_data'].apply(extract_performance)

# 데이터가 없는 녀석들 제외
df_magic_clean = df_magic.dropna(subset=['CO2_Affinity', 'Selectivity'])
df_bottle_clean = df_bottle.dropna(subset=['CO2_Affinity', 'Selectivity'])

# 4. 성능 비교 산점도(Scatter Plot) 그리기
plt.figure(figsize=(12, 8))

# 그룹 1: 파란색 (매직 윈도우 - 고순도 타겟)
plt.scatter(df_magic_clean['CO2_Affinity'], df_magic_clean['Selectivity'], 
            alpha=0.6, s=100, c='#1f77b4', edgecolors='white', label='Group 1: Magic Window (Narrow)')

# 그룹 2: 주황색 (호리병 - 고용량/고속 타겟)
plt.scatter(df_bottle_clean['CO2_Affinity'], df_bottle_clean['Selectivity'], 
            alpha=0.6, s=100, c='#ff7f0e', edgecolors='white', label='Group 2: Bottleneck (Wide)')

# 축 설정 (값이 미세하므로 로그 스케일 권장)
plt.xscale('log')
plt.yscale('log')

plt.title('MOF Performance Showdown: Affinity vs Selectivity', fontsize=16, fontweight='bold')
plt.xlabel('CO2 Affinity (Henry\'s Constant) - 대용량 지표', fontsize=12)
plt.ylabel('Selectivity (CO2 / N2) - 고순도 지표', fontsize=12)
plt.grid(True, which="both", ls="--", alpha=0.5)
plt.legend(fontsize=12)

# 각 점이 어떤 MOF인지 ID 표시 (너무 많으면 복잡하니 상위 몇 개만 표시하는 것도 방법입니다)
plt.tight_layout()
plt.show()

print(f"📊 분석 결과:")
print(f"- 매직 윈도우 유효 데이터: {len(df_magic_clean)}개")
print(f"- 호리병 구조 유효 데이터: {len(df_bottle_clean)}개")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# 1. 원본 데이터(df)에서 J열(index 9) 타겟팅 및 기본 물리량 세팅
df['GEMC_data'] = df.iloc[:, 9] 
df['PLD_value'] = df['Zeopp'].apply(lambda x: x.get('PLD', 0) if isinstance(x, dict) else 0)
df['LCD_value'] = df['Zeopp'].apply(lambda x: x.get('LCD', 0) if isinstance(x, dict) else 0)
df['water_class'] = df['water'].apply(lambda x: x.get('water_classification', 'unknown') if isinstance(x, dict) else 'unknown')

# 2. CSV 파일 대신, 원본에서 바로 두 그룹으로 나누기 (.copy()로 안전하게 복사)
df_magic = df[
    (df['PLD_value'] >= 3.3) & (df['PLD_value'] <= 3.6) & 
    (df['water_class'] == 'weak')
].copy()

df_bottle = df[
    (df['PLD_value'] >= 3.7) & (df['PLD_value'] <= 4.2) & 
    (df['LCD_value'] >= 5.5) & 
    (df['water_class'] == 'weak')
].copy()

# 3. J열(GEMC_data)에서 Widom 성적표를 추출하는 함수
def extract_performance_from_dict(gemc_data):
    try:
        # 원본 데이터는 문자열이 아닌 딕셔너리 형태로 바로 들어있습니다
        if isinstance(gemc_data, dict) and 'Widom' in gemc_data:
            widom = gemc_data['Widom']
            if isinstance(widom, list) and len(widom) >= 2:
                co2_affinity = widom[0]
                n2_affinity = widom[1]
                selectivity = co2_affinity / n2_affinity if n2_affinity > 0 else 0
                return pd.Series([co2_affinity, selectivity])
    except:
        pass
    return pd.Series([np.nan, np.nan])

# 4. 성적 계산 및 결측치(데이터 없는 애들) 제거
print("🔄 [ 성능 지표(Widom) 추출 및 차트 준비 중... ]")
df_magic[['CO2_Affinity', 'Selectivity']] = df_magic['GEMC_data'].apply(extract_performance_from_dict)
df_bottle[['CO2_Affinity', 'Selectivity']] = df_bottle['GEMC_data'].apply(extract_performance_from_dict)

df_magic_clean = df_magic.dropna(subset=['CO2_Affinity', 'Selectivity'])
df_bottle_clean = df_bottle.dropna(subset=['CO2_Affinity', 'Selectivity'])

# 5. 성능 비교 산점도(Scatter Plot) 그리기
plt.figure(figsize=(12, 8))

# 그룹 1: 파란색 (매직 윈도우 - 고순도 타겟)
plt.scatter(df_magic_clean['CO2_Affinity'], df_magic_clean['Selectivity'], 
            alpha=0.6, s=100, c='#1f77b4', edgecolors='white', label='Group 1: Magic Window (Narrow)')

# 그룹 2: 주황색 (호리병 - 고용량/고속 타겟)
plt.scatter(df_bottle_clean['CO2_Affinity'], df_bottle_clean['Selectivity'], 
            alpha=0.6, s=100, c='#ff7f0e', edgecolors='white', label='Group 2: Bottleneck (Wide)')

# 축 설정 (값이 미세하므로 로그 스케일 적용)
plt.xscale('log')
plt.yscale('log')

plt.title('MOF Performance Showdown: Affinity vs Selectivity', fontsize=16, fontweight='bold')
plt.xlabel('CO2 Affinity (Henry\'s Constant) ➡️ 대용량 지표', fontsize=12)
plt.ylabel('Selectivity (CO2 / N2) ➡️ 고순도 지표', fontsize=12)
plt.grid(True, which="both", ls="--", alpha=0.5)
plt.legend(fontsize=12, loc='upper left')

plt.tight_layout()
plt.show()

print(f"📊 분석 완료!")
print(f"- 매직 윈도우 유효 데이터: {len(df_magic_clean)}개")
print(f"- 호리병 구조 유효 데이터: {len(df_bottle_clean)}개")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# ---------------------------------------------------------
# 🌟 한글 폰트 깨짐 방지 패치 (윈도우용)
# ---------------------------------------------------------
plt.rc('font', family='Malgun Gothic')
plt.rcParams['axes.unicode_minus'] = False # 마이너스(-) 기호가 깨지는 것도 방지해 줍니다.

# 1. 원본 데이터(df)에서 J열(index 9) 타겟팅 및 기본 물리량 세팅
df['GEMC_data'] = df.iloc[:, 9] 
df['PLD_value'] = df['Zeopp'].apply(lambda x: x.get('PLD', 0) if isinstance(x, dict) else 0)
df['LCD_value'] = df['Zeopp'].apply(lambda x: x.get('LCD', 0) if isinstance(x, dict) else 0)
df['water_class'] = df['water'].apply(lambda x: x.get('water_classification', 'unknown') if isinstance(x, dict) else 'unknown')

# 2. 원본에서 바로 두 그룹으로 나누기
df_magic = df[
    (df['PLD_value'] >= 3.3) & (df['PLD_value'] <= 3.6) & 
    (df['water_class'] == 'weak')
].copy()

df_bottle = df[
    (df['PLD_value'] >= 3.7) & (df['PLD_value'] <= 4.2) & 
    (df['LCD_value'] >= 5.5) & 
    (df['water_class'] == 'weak')
].copy()

# 3. J열(GEMC_data)에서 Widom 성적표를 추출하는 함수
def extract_performance_from_dict(gemc_data):
    try:
        if isinstance(gemc_data, dict) and 'Widom' in gemc_data:
            widom = gemc_data['Widom']
            if isinstance(widom, list) and len(widom) >= 2:
                co2_affinity = widom[0]
                n2_affinity = widom[1]
                selectivity = co2_affinity / n2_affinity if n2_affinity > 0 else 0
                return pd.Series([co2_affinity, selectivity])
    except:
        pass
    return pd.Series([np.nan, np.nan])

# 4. 성적 계산 및 결측치 제거
print("🔄 [ 성능 지표(Widom) 추출 및 차트 준비 중... ]")
df_magic[['CO2_Affinity', 'Selectivity']] = df_magic['GEMC_data'].apply(extract_performance_from_dict)
df_bottle[['CO2_Affinity', 'Selectivity']] = df_bottle['GEMC_data'].apply(extract_performance_from_dict)

df_magic_clean = df_magic.dropna(subset=['CO2_Affinity', 'Selectivity'])
df_bottle_clean = df_bottle.dropna(subset=['CO2_Affinity', 'Selectivity'])

# 5. 성능 비교 산점도(Scatter Plot) 그리기
plt.figure(figsize=(12, 8))

# 그룹 1: 파란색 (매직 윈도우 - 고순도 타겟)
plt.scatter(df_magic_clean['CO2_Affinity'], df_magic_clean['Selectivity'], 
            alpha=0.6, s=100, c='#1f77b4', edgecolors='white', label='Group 1: 매직 윈도우 (고순도)')

# 그룹 2: 주황색 (호리병 - 고용량/고속 타겟)
plt.scatter(df_bottle_clean['CO2_Affinity'], df_bottle_clean['Selectivity'], 
            alpha=0.6, s=100, c='#ff7f0e', edgecolors='white', label='Group 2: 호리병 (고용량/고속)')

# 축 설정 (로그 스케일 적용)
plt.xscale('log')
plt.yscale('log')

plt.title('MOF 성능 최종 서열: 흡착력 vs 선택도', fontsize=16, fontweight='bold')
plt.xlabel('CO2 흡착력 (Henry\'s Constant) ➡️ 우측일수록 대용량', fontsize=12)
plt.ylabel('선택도 (CO2 / N2) ➡️ 상단일수록 고순도', fontsize=12)
plt.grid(True, which="both", ls="--", alpha=0.5)
plt.legend(fontsize=12, loc='upper left')

plt.tight_layout()
plt.show()

print(f"📊 분석 완료!")
print(f"- 매직 윈도우 유효 데이터: {len(df_magic_clean)}개")
print(f"- 호리병 구조 유효 데이터: {len(df_bottle_clean)}개")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# 1. 원본 데이터(df)에서 J열(index 9) 타겟팅 및 기본 물리량 세팅
df['GEMC_data'] = df.iloc[:, 9] 
df['PLD_value'] = df['Zeopp'].apply(lambda x: x.get('PLD', 0) if isinstance(x, dict) else 0)
df['LCD_value'] = df['Zeopp'].apply(lambda x: x.get('LCD', 0) if isinstance(x, dict) else 0)
df['water_class'] = df['water'].apply(lambda x: x.get('water_classification', 'unknown') if isinstance(x, dict) else 'unknown')

# 2. 원본에서 바로 두 그룹으로 나누기
df_magic = df[
    (df['PLD_value'] >= 3.3) & (df['PLD_value'] <= 3.6) & 
    (df['water_class'] == 'weak')
].copy()

df_bottle = df[
    (df['PLD_value'] >= 3.7) & (df['PLD_value'] <= 4.2) & 
    (df['LCD_value'] >= 5.5) & 
    (df['water_class'] == 'weak')
].copy()

# 3. J열(GEMC_data)에서 Widom 성적표를 추출하는 함수
def extract_performance_from_dict(gemc_data):
    try:
        if isinstance(gemc_data, dict) and 'Widom' in gemc_data:
            widom = gemc_data['Widom']
            if isinstance(widom, list) and len(widom) >= 2:
                co2_affinity = widom[0]
                n2_affinity = widom[1]
                selectivity = co2_affinity / n2_affinity if n2_affinity > 0 else 0
                return pd.Series([co2_affinity, selectivity])
    except:
        pass
    return pd.Series([np.nan, np.nan])

# 4. 성적 계산 및 결측치 제거
print("🔄 [ Extracting Performance Data (Widom) & Preparing Chart... ]")
df_magic[['CO2_Affinity', 'Selectivity']] = df_magic['GEMC_data'].apply(extract_performance_from_dict)
df_bottle[['CO2_Affinity', 'Selectivity']] = df_bottle['GEMC_data'].apply(extract_performance_from_dict)

df_magic_clean = df_magic.dropna(subset=['CO2_Affinity', 'Selectivity'])
df_bottle_clean = df_bottle.dropna(subset=['CO2_Affinity', 'Selectivity'])

# 5. 성능 비교 산점도(Scatter Plot) 그리기
plt.figure(figsize=(12, 8))

# 그룹 1: 파란색 (매직 윈도우 - 고순도 타겟)
plt.scatter(df_magic_clean['CO2_Affinity'], df_magic_clean['Selectivity'], 
            alpha=0.6, s=100, c='#1f77b4', edgecolors='white', label='Group 1: Magic Window (High Purity)')

# 그룹 2: 주황색 (호리병 - 고용량/고속 타겟)
plt.scatter(df_bottle_clean['CO2_Affinity'], df_bottle_clean['Selectivity'], 
            alpha=0.6, s=100, c='#ff7f0e', edgecolors='white', label='Group 2: Bottleneck (High Capacity/Speed)')

# 축 설정 (로그 스케일 적용)
plt.xscale('log')
plt.yscale('log')

plt.title('MOF Performance Showdown: CO2 Affinity vs Selectivity', fontsize=16, fontweight='bold')
plt.xlabel('CO2 Affinity (Henry\'s Constant) -> Higher is better capacity', fontsize=12)
plt.ylabel('Selectivity (CO2 / N2) -> Higher is better purity', fontsize=12)
plt.grid(True, which="both", ls="--", alpha=0.5)
plt.legend(fontsize=12, loc='upper left')

plt.tight_layout()
plt.show()

print(f"📊 Analysis Complete!")
print(f"- Magic Window Valid Data: {len(df_magic_clean)}")
print(f"- Bottleneck Valid Data: {len(df_bottle_clean)}")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# 🌟 1. 환경 설정 (마이너스 기호 깨짐 방지)
plt.rcParams['axes.unicode_minus'] = False 

# 2. 데이터 전처리 및 물리량 추출 (J열 index 9 사용)
df['GEMC_data'] = df.iloc[:, 9] 
df['PLD_value'] = df['Zeopp'].apply(lambda x: x.get('PLD', 0) if isinstance(x, dict) else 0)
df['LCD_value'] = df['Zeopp'].apply(lambda x: x.get('LCD', 0) if isinstance(x, dict) else 0)
df['water_class'] = df['water'].apply(lambda x: x.get('water_classification', 'unknown') if isinstance(x, dict) else 'unknown')

# 3. 그룹 재정의 (기술적 명칭 적용)
# Group A: N2-Sieving (질소 지름 3.64Å 미만 필터링)
n2_sieving_group = df[
    (df['PLD_value'] >= 3.3) & (df['PLD_value'] <= 3.6) & 
    (df['water_class'] == 'weak')
].copy()

# Group B: High-Flux Bottleneck (입구는 확보하되 내부는 넓은 대용량 구조)
high_flux_group = df[
    (df['PLD_value'] >= 3.7) & (df['PLD_value'] <= 4.2) & 
    (df['LCD_value'] >= 5.5) & 
    (df['water_class'] == 'weak')
].copy()

# 4. 성능 지표(Widom) 추출 함수
def extract_metrics(gemc_data):
    try:
        if isinstance(gemc_data, dict) and 'Widom' in gemc_data:
            widom = gemc_data['Widom']
            if isinstance(widom, list) and len(widom) >= 2:
                co2_affinity = widom[0]
                n2_affinity = widom[1]
                selectivity = co2_affinity / n2_affinity if n2_affinity > 0 else 0
                return pd.Series([co2_affinity, selectivity])
    except:
        pass
    return pd.Series([np.nan, np.nan])

# 5. 데이터 가공 및 파일 저장
print("🔄 Processing N2-Sieving and High-Flux groups...")
n2_sieving_group[['CO2_Affinity', 'Selectivity']] = n2_sieving_group['GEMC_data'].apply(extract_metrics)
high_flux_group[['CO2_Affinity', 'Selectivity']] = high_flux_group['GEMC_data'].apply(extract_metrics)

# CSV 파일 이름 수정 저장
n2_sieving_group.to_csv('N2_Sieving_Group_63_MOFs.csv', index=False, encoding='utf-8-sig')
high_flux_group.to_csv('High_Flux_Bottleneck_52_MOFs.csv', index=False, encoding='utf-8-sig')

# 6. 최종 성능 비교 차트 (영문 통일)
plt.figure(figsize=(12, 8))

plt.scatter(n2_sieving_group['CO2_Affinity'], n2_sieving_group['Selectivity'], 
            alpha=0.6, s=100, c='#1f77b4', edgecolors='white', label='Group A: N2-Sieving (Size Exclusion)')

plt.scatter(high_flux_group['CO2_Affinity'], high_flux_group['Selectivity'], 
            alpha=0.6, s=100, c='#ff7f0e', edgecolors='white', label='Group B: High-Flux Bottleneck (Capacitive)')

plt.xscale('log')
plt.yscale('log')
plt.title('Performance Mapping: N2-Sieving vs High-Flux Groups', fontsize=16, fontweight='bold')
plt.xlabel('CO2 Affinity (Henry\'s Constant) -> Capacity indicator', fontsize=12)
plt.ylabel('Selectivity (CO2 / N2) -> Purity indicator', fontsize=12)
plt.grid(True, which="both", ls="--", alpha=0.5)
plt.legend(fontsize=12)

plt.tight_layout()
plt.show()

print(f"📊 Final Report:")
print(f"- N2_Sieving_Group_63_MOFs.csv saved.")
print(f"- High_Flux_Bottleneck_52_MOFs.csv saved.")

In [ ]:
!pip install seaborn

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np
import ast
import seaborn as sns

# 🌟 1. 철통 방어: 폰트 및 마이너스 기호 강제 설정
mpl.rcParams['font.family'] = 'sans-serif'
mpl.rcParams['font.sans-serif'] = ['Arial', 'DejaVu Sans', 'Liberation Sans']
mpl.rcParams['axes.unicode_minus'] = False # 마이너스 기호 깨짐 최종 방어선

# 2. 필요한 열 타겟팅 (G열=인덱스 6, J열=인덱스 9)
df['metal_info'] = df.iloc[:, 6] # G열: 금속 및 OMS 정보
df['GEMC_data'] = df.iloc[:, 9]  # J열: Widom 시뮬레이션 성적표

# 기본 물리량 세팅
df['PLD_value'] = df['Zeopp'].apply(lambda x: x.get('PLD', 0) if isinstance(x, dict) else 0)
df['LCD_value'] = df['Zeopp'].apply(lambda x: x.get('LCD', 0) if isinstance(x, dict) else 0)
df['water_class'] = df['water'].apply(lambda x: x.get('water_classification', 'unknown') if isinstance(x, dict) else 'unknown')

# 3. 그룹 분리 (기술적 명칭 유지)
n2_sieving = df[
    (df['PLD_value'] >= 3.3) & (df['PLD_value'] <= 3.6) & 
    (df['water_class'] == 'weak')
].copy()
n2_sieving['Filter_Type'] = 'N2-Sieving (Narrow)'

high_flux = df[
    (df['PLD_value'] >= 3.7) & (df['PLD_value'] <= 4.2) & 
    (df['LCD_value'] >= 5.5) & 
    (df['water_class'] == 'weak')
].copy()
high_flux['Filter_Type'] = 'High-Flux (Wide)'

# 두 그룹을 하나의 데이터프레임으로 합치기 (Seaborn 처리를 위해)
combined_df = pd.concat([n2_sieving, high_flux], ignore_index=True)

# 4. 데이터 추출 함수들
def extract_metrics(gemc_data):
    try:
        if isinstance(gemc_data, str):
            gemc_data = ast.literal_eval(gemc_data)
        if isinstance(gemc_data, dict) and 'Widom' in gemc_data:
            widom = gemc_data['Widom']
            if isinstance(widom, list) and len(widom) >= 2:
                co2_affinity = widom[0]
                n2_affinity = widom[1]
                selectivity = co2_affinity / n2_affinity if n2_affinity > 0 else 0
                return pd.Series([co2_affinity, selectivity])
    except:
        pass
    return pd.Series([np.nan, np.nan])

def extract_metal_category(metal_data):
    try:
        if isinstance(metal_data, str):
            metal_data = ast.literal_eval(metal_data)
        if isinstance(metal_data, dict):
            metal = metal_data.get('metal_type', 'Unknown')
            oms = metal_data.get('has_OMS', 'Unknown')
            return f"{metal} (OMS: {oms})"
    except:
        pass
    return "Unknown"

# 추출 실행
print("🔄 Extracting Multi-dimensional Data...")
combined_df[['CO2_Affinity', 'Selectivity']] = combined_df['GEMC_data'].apply(extract_metrics)
combined_df['Chemistry'] = combined_df['metal_info'].apply(extract_metal_category)

# 결측치 제거 및 데이터 정리
final_clean_df = combined_df.dropna(subset=['CO2_Affinity', 'Selectivity'])
# Unknown 등 분석에 방해되는 찌꺼기 데이터 필터링
final_clean_df = final_clean_df[final_clean_df['Chemistry'] != 'Unknown (OMS: Unknown)']

# 5. 궁극의 다차원 산점도 (Seaborn)
plt.figure(figsize=(14, 9))

# hue: 색상은 금속 화학종, style: 점의 모양은 N2-Sieving/High-Flux 그룹
ax = sns.scatterplot(
    data=final_clean_df, 
    x='CO2_Affinity', 
    y='Selectivity', 
    hue='Chemistry', 
    style='Filter_Type',
    s=150,           # 점 크기
    alpha=0.85,      # 투명도
    edgecolor='black',
    palette='tab20'  # 색상 팔레트 (다양한 색상 지원)
)

# 축 설정 (로그 스케일)
plt.xscale('log')
plt.yscale('log')

# 디자인 다듬기
plt.title('MOF CCUS Frontier: Chemistry & Structural Analysis', fontsize=18, fontweight='bold', pad=15)
plt.xlabel('CO2 Affinity (Henry\'s Constant) -> Higher Capacity', fontsize=14)
plt.ylabel('Selectivity (CO2 / N2) -> Higher Purity', fontsize=14)
plt.grid(True, which="both", ls="--", alpha=0.4)

# 범례(Legend)를 보기 좋게 바깥으로 빼기
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', borderaxespad=0., fontsize=11, title='Material Properties', title_fontsize=12)

plt.tight_layout()
plt.show()

print(f"📊 Visualization Complete! Total valid MOFs plotted: {len(final_clean_df)}")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# 1. 3개의 그래프에서 동일한 금속이 동일한 색상을 가지도록 '색상 순서(hue_order)'를 고정합니다.
chemistry_order = sorted(final_clean_df['Chemistry'].unique())

# 2. 개별 그룹 데이터 분리
n2_sieving_df = final_clean_df[final_clean_df['Filter_Type'] == 'N2-Sieving (Narrow)']
high_flux_df  = final_clean_df[final_clean_df['Filter_Type'] == 'High-Flux (Wide)']

# 3. 1행 3열의 넓은 도화지(Figure) 생성
fig, axes = plt.subplots(1, 3, figsize=(24, 8))

# ==========================================
# 차트 1: N2-Sieving 단독 (왼쪽)
# ==========================================
sns.scatterplot(
    data=n2_sieving_df, x='CO2_Affinity', y='Selectivity', 
    hue='Chemistry', hue_order=chemistry_order, style='Filter_Type',
    s=150, alpha=0.85, edgecolor='black', palette='tab20', ax=axes[0], legend=False
)
axes[0].set_title('(a) N2-Sieving Group Only', fontsize=16, fontweight='bold', pad=10)
axes[0].set_xscale('log')
axes[0].set_yscale('log')
axes[0].set_xlabel('CO2 Affinity (Capacity)', fontsize=12)
axes[0].set_ylabel('Selectivity (Purity)', fontsize=12)
axes[0].grid(True, which="both", ls="--", alpha=0.4)

# ==========================================
# 차트 2: High-Flux Bottleneck 단독 (가운데)
# ==========================================
sns.scatterplot(
    data=high_flux_df, x='CO2_Affinity', y='Selectivity', 
    hue='Chemistry', hue_order=chemistry_order, style='Filter_Type',
    s=150, alpha=0.85, edgecolor='black', palette='tab20', ax=axes[1], legend=False
)
axes[1].set_title('(b) High-Flux Bottleneck Group Only', fontsize=16, fontweight='bold', pad=10)
axes[1].set_xscale('log')
axes[1].set_yscale('log')
axes[1].set_xlabel('CO2 Affinity (Capacity)', fontsize=12)
axes[1].set_ylabel('Selectivity (Purity)', fontsize=12)
axes[1].grid(True, which="both", ls="--", alpha=0.4)

# ==========================================
# 차트 3: Combined 통합본 (오른쪽)
# ==========================================
sns.scatterplot(
    data=final_clean_df, x='CO2_Affinity', y='Selectivity', 
    hue='Chemistry', hue_order=chemistry_order, style='Filter_Type',
    s=150, alpha=0.85, edgecolor='black', palette='tab20', ax=axes[2]
)
axes[2].set_title('(c) Combined View', fontsize=16, fontweight='bold', pad=10)
axes[2].set_xscale('log')
axes[2].set_yscale('log')
axes[2].set_xlabel('CO2 Affinity (Capacity)', fontsize=12)
axes[2].set_ylabel('Selectivity (Purity)', fontsize=12)
axes[2].grid(True, which="both", ls="--", alpha=0.4)

# 오른쪽 차트에만 범례(Legend) 추가 및 그래프 밖으로 빼기
axes[2].legend(bbox_to_anchor=(1.05, 1), loc='upper left', borderaxespad=0., fontsize=11, title='Chemistry (Metal & OMS)', title_fontsize=12)

# 전체 레이아웃 조정 및 출력
plt.suptitle('Performance Frontier Analysis: CCUS Adsorbents', fontsize=22, fontweight='bold', y=1.05)
plt.tight_layout()
plt.show()

In [ ]:
# 현재까지 가공한 완벽한 원본 통합 데이터(df)를 피클로 영구 보존!
combined_df.to_pickle('MOF_Project_Master_Data.pkl')
print("✅ 마스터 데이터가 피클로 안전하게 저장되었습니다!")

In [ ]:
import pandas as pd
import numpy as np
import ast
import matplotlib.pyplot as plt
import seaborn as sns

# 1. 🌟 피클 대신 안전한 CSV 파일로 데이터 불러오기
try:
    df_n2 = pd.read_csv('N2_Sieving_Group_63_MOFs.csv')
    df_flux = pd.read_csv('High_Flux_Bottleneck_52_MOFs.csv')
    
    # 그룹 태그 다시 달아주기
    df_n2['Filter_Type'] = 'N2-Sieving (Narrow)'
    df_flux['Filter_Type'] = 'High-Flux (Wide)'
    
    # 두 그룹을 하나로 합치기
    combined_df = pd.concat([df_n2, df_flux], ignore_index=True)
    
except FileNotFoundError:
    print("앗! CSV 파일이 같은 폴더에 없습니다. 폴더 위치를 확인해주세요.")

# 2. 성능 지표(Affinity, Selectivity) 및 화학종 추출 함수 (어제와 동일)
def extract_metrics(gemc_data):
    try:
        if isinstance(gemc_data, str):
            gemc_data = ast.literal_eval(gemc_data)
        if isinstance(gemc_data, dict) and 'Widom' in gemc_data:
            widom = gemc_data['Widom']
            if len(widom) >= 2:
                co2_affinity = widom[0]
                n2_affinity = widom[1]
                selectivity = co2_affinity / n2_affinity if n2_affinity > 0 else 0
                return pd.Series([co2_affinity, selectivity])
    except:
        pass
    return pd.Series([np.nan, np.nan])

def extract_metal_category(metal_data):
    try:
        if isinstance(metal_data, str):
            metal_data = ast.literal_eval(metal_data)
        if isinstance(metal_data, dict):
            metal = metal_data.get('metal_type', 'Unknown')
            oms = metal_data.get('has_OMS', 'Unknown')
            return f"{metal} (OMS: {oms})"
    except:
        pass
    return "Unknown"

print("🔄 CSV 데이터 기반으로 마스터 데이터 복구 중...")

# 3. 데이터 가공 실행
# (CSV 저장 시 열 이름이 'metal'로 저장되었을 수 있으므로 자동 대응)
metal_col = 'metal_info' if 'metal_info' in combined_df.columns else 'metal'

combined_df[['CO2_Affinity', 'Selectivity']] = combined_df['GEMC_data'].apply(extract_metrics)
combined_df['Chemistry'] = combined_df[metal_col].apply(extract_metal_category)

# 4. 결측치 및 찌꺼기 데이터 정리
final_clean_df = combined_df.dropna(subset=['CO2_Affinity', 'Selectivity'])
final_clean_df = final_clean_df[final_clean_df['Chemistry'] != 'Unknown (OMS: Unknown)']

print(f"✅ 데이터 복구 완벽하게 성공! 총 {len(final_clean_df)}개의 MOF 데이터가 장전되었습니다.")

In [ ]:
import pandas as pd
import numpy as np
import ast
import re

# 1. 텍스트에서 등온선(Pressure, Uptake) 데이터를 추출하는 파서(Parser)
def extract_isotherm_data(gemc_str):
    try:
        # 문자열을 파이썬 딕셔너리로 변환
        if isinstance(gemc_str, str):
            gemc_dict = ast.literal_eval(gemc_str)
        else:
            gemc_dict = gemc_str
            
        gemc_lines = gemc_dict.get('GEMC', [])
        
        pressures = []
        uptakes = []
        
        # AIF 포맷에서 숫자 데이터 추출
        is_data_section = False
        for line in gemc_lines:
            # '_adsorp_amount' 태그 이후부터가 실제 숫자 데이터임
            if '_adsorp_amount' in line:
                is_data_section = True
                continue
                
            if is_data_section:
                # '0.1 4540 0.00285' 형태의 줄에서 숫자 추출
                parts = line.strip().split()
                if len(parts) >= 3:
                    try:
                        # parts[0]은 압력(Pa 또는 bar), parts[2]는 흡착량(mmol/g 등)
                        p_val = float(parts[0])
                        q_val = float(parts[2])
                        pressures.append(p_val)
                        uptakes.append(q_val)
                    except ValueError:
                        pass
        
        # 데이터가 없으면 None 반환
        if len(pressures) == 0:
            return None
            
        # 압력과 흡착량을 numpy 배열로 묶어서 반환
        return {'Pressure': np.array(pressures), 'Uptake': np.array(uptakes)}
        
    except Exception as e:
        return None

print("🔄 115개 정예 MOF에서 등온선(Isotherm) 데이터 추출을 시작합니다...")

# 2. 데이터 추출 실행 (final_clean_df가 있다고 가정)
# 만약 final_clean_df가 없다면 combined_df를 대신 넣으시면 됩니다.
final_clean_df['Isotherm_Data'] = final_clean_df['GEMC_data'].apply(extract_isotherm_data)

# 3. 데이터가 성공적으로 추출된 녀석들만 남기기
iast_ready_df = final_clean_df.dropna(subset=['Isotherm_Data']).copy()

print(f"✅ 추출 완료! 총 {len(final_clean_df)}개 중 {len(iast_ready_df)}개의 MOF에서 완벽한 등온선 데이터를 확보했습니다.")

# 4. 첫 번째 데이터 샘플 확인 (정상적으로 뽑혔는지 체크)
sample_mof = iast_ready_df.iloc[0]
print(f"\n🧪 [샘플 확인] MOF 화학종: {sample_mof['Chemistry']}")
print(f"   - 압력 포인트들: {sample_mof['Isotherm_Data']['Pressure'][:5]} ...")
print(f"   - 흡착량 포인트들: {sample_mof['Isotherm_Data']['Uptake'][:5]} ...")

In [ ]:
import pandas as pd
import numpy as np
from scipy.optimize import curve_fit
import warnings

# 경고 메시지 숨기기 (곡선 피팅 중 발생하는 자잘한 수학적 경고 무시)
warnings.filterwarnings('ignore')

print("🚀 2단계: 115개 MOF 대상 VSA 공정 성능(Working Capacity & Energy) 계산 시작...")

# 1. 랭뮤어(Langmuir) 방정식 정의
def langmuir_eq(P, q_m, K):
    return (q_m * K * P) / (1 + K * P)

# 결과를 저장할 리스트와 카운터 설정
results_list = []
failed_mofs = []  # 피팅에 실패한 MOF들의 이름을 담을 보관소
failed_count = 0  # 실패 횟수 카운터

# VSA 공정 기본 조건 설정
P_adsorption = 0.15  # 배가스 CO2 분압 (bar)
P_desorption = 0.01  # 진공 펌프 탈착 압력 (bar)
Temperature = 298.0  # K
R_gas = 8.314

# 2. iast_ready_df의 각 MOF에 대해 루프 돌며 계산 수행
for index, row in iast_ready_df.iterrows():
    mof_name = row['id']
    iso_data = row['Isotherm_Data']
    
    # 압력 단위 자동 보정 (Pa -> bar)
    P_raw = iso_data['Pressure']
    if max(P_raw) > 1000:  
        P_bar = P_raw / 100000.0
    else:
        P_bar = P_raw
        
    Uptake = iso_data['Uptake']
    
    try:
        # Step A: 랭뮤어 곡선 피팅
        popt, _ = curve_fit(langmuir_eq, P_bar, Uptake, p0=[max(Uptake)*1.2, 10.0], bounds=(0, np.inf))
        q_m_fit, K_fit = popt
        
        # Step B: 공정 조건에서의 흡착량 역산
        q_ads = langmuir_eq(P_adsorption, q_m_fit, K_fit)
        q_des = langmuir_eq(P_desorption, q_m_fit, K_fit)
        
        # Step C: 지표 계산
        working_capacity = q_ads - q_des
        recovery = (working_capacity / q_ads) * 100 if q_ads > 0 else 0
        performance_score = working_capacity * row['Selectivity'] 
        
        results_list.append({
            'MOF_ID': row['id'],
            'Chemistry': row['Chemistry'],
            'Filter_Type': row['Filter_Type'],
            'q_ads (Capacity at 0.15bar)': round(q_ads, 3),
            'q_des (Capacity at 0.01bar)': round(q_des, 3),
            'Working_Capacity (mmol/g)': round(working_capacity, 3),
            'Recovery (%)': round(recovery, 1),
            'Selectivity': round(row['Selectivity'], 2),
            'Overall_Performance_Score': round(performance_score, 2)
        })
        
    except Exception as e:
        # 🌟 실패한 경우 카운터를 올리고 명단에 추가
        failed_count += 1
        failed_mofs.append(mof_name)
        continue

# 3. 데이터프레임 변환 및 정렬
process_metrics_df = pd.DataFrame(results_list)
process_metrics_df = process_metrics_df.sort_values(by='Working_Capacity (mmol/g)', ascending=False).reset_index(drop=True)

# 4. 파일 저장
process_metrics_df.to_csv('Final_VSA_Process_Metrics.csv', index=False, encoding='utf-8-sig')

# 5. 최종 리포트 출력 (실패 내역 포함)
print("-" * 50)
print(f"✅ 총 분석 대상: {len(iast_ready_df)}개")
print(f"🎯 성공적으로 피팅된 MOF (Working Capacity 도출 완료): {len(process_metrics_df)}개")
print(f"⚠️ 랭뮤어 피팅 실패 (데이터 누락/비정상 등온선): {failed_count}개")

# 실패한 항목이 있다면 어떤 녀석들인지 살짝 보여주기
if failed_count > 0:
    print(f"\n[참고] 피팅 실패 명단 (최대 5개만 표시):")
    for bad_mof in failed_mofs[:5]:
        print(f" - {bad_mof}")

print("-" * 50)
print("💾 'Final_VSA_Process_Metrics.csv' 파일이 저장되었습니다.")

# 상위 5개 랭킹 보여주기
display(process_metrics_df.head(5))

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from collections import Counter
import warnings

# 1. 랭뮤어 방정식 (어제와 동일)
def langmuir_eq(P, q_m, K):
    return (q_m * K * P) / (1 + K * P)

error_types = []
failed_plots = [] # 그래프를 그리기 위해 실패한 데이터 일부를 수집

print("🔍 97개의 실패 원인을 정밀 분석 중입니다...\n")

# 2. 115개 데이터를 하나씩 뜯어보며 진단
for index, row in iast_ready_df.iterrows():
    mof_name = row['id']
    iso_data = row['Isotherm_Data']
    
    P_raw = iso_data['Pressure']
    Uptake = iso_data['Uptake']
    
    # 압력 단위 보정 (Pa -> bar)
    if max(P_raw) > 1000:  
        P_bar = P_raw / 100000.0
    else:
        P_bar = P_raw
        
    # --- [진단 1]: 점이 너무 적은 경우 ---
    if len(P_bar) < 3:
        error_types.append("에러 A: 데이터 포인트 부족 (3개 미만)")
        continue
        
    # --- [진단 2]: 아예 흡착을 안 하는 경우 ---
    if max(Uptake) <= 1e-5:
        error_types.append("에러 B: 유효 흡착량 0 (Zero Uptake)")
        if len(failed_plots) < 4:
            failed_plots.append((mof_name, P_bar, Uptake, "Zero Uptake"))
        continue

    # --- [진단 3]: 수학적 피팅 에러 ---
    try:
        # maxfev를 10000으로 늘려도 수렴하지 않으면 수학적으로 불가한 형태임
        popt, _ = curve_fit(langmuir_eq, P_bar, Uptake, p0=[max(Uptake)*1.2, 10.0], bounds=(0, np.inf), maxfev=10000)
        
        # 피팅이 '되긴 했지만' 물리적으로 말이 안 되는 경우 (최대 흡착량이 무한대로 발산)
        if popt[0] > max(Uptake) * 100:
            error_types.append("에러 C: 곡선이 아닌 완벽한 직선형 (q_m 발산)")
            if len(failed_plots) < 4:
                failed_plots.append((mof_name, P_bar, Uptake, "Perfectly Linear"))
        else:
            error_types.append("정상 피팅 성공")
            
    except RuntimeError:
        error_types.append("에러 D: 알고리즘 수렴 실패 (데이터 노이즈 심각)")
        if len(failed_plots) < 4:
            failed_plots.append((mof_name, P_bar, Uptake, "RuntimeError"))
    except Exception as e:
        err_name = type(e).__name__
        error_types.append(f"기타 에러: {err_name}")

# 3. 진단 결과 요약 리포트 출력
print("-" * 50)
print("📊 [실패 원인 통계 리포트]")
count = Counter(error_types)
for key, val in count.items():
    print(f" - {key}: {val}개")
print("-" * 50)

# 4. 실패한 등온선 시각화 (눈으로 확인하기)
if failed_plots:
    print("\n📉 대표적인 실패 등온선 샘플 4개를 시각화합니다.")
    fig, axes = plt.subplots(2, 2, figsize=(12, 10))
    axes = axes.flatten()
    
    for i, (name, p, q, err_type) in enumerate(failed_plots):
        if i >= 4: break
        ax = axes[i]
        ax.plot(p, q, marker='o', linestyle='-', color='red', alpha=0.7)
        ax.set_title(f"[{err_type}]\n{name[:30]}...", fontsize=10)
        ax.set_xlabel("Pressure (bar)")
        ax.set_ylabel("Uptake")
        ax.grid(True, ls='--', alpha=0.5)
        
    plt.tight_layout()
    plt.show()

In [ ]:
import pandas as pd
import numpy as np
from scipy.optimize import curve_fit
import warnings

warnings.filterwarnings('ignore')

print("🚀 [하이브리드 모드] 115개 MOF 대상 VSA 공정 성능 계산 시작...")

# 1. 랭뮤어 방정식 (곡선용)
def langmuir_eq(P, q_m, K):
    return (q_m * K * P) / (1 + K * P)

# 2. 헨리 방정식 (직선용)
def henry_eq(P, kH):
    return kH * P

results_list = []
P_adsorption = 0.15  # 배가스 CO2 분압 (bar)
P_desorption = 0.01  # 진공 펌프 탈착 압력 (bar)

# 3. 데이터 순회 및 하이브리드 피팅
for index, row in iast_ready_df.iterrows():
    iso_data = row['Isotherm_Data']
    
    P_raw = iso_data['Pressure']
    P_bar = P_raw / 100000.0 if max(P_raw) > 1000 else P_raw
    Uptake = iso_data['Uptake']
    
    # 데이터 포인트가 2개 이하인 심각한 에러는 건너뜀
    if len(P_bar) < 3 or max(Uptake) <= 1e-5:
        continue
        
    fitting_method = "Unknown"
    
    try:
        # [시도 1] 랭뮤어 곡선 피팅 시도
        popt, _ = curve_fit(langmuir_eq, P_bar, Uptake, p0=[max(Uptake)*1.2, 10.0], bounds=(0, np.inf), maxfev=10000)
        q_m_fit, K_fit = popt
        
        # 만약 q_m이 비정상적으로 발산하면(직선 형태면) 강제로 에러를 발생시켜 헨리로 넘김
        if q_m_fit > max(Uptake) * 100:
            raise ValueError("Linear Data")
            
        # 랭뮤어가 정상적이라면 계산
        q_ads = langmuir_eq(P_adsorption, q_m_fit, K_fit)
        q_des = langmuir_eq(P_desorption, q_m_fit, K_fit)
        fitting_method = "Langmuir (Curve)"
        
    except (RuntimeError, ValueError):
        # [시도 2] 랭뮤어 실패 시 헨리(Henry) 선형 피팅으로 우회 (Bypass)
        try:
            popt_h, _ = curve_fit(henry_eq, P_bar, Uptake)
            kH_fit = popt_h[0]
            
            q_ads = henry_eq(P_adsorption, kH_fit)
            q_des = henry_eq(P_desorption, kH_fit)
            fitting_method = "Henry (Linear)"
        except:
            continue # 선형 피팅조차 실패하면 진짜 버림
            
    # 최종 지표 계산
    working_capacity = q_ads - q_des
    recovery = (working_capacity / q_ads) * 100 if q_ads > 0 else 0
    performance_score = working_capacity * row['Selectivity']
    
    results_list.append({
        'MOF_ID': row['id'],
        'Chemistry': row['Chemistry'],
        'Filter_Type': row['Filter_Type'],
        'Fitting_Model': fitting_method,
        'q_ads_0.15bar': round(q_ads, 3),
        'q_des_0.01bar': round(q_des, 3),
        'Working_Capacity_mmol/g': round(working_capacity, 3),
        'Recovery_%': round(recovery, 1),
        'Selectivity': round(row['Selectivity'], 2),
        'Performance_Score': round(performance_score, 2)
    })

# 4. 결과 정리 및 파일 저장
process_metrics_df = pd.DataFrame(results_list)
process_metrics_df = process_metrics_df.sort_values(by='Performance_Score', ascending=False).reset_index(drop=True)

process_metrics_df.to_csv('Hybrid_VSA_Process_Metrics.csv', index=False, encoding='utf-8-sig')

print("-" * 50)
print(f"✅ 하이브리드 피팅 완료! 성공한 MOF 개수: {len(process_metrics_df)}개")
print("💾 'Hybrid_VSA_Process_Metrics.csv' 저장 완료.")
print("-" * 50)

# 상위 5개 결과 확인
display(process_metrics_df.head(5))

In [ ]:
import pandas as pd
import numpy as np
from scipy.optimize import curve_fit
import warnings

warnings.filterwarnings('ignore')

print("🚀 [하이브리드 모드] 115개 MOF 대상 VSA 공정 성능 계산 시작...")

# 1. 랭뮤어 방정식 (곡선용)
def langmuir_eq(P, q_m, K):
    return (q_m * K * P) / (1 + K * P)

# 2. 헨리 방정식 (직선용)
def henry_eq(P, kH):
    return kH * P

results_list = []
P_adsorption = 0.15  # 배가스 CO2 분압 (bar)
P_desorption = 0.01  # 진공 펌프 탈착 압력 (bar)

# 3. 데이터 순회 및 하이브리드 피팅
for index, row in iast_ready_df.iterrows():
    iso_data = row['Isotherm_Data']
    
    P_raw = iso_data['Pressure']
    P_bar = P_raw / 100000.0 if max(P_raw) > 1000 else P_raw
    Uptake = iso_data['Uptake']
    
    # 데이터 포인트가 2개 이하인 심각한 에러는 건너뜀
    if len(P_bar) < 3 or max(Uptake) <= 1e-5:
        continue
        
    fitting_method = "Unknown"
    
    try:
        # [시도 1] 랭뮤어 곡선 피팅 시도
        popt, _ = curve_fit(langmuir_eq, P_bar, Uptake, p0=[max(Uptake)*1.2, 10.0], bounds=(0, np.inf), maxfev=10000)
        q_m_fit, K_fit = popt
        
        # 만약 q_m이 비정상적으로 발산하면(직선 형태면) 강제로 에러를 발생시켜 헨리로 넘김
        if q_m_fit > max(Uptake) * 100:
            raise ValueError("Linear Data")
            
        # 랭뮤어가 정상적이라면 계산
        q_ads = langmuir_eq(P_adsorption, q_m_fit, K_fit)
        q_des = langmuir_eq(P_desorption, q_m_fit, K_fit)
        fitting_method = "Langmuir (Curve)"
        
    except (RuntimeError, ValueError):
        # [시도 2] 랭뮤어 실패 시 헨리(Henry) 선형 피팅으로 우회 (Bypass)
        try:
            popt_h, _ = curve_fit(henry_eq, P_bar, Uptake)
            kH_fit = popt_h[0]
            
            q_ads = henry_eq(P_adsorption, kH_fit)
            q_des = henry_eq(P_desorption, kH_fit)
            fitting_method = "Henry (Linear)"
        except:
            continue # 선형 피팅조차 실패하면 진짜 버림
            
    # 최종 지표 계산
    working_capacity = q_ads - q_des
    recovery = (working_capacity / q_ads) * 100 if q_ads > 0 else 0
    performance_score = working_capacity * row['Selectivity']
    
    results_list.append({
        'MOF_ID': row['id'],
        'Chemistry': row['Chemistry'],
        'Filter_Type': row['Filter_Type'],
        'Fitting_Model': fitting_method,
        'q_ads_0.15bar': round(q_ads, 3),
        'q_des_0.01bar': round(q_des, 3),
        'Working_Capacity_mmol/g': round(working_capacity, 3),
        'Recovery_%': round(recovery, 1),
        'Selectivity': round(row['Selectivity'], 2),
        'Performance_Score': round(performance_score, 2)
    })

# 4. 결과 정리 및 파일 저장
process_metrics_df = pd.DataFrame(results_list)
process_metrics_df = process_metrics_df.sort_values(by='Performance_Score', ascending=False).reset_index(drop=True)

process_metrics_df.to_csv('Hybrid_VSA_Process_Metrics.csv', index=False, encoding='utf-8-sig')

print("-" * 50)
print(f"✅ 하이브리드 피팅 완료! 성공한 MOF 개수: {len(process_metrics_df)}개")
print("💾 'Hybrid_VSA_Process_Metrics.csv' 저장 완료.")
print("-" * 50)

# 상위 5개 결과 확인
display(process_metrics_df.head(5))

In [ ]:
import pandas as pd
import numpy as np
from scipy.optimize import curve_fit
import warnings

# 경고 메시지 숨기기
warnings.filterwarnings('ignore')

print("🚀 [버그 수정본] 115개 MOF 대상 VSA 공정 성능 계산 시작...")

# 1. 랭뮤어 방정식 & 헨리 방정식
def langmuir_eq(P, q_m, K):
    return (q_m * K * P) / (1 + K * P)

def henry_eq(P, kH):
    return kH * P

results_list = []
P_adsorption = 0.15  # 배가스 CO2 분압 (bar)
P_desorption = 0.01  # 진공 펌프 탈착 압력 (bar)

# 2. 데이터 순회
for index, row in iast_ready_df.iterrows():
    iso_data = row['Isotherm_Data']
    
    # 🌟 [버그 수정 핵심] 조건문 삭제! 무조건 10만으로 나누어 확실하게 bar 단위로 변환
    P_raw = iso_data['Pressure']
    P_bar = P_raw / 100000.0  
    Uptake = iso_data['Uptake']
    
    if len(P_bar) < 3 or max(Uptake) <= 1e-5:
        continue
        
    fitting_method = "Unknown"
    
    try:
        # [시도 1] 랭뮤어 곡선 피팅
        popt, _ = curve_fit(langmuir_eq, P_bar, Uptake, p0=[max(Uptake)*1.2, 10.0], bounds=(0, np.inf), maxfev=10000)
        q_m_fit, K_fit = popt
        
        if q_m_fit > max(Uptake) * 100:
            raise ValueError("Linear Data")
            
        q_ads = langmuir_eq(P_adsorption, q_m_fit, K_fit)
        q_des = langmuir_eq(P_desorption, q_m_fit, K_fit)
        fitting_method = "Langmuir (Curve)"
        
    except (RuntimeError, ValueError):
        # [시도 2] 헨리(Henry) 선형 외삽법 (Linear Extrapolation)
        try:
            popt_h, _ = curve_fit(henry_eq, P_bar, Uptake)
            kH_fit = popt_h[0]
            
            q_ads = henry_eq(P_adsorption, kH_fit)
            q_des = henry_eq(P_desorption, kH_fit)
            fitting_method = "Henry (Linear)"
        except:
            continue
            
    # 최종 지표 산출
    working_capacity = q_ads - q_des
    recovery = (working_capacity / q_ads) * 100 if q_ads > 0 else 0
    performance_score = working_capacity * row['Selectivity']
    
    results_list.append({
        'MOF_ID': row['id'],
        'Chemistry': row['Chemistry'],
        'Filter_Type': row['Filter_Type'],
        'Fitting_Model': fitting_method,
        'q_ads_0.15bar': round(q_ads, 3),
        'q_des_0.01bar': round(q_des, 3),
        'Working_Capacity_mmol/g': round(working_capacity, 3),
        'Recovery_%': round(recovery, 1),
        'Selectivity': round(row['Selectivity'], 2),
        'Performance_Score': round(performance_score, 2)
    })

# 3. 결과 정리 및 파일 저장
process_metrics_df = pd.DataFrame(results_list)

# Performance Score (유효 흡착량 * 선택도 = 종합점수) 기준으로 1등부터 줄 세우기
process_metrics_df = process_metrics_df.sort_values(by='Performance_Score', ascending=False).reset_index(drop=True)

process_metrics_df.to_csv('Fixed_VSA_Process_Metrics.csv', index=False, encoding='utf-8-sig')

print("-" * 50)
print(f"✅ 압력 단위(Pa->bar) 버그 완벽 수정! 성공한 MOF 개수: {len(process_metrics_df)}개")
print("💾 'Fixed_VSA_Process_Metrics.csv' 저장 완료.")
print("-" * 50)

# 영광의 상위 5개 결과 확인
display(process_metrics_df.head(5))

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

print("🔍 [정밀 분석] Langmuir 곡선 피팅이 완료된 검증된 MOF만 추출합니다...\n")

# 1. Langmuir 모델로 피팅된 데이터만 필터링
langmuir_df = process_metrics_df[process_metrics_df['Fitting_Model'] == 'Langmuir (Curve)'].copy()

# 2. 유효 흡착량 기준으로 재정렬
langmuir_df = langmuir_df.sort_values(by='Working_Capacity_mmol/g', ascending=False).reset_index(drop=True)

# 3. 데이터 추이 요약 통계 (현실적인 수치 범위 확인)
print("-" * 50)
print(f"✅ 검증된 Langmuir MOF 개수: {len(langmuir_df)}개")
print("-" * 50)
print("[핵심 지표 통계 요약]")
print(langmuir_df[['q_ads_0.15bar', 'Working_Capacity_mmol/g', 'Selectivity', 'Performance_Score']].describe().round(3))
print("-" * 50)

# 4. 성능 분포 시각화 (Working Capacity vs Selectivity)
plt.figure(figsize=(10, 6))
sns.scatterplot(
    data=langmuir_df, 
    x='Working_Capacity_mmol/g', 
    y='Selectivity', 
    hue='Chemistry', 
    s=200, alpha=0.8, edgecolor='black', palette='tab10'
)
plt.title('Performance Frontier: Verified Langmuir MOFs Only', fontsize=16, fontweight='bold')
plt.xlabel('Working Capacity (mmol/g) -> Recovery & Energy', fontsize=12)
plt.ylabel('Selectivity (CO2/N2) -> Purity', fontsize=12)
plt.grid(True, ls='--', alpha=0.5)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

# 5. 영광의 Langmuir 진짜 1등 ~ 5등 명단 출력
print("\n🏆 [검증된 Langmuir 랭킹 Top 5]")
display(langmuir_df[['MOF_ID', 'Chemistry', 'q_ads_0.15bar', 'Working_Capacity_mmol/g', 'Selectivity', 'Performance_Score']].head(5))

# 원하신다면 이 정예 멤버들만 따로 저장
langmuir_df.to_csv('Verified_Langmuir_MOFs.csv', index=False, encoding='utf-8-sig')

In [ ]:
import pandas as pd

# 1. 아까 저장한 Langmuir 데이터 불러오기
df = pd.read_csv("Verified_Langmuir_MOFs.csv")

# 2. 물리적 한계치 필터 적용: 0.15 bar에서 흡착량이 12 mmol/g 이하인 현실적인 데이터만 남김
physical_limit = 12.0
realistic_df = df[df['q_ads_0.15bar'] <= physical_limit].copy()

print(f"🛡️ 물리적 필터 통과: {len(realistic_df)}개의 완벽한 타겟 확보!\n")

# 3. 진짜 1등 공개!
print("🏆 [현실성 100% 반영된 진짜 최종 랭킹 Top 5]")
display(realistic_df[['Chemistry', 'Filter_Type', 'q_ads_0.15bar', 'Working_Capacity_mmol/g', 'Selectivity', 'Performance_Score']].head(5))

In [ ]:
import pandas as pd
import numpy as np
import ast
from scipy.optimize import curve_fit

print("🚀 물리적 한계(Pore Volume)가 적용된 '유사-랭뮤어(Pseudo-Langmuir)' 외삽 모델 가동 중...\n")

# 1. 헨리 법칙 피팅용 함수
def henry_eq(P, kH): return kH * P

# 2. GPV(기공 부피) 추출 함수
def extract_gpv(zeopp_data):
    try:
        if isinstance(zeopp_data, str):
            z_dict = ast.literal_eval(zeopp_data)
        else:
            z_dict = zeopp_data
        return z_dict.get('GPV', 0.5) # 없으면 일반적인 MOF 평균 부피 0.5 가정
    except:
        return 0.5

# 원본 데이터(combined_df 또는 final_clean_df)에서 Zeopp 컬럼을 가져와 GPV 계산
# (현재 환경에 맞게 원본 데이터프레임을 병합해줍니다)
if 'Zeopp' not in iast_ready_df.columns:
    # Zeopp 데이터가 담긴 원본 CSV 로드
    df_raw = pd.concat([pd.read_csv("High_Flux_Bottleneck_52_MOFs.csv"), pd.read_csv("Bottleneck_MOFS.csv")]).drop_duplicates(subset=['id'])
    iast_ready_df = pd.merge(iast_ready_df, df_raw[['id', 'Zeopp']], on='id', how='left')

iast_ready_df['GPV'] = iast_ready_df['Zeopp'].apply(extract_gpv)

results_list = []
P_adsorption = 0.15
P_desorption = 0.01

for index, row in iast_ready_df.iterrows():
    iso_data = row['Isotherm_Data']
    P_raw = iso_data['Pressure']
    P_bar = P_raw / 100000.0  # 단위 버그 방지 (무조건 bar로 변환)
    Uptake = iso_data['Uptake']
    GPV = row['GPV']
    
    if len(P_bar) < 3 or max(Uptake) <= 1e-5:
        continue
        
    # [핵심] Gurvich Rule 기반 절대 최대 흡착량(q_max) 계산 (단위: mmol/g)
    # CO2 액체 밀도 ~1.03 g/cm3, 분자량 44.01 g/mol
    q_max_physical = (GPV * 1.03 / 44.01) * 1000 
    if q_max_physical <= 0: q_max_physical = 10.0 # 예외 처리
    
    try:
        # [단계 1] 초저압 데이터로 헨리 상수(kH) 도출
        popt_h, _ = curve_fit(henry_eq, P_bar, Uptake)
        kH_fit = popt_h[0]
        
        # [단계 2] 유사-랭뮤어 모델 합성 (q_max와 kH의 결합)
        # 랭뮤어 모델: q = (q_m * K * P) / (1 + K * P)
        # 저압에서 q ~ q_m * K * P 이므로, kH = q_m * K. 따라서 K = kH / q_m
        K_synthetic = kH_fit / q_max_physical
        
        def pseudo_langmuir(P):
            return (q_max_physical * K_synthetic * P) / (1.0 + K_synthetic * P)
        
        q_ads = pseudo_langmuir(P_adsorption)
        q_des = pseudo_langmuir(P_desorption)
        fitting_method = "Pseudo-Langmuir (GPV Capped)"
        
    except:
        continue
        
    # 지표 산출
    working_capacity = q_ads - q_des
    recovery = (working_capacity / q_ads) * 100 if q_ads > 0 else 0
    performance_score = working_capacity * row['Selectivity']
    
    results_list.append({
        'MOF_ID': row['id'],
        'Chemistry': row['Chemistry'],
        'GPV (cm3/g)': round(GPV, 3),
        'Physical_q_max': round(q_max_physical, 2),
        'q_ads_0.15bar': round(q_ads, 3),
        'q_des_0.01bar': round(q_des, 3),
        'Working_Capacity_mmol/g': round(working_capacity, 3),
        'Selectivity': round(row['Selectivity'], 2),
        'Performance_Score': round(performance_score, 2)
    })

# 결과 정리
calibrated_df = pd.DataFrame(results_list)
calibrated_df = calibrated_df.sort_values(by='Working_Capacity_mmol/g', ascending=False).reset_index(drop=True)
calibrated_df.to_csv('Fully_Calibrated_PseudoLangmuir_Metrics.csv', index=False, encoding='utf-8-sig')

print(f"✅ 물리적 보정 완료! 총 {len(calibrated_df)}개의 MOF 데이터가 완벽하게 복구되었습니다.")
display(calibrated_df[['Chemistry', 'GPV (cm3/g)', 'Physical_q_max', 'q_ads_0.15bar', 'Working_Capacity_mmol/g', 'Selectivity']].head(10))

In [ ]:
import pandas as pd
import numpy as np
import ast
from scipy.optimize import curve_fit
import matplotlib.pyplot as plt
import seaborn as sns
import re

print("🚀 1. 원본 데이터 로드 및 Pseudo-Langmuir 열역학 보정 가동 중...")

# ==========================================
# 1. 물리적 보정 및 데이터 정제 함수 정의
# ==========================================
def process_group(filepath, group_name):
    df = pd.read_csv(filepath)
    results = []
    
    for idx, row in df.iterrows():
        # 1. GPV(기공 부피) 추출
        try:
            z_dict = ast.literal_eval(row['Zeopp']) if isinstance(row['Zeopp'], str) else row['Zeopp']
            GPV = z_dict.get('GPV', 0.5)
        except: GPV = 0.5
            
        # 2. 등온선(Isotherm) 추출
        try:
            gemc_dict = ast.literal_eval(row['GEMC_data']) if isinstance(row['GEMC_data'], str) else row['GEMC_data']
            lines = gemc_dict.get('GEMC', [])
            pressures, uptakes = [], []
            data_on = False
            for line in lines:
                if '_adsorp_amount' in line:
                    data_on = True; continue
                if data_on:
                    parts = line.strip().split()
                    if len(parts) >= 3:
                        try:
                            pressures.append(float(parts[0]))
                            uptakes.append(float(parts[2]))
                        except: pass
            P_bar = np.array(pressures) / 100000.0  # Pa -> bar 변환
            Uptake = np.array(uptakes)
        except: continue
            
        if len(P_bar) < 2 or max(Uptake) <= 1e-5: continue
            
        # 3. 물리적 최대 한계치 (q_max) 계산 (CO2 밀도 1.03 g/cm3 반영)
        q_max_physical = (GPV * 1.03 / 44.01) * 1000 
        if q_max_physical <= 0: q_max_physical = 5.0
            
        # 4. 헨리 법칙 피팅 및 Pseudo-Langmuir 보정
        def henry_eq(P, kH): return kH * P
        try:
            popt_h, _ = curve_fit(henry_eq, P_bar, Uptake)
            kH_fit = popt_h[0]
            K_synthetic = kH_fit / q_max_physical
            def pseudo_langmuir(P):
                return (q_max_physical * K_synthetic * P) / (1.0 + K_synthetic * P)
            q_ads = pseudo_langmuir(0.15)
            q_des = pseudo_langmuir(0.01)
        except: continue
            
        # 5. 깔끔한 ID(Readable MOF ID) 생성
        try:
            id_dict = ast.literal_eval(row['id'])
            mofid = id_dict.get('mofid-v1', '')
            parts = mofid.split(';')
            code_6 = parts[1][:6].upper() if len(parts) > 1 else "XXXXXX"
            metal_str = row.get('metal', 'Unknown')
            metal_clean = str(metal_str).split(' ')[0] if pd.notna(metal_str) else "X"
            if metal_clean == "nan": metal_clean = "X"
            readable_id = f"MOF-{code_6} ({metal_clean})"
        except: readable_id = "Unknown MOF"
            
        working_capacity = q_ads - q_des
        sel = row.get('Selectivity', 0)
        
        results.append({
            'Readable_MOF_ID': readable_id,
            'Group': group_name,
            'Chemistry (Metal)': metal_str,
            'Zeopp GPV (cm3/g)': GPV,
            'Physical q_max (mmol/g)': q_max_physical,
            'Henry Constant (kH)': kH_fit,
            'q_ads at 0.15 bar': q_ads,
            'q_des at 0.01 bar': q_des,
            'Working Capacity (mmol/g)': working_capacity,
            'Selectivity': sel,
            'Performance Score': working_capacity * sel
        })
        
    res_df = pd.DataFrame(results)
    return res_df.sort_values(by='Performance Score', ascending=False)

# 데이터 가공 실행 (파일명은 로컬 환경에 맞게 수정하세요)
df_hf = process_group("High_Flux_Bottleneck_52_MOFs.csv", "High Flux (Wide)")
df_n2 = process_group("N2_Sieving_Group_63_MOFs.csv", "N2 Sieving (Narrow)")
df_total = pd.concat([df_hf, df_n2], ignore_index=True)

# 엑셀 저장
df_hf.to_excel("High_Flux_Final_Metrics.xlsx", index=False)
df_n2.to_excel("N2_Sieving_Final_Metrics.xlsx", index=False)
print("✅ 엑셀 데이터 저장 완료!")

# ==========================================
# 2. 산점도 (Performance Frontier) 그래프 생성
# ==========================================
print("📈 2. 산점도 그래픽 생성 중...")
plt.style.use('default')
sns.set_context("talk", font_scale=0.9)
sns.set_style("ticks")
fig, ax = plt.subplots(figsize=(10, 8))

sns.scatterplot(
    data=df_total, x='Working Capacity (mmol/g)', y='Selectivity',
    hue='Group', style='Group', s=200, alpha=0.85,
    palette={'High Flux (Wide)': '#2980b9', 'N2 Sieving (Narrow)': '#e67e22'}, # 컬러 매칭
    edgecolor='black', ax=ax
)

# Top 5 라벨링
top5 = df_total.nlargest(5, 'Performance Score')
for idx, row in top5.iterrows():
    short_id = row['Readable_MOF_ID'].split()[0]
    ax.annotate(short_id, (row['Working Capacity (mmol/g)'], row['Selectivity']),
                xytext=(8, 8), textcoords='offset points', fontsize=11, fontweight='bold',
                bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="gray", alpha=0.9))

ax.set_xlabel('CO$_2$ Working Capacity (mmol/g) [0.15 $\\rightarrow$ 0.01 bar]', fontweight='bold')
ax.set_ylabel('CO$_2$/N$_2$ Selectivity', fontweight='bold')
ax.set_title('Performance Frontier of Top CCUS MOF Candidates', fontweight='bold', pad=20)
ax.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.savefig('MOF_Performance_Frontier.png', dpi=300)
print("✅ 고해상도 그래프 저장 완료!")

# ==========================================
# 3. 색상이 일치하는 PDF 표(Table) 생성 (WeasyPrint 필요)
# ==========================================
try:
    from weasyprint import HTML
    print("📄 3. 색상 매칭 PDF 테이블 생성 중...")
    
    def create_pdf(df, out_pdf, title, header_color):
        df_print = df.copy()
        for col in df_print.select_dtypes(include=['float64']).columns:
            df_print[col] = df_print[col].apply(lambda x: f"{x:.3f}")
            
        html_table = df_print.drop(columns=['Group']).to_html(index=False, border=0, classes='styled-table')
        
        html_content = f"""
        <html>
        <head>
        <meta charset="UTF-8">
        <style>
            @page {{ size: A4 landscape; margin: 10mm; }}
            body {{ font-family: 'Arial', sans-serif; font-size: 9pt; color: #333; }}
            h1 {{ color: {header_color}; text-align: center; border-bottom: 2px solid {header_color}; padding-bottom: 10px; }}
            .styled-table {{ width: 100%; border-collapse: collapse; margin: 15px 0; text-align: center; }}
            .styled-table thead tr {{ background-color: {header_color}; color: white; }}
            .styled-table th, .styled-table td {{ padding: 8px; border: 1px solid #ddd; }}
            .styled-table tbody tr:nth-of-type(even) {{ background-color: #f3f3f3; }}
        </style>
        </head>
        <body>
            <h1>{title}</h1>
            {html_table}
        </body>
        </html>
        """
        with open("temp.html", "w", encoding="utf-8") as f: f.write(html_content)
        HTML("temp.html").write_pdf(out_pdf)

    create_pdf(df_hf, 'High_Flux_Performance_Table.pdf', 'High Flux (Wide) Group - Final Metrics', '#2980b9')
    create_pdf(df_n2, 'N2_Sieving_Performance_Table.pdf', 'N2 Sieving (Narrow) Group - Final Metrics', '#e67e22')
    print("✅ PDF 테이블 생성 완료!")
    
except ImportError:
    print("⚠️ PDF 생성 생략: 'weasyprint' 라이브러리가 설치되지 않았습니다. (pip install weasyprint)")

print("🎉 모든 결론 도출 및 파일 생성이 완료되었습니다!")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import ast
import warnings
warnings.filterwarnings('ignore')

print("🚀 1. 데이터 로드 및 금속(Metal) 색상 동기화 중...")

# 1. 엑셀 데이터 로드 (이전 단계에서 생성된 Final 엑셀 파일 기준)
df_hf = pd.read_excel('High_Flux_Final_Metrics.xlsx')
df_n2 = pd.read_excel('N2_Sieving_Final_Metrics.xlsx')
df_hf['Group'] = 'High Flux (Wide)'
df_n2['Group'] = 'N2 Sieving (Narrow)'
df_total = pd.concat([df_hf, df_n2], ignore_index=True)

# 2. 금속(Metal)과 OMS 정보 파싱
def parse_metal(val):
    try:
        d = ast.literal_eval(val) if isinstance(val, str) else val
        return d.get('metal_type', 'Unknown'), d.get('has_OMS', 'Unknown')
    except:
        return "Unknown", "Unknown"

df_total[['Metal', 'OMS']] = df_total['Chemistry (Metal)'].apply(lambda x: pd.Series(parse_metal(x)))
df_total['OMS'] = df_total['OMS'].map({'Yes': 'With OMS', 'No': 'Without OMS', 'Unknown': 'Unknown'})

# 🌟 3. 색상 팔레트 고정 (산점도와 표의 색상을 100% 일치시키기 위한 핵심 로직)
unique_metals = df_total['Metal'].unique()
# Seaborn의 Set1 팔레트 헥스코드
set1_hex = ['#e41a1c', '#377eb8', '#4daf4a', '#984ea3', '#ff7f00', '#ffff33', '#a65628', '#f781bf', '#999999', '#a6cee3', '#1f78b4', '#b2df8a']
# 금속별로 고유 색상을 딕셔너리로 영구 할당
metal_colors = {metal: set1_hex[i % len(set1_hex)] for i, metal in enumerate(unique_metals)}

# ==========================================
# 4. 색상 동기화 산점도 (Metal & OMS) 생성
# ==========================================
print("📈 2. 산점도 그래픽 생성 중...")
plt.style.use('default')
sns.set_context("talk", font_scale=0.9)
sns.set_style("ticks")
fig, ax = plt.subplots(figsize=(12, 9))

# 팔레트를 아까 만든 metal_colors 딕셔너리로 강제 지정
sns.scatterplot(
    data=df_total, x='Working Capacity (mmol/g)', y='Selectivity',
    hue='Metal', style='OMS', palette=metal_colors,
    markers={'With OMS': 'o', 'Without OMS': 'X', 'Unknown': 's'},
    s=250, alpha=0.85, edgecolor='black', ax=ax
)

# Top 5 라벨링
top5 = df_total.nlargest(5, 'Performance Score')
for idx, row in top5.iterrows():
    short_id = str(row['Readable_MOF_ID']).split()[0]
    ax.annotate(short_id, (row['Working Capacity (mmol/g)'], row['Selectivity']),
                xytext=(8, 8), textcoords='offset points', fontsize=10, fontweight='bold',
                bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="gray", alpha=0.9))

ax.set_xlabel('CO$_2$ Working Capacity (mmol/g)', fontweight='bold')
ax.set_ylabel('CO$_2$/N$_2$ Selectivity', fontweight='bold')
ax.set_title('Performance by Metal Center and OMS Presence', fontweight='bold', pad=20)
ax.grid(True, linestyle='--', alpha=0.6)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', frameon=True)
plt.tight_layout()
plt.savefig('MOF_Metal_OMS_Sync.png', dpi=300)
print("✅ 금속 색상 동기화 그래프(PNG) 저장 완료!")

# ==========================================
# 5. 색상 동기화 HTML 표 생성 (weasyprint 대체 완벽 호환법)
# ==========================================
print("📄 3. 색상 매칭 HTML 테이블 생성 중...")

# 표의 Metal 셀 배경색을 칠해주는 함수
def style_metal_column(val):
    color = metal_colors.get(val, '#ffffff')
    # 노란색 계열 배경에는 검은 글씨, 어두운 배경에는 흰 글씨 적용
    text_color = '#000000' if color in ['#ffff33', '#b2df8a'] else '#ffffff'
    return f'background-color: {color}; color: {text_color}; font-weight: bold;'

def create_styled_html(df_subset, filename, title):
    df_print = df_subset.copy()
    
    # 숫자 포맷 깔끔하게 정리
    for col in df_print.select_dtypes(include=['float64']).columns:
        df_print[col] = df_print[col].apply(lambda x: f"{x:.3f}")

    # 보기 힘든 딕셔너리 원본 컬럼 삭제 및 순서 재배치
    if 'Chemistry (Metal)' in df_print.columns:
        df_print = df_print.drop(columns=['Chemistry (Metal)'])
    cols = ['Readable_MOF_ID', 'Metal', 'OMS'] + [c for c in df_print.columns if c not in ['Readable_MOF_ID', 'Metal', 'OMS', 'Group']]
    df_print = df_print[cols]

    # Pandas Styler를 이용한 아름다운 HTML 표 렌더링
    styled = df_print.style\
        .map(style_metal_column, subset=['Metal'])\
        .set_caption(f"<h2 style='font-family: Arial; color: #2c3e50;'>{title}</h2>")\
        .set_table_styles([
            {'selector': 'th', 'props': [('background-color', '#2c3e50'), ('color', 'white'), ('padding', '12px'), ('font-family', 'Arial')]},
            {'selector': 'td', 'props': [('padding', '10px'), ('border', '1px solid #ddd'), ('text-align', 'center'), ('font-family', 'Arial')]},
            {'selector': 'table', 'props': [('border-collapse', 'collapse'), ('width', '100%'), ('box-shadow', '0 2px 3px rgba(0,0,0,0.1)')]}
        ])
    
    with open(filename, 'w', encoding='utf-8') as f:
        f.write(styled.to_html())

create_styled_html(df_total[df_total['Group'] == 'High Flux (Wide)'], 'High_Flux_Color_Table.html', 'High Flux Group - Metal Colored Metrics')
create_styled_html(df_total[df_total['Group'] == 'N2 Sieving (Narrow)'], 'N2_Sieving_Color_Table.html', 'N2 Sieving Group - Metal Colored Metrics')

print("✅ 완벽하게 색상이 동기화된 HTML 표 생성 완료!")
print("\n💡 [PDF 변환 팁]")
print("생성된 'High_Flux_Color_Table.html' 파일을 크롬/엣지 브라우저에서 더블 클릭하여 여신 후, [Ctrl + P (인쇄)] -> [대상: PDF로 저장], [설정: 배경 그래픽 포함 체크] 하시면 완벽한 PDF 보고서가 완성됩니다!")

In [ ]:
# 4. 공통 그래프 생성 및 개별 PDF 저장 함수 (범례 간격 완벽 조정 버전)
def save_scatter_pdf(df, title, filename):
    plt.style.use('default')
    sns.set_context("talk", font_scale=2.0) 
    sns.set_style("ticks")

    fig, ax = plt.subplots(figsize=(24, 18))

    sns.scatterplot(
        data=df, x='CO2_Affinity', y='Selectivity', 
        hue='Metal', style='Filter_Type',
        palette=metal_colors,
        s=600, alpha=0.85, edgecolor='black', ax=ax
    )

    ax.set_title(title, fontsize=36, fontweight='bold', pad=25)
    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.set_xlabel('CO$_2$ Affinity (Capacity)', fontsize=28, fontweight='bold')
    ax.set_ylabel('CO$_2$/N$_2$ Selectivity (Purity)', fontsize=28, fontweight='bold')
    ax.grid(True, which="both", ls="--", alpha=0.4)
    ax.tick_params(labelsize=22)

    # 🌟 [수정된 부분] 범례(Legend) 박스 안의 공간/간격 정밀 튜닝
    ax.legend(
        bbox_to_anchor=(1.02, 1), loc='upper left', borderaxespad=0., 
        title='Metal & Filter', title_fontsize=28, fontsize=24, 
        markerscale=1.2,       # 마커 크기가 너무 무식하게 크지 않도록 살짝 축소
        labelspacing=1.5,      # 🌟 항목 위아래(세로) 줄 간격 대폭 확대
        handletextpad=0.8,     # 🌟 동그라미(기호)와 글씨 사이의 가로 여백 확대
        borderpad=1.2,         # 🌟 범례 네모 박스 안쪽의 전체 테두리 여백 확대
        frameon=True,          # 테두리 선 활성화
        edgecolor='black'      # 범례 박스 테두리 색상
    )

    plt.tight_layout()
    plt.savefig(filename.replace('.pdf', '.png'), dpi=600, bbox_inches='tight')
    plt.savefig(filename, dpi=600, bbox_inches='tight')
    plt.close()
    
    print(f"✅ 대형 캔버스 & 범례 간격 조정 완료: {filename}")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

print("🚀 1. 금속(Metal) 색상 동기화 및 초고해상도(600DPI) 세팅 중...")

# 1. Chemistry 열에서 금속 기호만 추출 (예: "Cu (OMS: Yes)" -> "Cu")
def get_metal(chemistry_str):
    if pd.isna(chemistry_str): return "Unknown"
    return str(chemistry_str).split(' ')[0]

final_clean_df['Metal'] = final_clean_df['Chemistry'].apply(get_metal)

# 🌟 2. 이전 최종 그래프와 100% 동일한 색상 팔레트 하드코딩
set1_hex = ['#e41a1c', '#377eb8', '#4daf4a', '#984ea3', '#ff7f00', '#ffff33', '#a65628', '#f781bf', '#999999', '#a6cee3', '#1f78b4', '#b2df8a']
unique_metals = sorted(final_clean_df['Metal'].unique())
metal_colors = {metal: set1_hex[i % len(set1_hex)] for i, metal in enumerate(unique_metals)}

# 3. 개별 그룹 데이터 분리
n2_sieving_df = final_clean_df[final_clean_df['Filter_Type'] == 'N2-Sieving (Narrow)']
high_flux_df  = final_clean_df[final_clean_df['Filter_Type'] == 'High-Flux (Wide)']

# 4. 공통 그래프 생성 및 개별 PDF 저장 함수 (대형 캔버스 & 범례 간격 완벽 조정 버전)
def save_scatter_pdf(df, title, filename):
    plt.style.use('default')
    # 캔버스가 커진 만큼 기본 폰트 스케일 2.0배 확대
    sns.set_context("talk", font_scale=2.0) 
    sns.set_style("ticks")

    # 🌟 [수정 포인트 1] 3배 거대한 도화지 생성 (기존 8x6 -> 24x18)
    fig, ax = plt.subplots(figsize=(24, 18))

    # 🌟 [수정 포인트 2] 마커 크기 대폭 확대 (s=600)
    sns.scatterplot(
        data=df, x='CO2_Affinity', y='Selectivity', 
        hue='Metal', style='Filter_Type',
        palette=metal_colors,
        s=600, alpha=0.85, edgecolor='black', ax=ax
    )

    # 🌟 [수정 포인트 3] 축 제목 및 틱(눈금) 크기 비례 확대
    ax.set_title(title, fontsize=36, fontweight='bold', pad=25)
    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.set_xlabel('CO$_2$ Affinity (Capacity)', fontsize=28, fontweight='bold')
    ax.set_ylabel('CO$_2$/N$_2$ Selectivity (Purity)', fontsize=28, fontweight='bold')
    ax.grid(True, which="both", ls="--", alpha=0.4)
    ax.tick_params(labelsize=22)

    # 🌟 [수정 포인트 4] 범례(Legend) 박스 안의 줄 간격 및 여백 정밀 튜닝 (겹침 완벽 해결)
    ax.legend(
        bbox_to_anchor=(1.02, 1), loc='upper left', borderaxespad=0., 
        title='Metal & Filter', title_fontsize=28, fontsize=24, 
        markerscale=1.2,       # 마커 크기가 텍스트를 가리지 않게 적절히 축소
        labelspacing=1.5,      # 항목 위아래(세로) 줄 간격 대폭 확보 (겹침 방지)
        handletextpad=0.8,     # 동그라미(기호)와 글씨 사이의 가로 여백 확보
        borderpad=1.2,         # 네모 박스 안쪽의 전체 테두리 여백 확보
        frameon=True,          # 테두리 선 활성화
        edgecolor='black'      # 범례 박스 테두리 색상
    )

    plt.tight_layout()
    
    # 🌟 [수정 포인트 5] 초고해상도(600 DPI)로 PNG 및 PDF 동시 저장
    plt.savefig(filename.replace('.pdf', '.png'), dpi=600, bbox_inches='tight')
    plt.savefig(filename, dpi=600, bbox_inches='tight')
    plt.close()
    
    print(f"✅ 대형 캔버스 & 범례 조정 600DPI 저장 완료: {filename}")

# ==========================================
# 5. 각각의 개별 PDF 및 PNG 파일 생성 실행
# ==========================================
print("📈 개별 초고해상도 PDF/PNG 그래프 생성을 시작합니다...\n")

# 차트 (a): N2-Sieving 단독
save_scatter_pdf(
    n2_sieving_df, 
    '(a) N2-Sieving Group Only', 
    'Frontier_a_N2_Sieving_Only_Large.pdf'
)

# 차트 (b): High-Flux Bottleneck 단독
save_scatter_pdf(
    high_flux_df, 
    '(b) High-Flux Bottleneck Group Only', 
    'Frontier_b_High_Flux_Only_Large.pdf'
)

# 차트 (c): Combined 통합본
save_scatter_pdf(
    final_clean_df, 
    '(c) Combined View', 
    'Frontier_c_Combined_Large.pdf'
)

print("\n🎉 네이처(Nature) 급 초고해상도 3종 세트가 완벽한 비율로 추출되었습니다!")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import ast
import warnings
warnings.filterwarnings('ignore')

print("🚀 1. 데이터 로드 및 금속(Metal) 색상 동기화 중...")

# 1. 엑셀 데이터 로드 (이전 단계에서 생성된 Final 엑셀 파일 기준)
df_hf = pd.read_excel('High_Flux_Final_Metrics.xlsx')
df_n2 = pd.read_excel('N2_Sieving_Final_Metrics.xlsx')
df_hf['Group'] = 'High Flux (Wide)'
df_n2['Group'] = 'N2 Sieving (Narrow)'
df_total = pd.concat([df_hf, df_n2], ignore_index=True)

# 2. 금속(Metal)과 OMS 정보 파싱
def parse_metal(val):
    try:
        d = ast.literal_eval(val) if isinstance(val, str) else val
        return d.get('metal_type', 'Unknown'), d.get('has_OMS', 'Unknown')
    except:
        return "Unknown", "Unknown"

df_total[['Metal', 'OMS']] = df_total['Chemistry (Metal)'].apply(lambda x: pd.Series(parse_metal(x)))
df_total['OMS'] = df_total['OMS'].map({'Yes': 'With OMS', 'No': 'Without OMS', 'Unknown': 'Unknown'})

# 🌟 3. 색상 팔레트 고정 (산점도와 표의 색상을 100% 일치시키기 위한 핵심 로직)
unique_metals = df_total['Metal'].unique()
# Seaborn의 Set1 팔레트 헥스코드
set1_hex = ['#e41a1c', '#377eb8', '#4daf4a', '#984ea3', '#ff7f00', '#ffff33', '#a65628', '#f781bf', '#999999', '#a6cee3', '#1f78b4', '#b2df8a']
# 금속별로 고유 색상을 딕셔너리로 영구 할당
metal_colors = {metal: set1_hex[i % len(set1_hex)] for i, metal in enumerate(unique_metals)}

# ==========================================
# 4. 색상 동기화 산점도 (Metal & OMS) 생성
# ==========================================
print("📈 2. 산점도 그래픽 생성 중...")
plt.style.use('default')
sns.set_context("talk", font_scale=0.9)
sns.set_style("ticks")
fig, ax = plt.subplots(figsize=(12, 9))

# 팔레트를 아까 만든 metal_colors 딕셔너리로 강제 지정
sns.scatterplot(
    data=df_total, x='Working Capacity (mmol/g)', y='Selectivity',
    hue='Metal', style='OMS', palette=metal_colors,
    markers={'With OMS': 'o', 'Without OMS': 'X', 'Unknown': 's'},
    s=250, alpha=0.85, edgecolor='black', ax=ax
)

# Top 5 라벨링
top5 = df_total.nlargest(5, 'Performance Score')
for idx, row in top5.iterrows():
    short_id = str(row['Readable_MOF_ID']).split()[0]
    ax.annotate(short_id, (row['Working Capacity (mmol/g)'], row['Selectivity']),
                xytext=(8, 8), textcoords='offset points', fontsize=10, fontweight='bold',
                bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="gray", alpha=0.9))

ax.set_xlabel('CO$_2$ Working Capacity (mmol/g)', fontweight='bold')
ax.set_ylabel('CO$_2$/N$_2$ Selectivity', fontweight='bold')
ax.set_title('Performance by Metal Center and OMS Presence', fontweight='bold', pad=20)
ax.grid(True, linestyle='--', alpha=0.6)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', frameon=True)
plt.tight_layout()
plt.savefig('MOF_Metal_OMS_Sync.png', dpi=300)
print("✅ 금속 색상 동기화 그래프(PNG) 저장 완료!")

# ==========================================
# 5. 색상 동기화 HTML 표 생성 (weasyprint 대체 완벽 호환법)
# ==========================================
print("📄 3. 색상 매칭 HTML 테이블 생성 중...")

# 표의 Metal 셀 배경색을 칠해주는 함수
def style_metal_column(val):
    color = metal_colors.get(val, '#ffffff')
    # 노란색 계열 배경에는 검은 글씨, 어두운 배경에는 흰 글씨 적용
    text_color = '#000000' if color in ['#ffff33', '#b2df8a'] else '#ffffff'
    return f'background-color: {color}; color: {text_color}; font-weight: bold;'

def create_styled_html(df_subset, filename, title):
    df_print = df_subset.copy()
    
    # 숫자 포맷 깔끔하게 정리
    for col in df_print.select_dtypes(include=['float64']).columns:
        df_print[col] = df_print[col].apply(lambda x: f"{x:.3f}")

    # 보기 힘든 딕셔너리 원본 컬럼 삭제 및 순서 재배치
    if 'Chemistry (Metal)' in df_print.columns:
        df_print = df_print.drop(columns=['Chemistry (Metal)'])
    cols = ['Readable_MOF_ID', 'Metal', 'OMS'] + [c for c in df_print.columns if c not in ['Readable_MOF_ID', 'Metal', 'OMS', 'Group']]
    df_print = df_print[cols]

    # Pandas Styler를 이용한 아름다운 HTML 표 렌더링
    styled = df_print.style\
        .map(style_metal_column, subset=['Metal'])\
        .set_caption(f"<h2 style='font-family: Arial; color: #2c3e50;'>{title}</h2>")\
        .set_table_styles([
            {'selector': 'th', 'props': [('background-color', '#2c3e50'), ('color', 'white'), ('padding', '12px'), ('font-family', 'Arial')]},
            {'selector': 'td', 'props': [('padding', '10px'), ('border', '1px solid #ddd'), ('text-align', 'center'), ('font-family', 'Arial')]},
            {'selector': 'table', 'props': [('border-collapse', 'collapse'), ('width', '100%'), ('box-shadow', '0 2px 3px rgba(0,0,0,0.1)')]}
        ])
    
    with open(filename, 'w', encoding='utf-8') as f:
        f.write(styled.to_html())

create_styled_html(df_total[df_total['Group'] == 'High Flux (Wide)'], 'High_Flux_Color_Table.html', 'High Flux Group - Metal Colored Metrics')
create_styled_html(df_total[df_total['Group'] == 'N2 Sieving (Narrow)'], 'N2_Sieving_Color_Table.html', 'N2 Sieving Group - Metal Colored Metrics')

print("✅ 완벽하게 색상이 동기화된 HTML 표 생성 완료!")
print("\n💡 [PDF 변환 팁]")
print("생성된 'High_Flux_Color_Table.html' 파일을 크롬/엣지 브라우저에서 더블 클릭하여 여신 후, [Ctrl + P (인쇄)] -> [대상: PDF로 저장], [설정: 배경 그래픽 포함 체크] 하시면 완벽한 PDF 보고서가 완성됩니다!")

In [ ]:
import pandas as pd
import requests
import ast
from tqdm import tqdm
import time
import warnings
warnings.filterwarnings('ignore')

print("🔍 1. 원본 데이터에서 115개 MOF의 DOI 추출 중...")
# 원본 CSV 파일 로드 (경로는 환경에 맞게 수정)
df_hf = pd.read_csv("High_Flux_Bottleneck_52_MOFs.csv")
df_n2 = pd.read_csv("N2_Sieving_Group_63_MOFs.csv")
df_orig = pd.concat([df_hf, df_n2], ignore_index=True)

def get_doi_and_id(row):
    # Readable ID 추출
    try:
        id_dict = ast.literal_eval(row['id'])
        parts = id_dict.get('mofid-v1', '').split(';')
        code_6 = parts[1][:6].upper() if len(parts) > 1 else "XXXXXX"
        metal_clean = str(row.get('metal', 'X')).split(' ')[0]
        if metal_clean == "nan": metal_clean = "X"
        readable_id = f"MOF-{code_6} ({metal_clean})"
    except: readable_id = "Unknown MOF"
    
    # DOI 추출
    try:
        ref_dict = ast.literal_eval(row['reference'])
        doi = ref_dict.get('DOI', '')
    except: doi = ""
    
    return pd.Series([readable_id, doi])

df_orig[['Readable_MOF_ID', 'DOI']] = df_orig.apply(get_doi_and_id, axis=1)

# 중복 논문 제거 (동일 논문에서 여러 MOF가 나온 경우 API 중복 호출 방지)
unique_dois = df_orig[df_orig['DOI'] != '']['DOI'].unique()
print(f"총 {len(unique_dois)}개의 고유 논문 DOI가 발견되었습니다. API 텍스트 마이닝을 시작합니다...\n")

# 🌟 NLP 소수성/친수성 키워드 사전
hydrophobic_keywords = ['hydrophobic', 'water-stable', 'water stable', 'moisture-resistant', 'water resistant', 'fluorinat', 'water repulsion']
hydrophilic_keywords = ['hydrophilic', 'water sorption', 'water uptake', 'water soluble', 'degrad', 'unstable in water']

doi_results = {}

# Crossref API 통신 및 초록(Abstract) 분석
for doi in tqdm(unique_dois, desc="논문 초록(Abstract) 분석 진행률"):
    url = f"https://api.crossref.org/works/{doi}"
    try:
        # API 권장 헤더 설정
        headers = {'User-Agent': 'MOF_Research_NLP/1.0 (mailto:researcher@university.edu)'}
        response = requests.get(url, headers=headers, timeout=7)
        
        if response.status_code == 200:
            data = response.json()
            abstract = data['message'].get('abstract', '').lower()
            title = data['message'].get('title', [''])[0].lower()
            text = abstract + " " + title # 제목과 초록을 합쳐서 검색
            
            is_phobic = any(kw in text for kw in hydrophobic_keywords)
            is_philic = any(kw in text for kw in hydrophilic_keywords)
            
            # 텍스트 판별 로직
            if is_phobic and is_philic:
                status = "Mixed / Tunable (부분 소수성 개질 가능)"
            elif is_phobic:
                status = "Hydrophobic 🎯 (강력한 소수성 후보 - 상용화 1순위)"
            elif is_philic:
                status = "Hydrophilic 💧 (수분 취약/흡착형)"
            else:
                status = "Not Explicitly Mentioned (초록에 언급 없음)"
        else:
            status = "Abstract Not Available"
    except:
        status = "Connection Timeout"
        
    doi_results[doi] = status
    time.sleep(0.5) # Crossref 서버 과부하 방지를 위한 0.5초 딜레이

# 3. 마이닝 결과를 데이터프레임에 매핑
df_orig['Literature_Hydrophobicity'] = df_orig['DOI'].map(doi_results)

# 4. 보기 좋게 정리하여 엑셀 추출
final_columns = ['Readable_MOF_ID', 'Filter_Type', 'Chemistry', 'DOI', 'Literature_Hydrophobicity']
df_result = df_orig[['Readable_MOF_ID', 'DOI', 'Literature_Hydrophobicity']].drop_duplicates(subset=['Readable_MOF_ID'])

# 최종 파일 저장
df_result.to_excel("MOF_Literature_Hydrophobicity_Check.xlsx", index=False)
print("\n✅ 문헌 기반 소수성 마이닝이 완벽하게 끝났습니다!")
print("결과물 [MOF_Literature_Hydrophobicity_Check.xlsx]를 열어서 '🎯' 표시가 붙은 진짜 상용화 타겟을 확인하세요!")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import ast

print("🚀 1. 고성능 데이터 로드 및 Tier 1 교차 검증 중...")

# 1. 기존 고성능 115개 데이터 로드
try:
    df_hf = pd.read_csv("High_Flux_Bottleneck_52_MOFs.csv")
    df_n2 = pd.read_csv("N2_Sieving_Group_63_MOFs.csv")
    df_perf = pd.concat([df_hf, df_n2], ignore_index=True)
except FileNotFoundError:
    df_perf = pd.concat([pd.read_csv("MAGIC_WINDOW_63_MOFS.xlsx - MAGIC_WINDOW_63_MOFS.csv"), 
                         pd.read_csv("Bottleneck_MOFS.xlsx - Bottleneck_MOFS.csv")], ignore_index=True)

# 2. 메타데이터 추출
def extract_metadata(row):
    try:
        id_dict = ast.literal_eval(row['id'])
        mofid = id_dict.get('mofid-v1', '')
        short_id = mofid.split(';')[1][:6].upper() if ';' in mofid else "XXXXXX"
    except: short_id = "Unknown"
        
    try:
        metal_dict = ast.literal_eval(row['metal'])
        metal_type = metal_dict.get('metal_type', 'Unknown')
        has_oms = metal_dict.get('has_OMS', 'Unknown')
    except: 
        metal_type, has_oms = "Unknown", "Unknown"
        
    try:
        pld = ast.literal_eval(row['Zeopp']).get('PLD', 0.0)
    except: pld = 0.0
    
    return pd.Series([short_id, metal_type, has_oms, pld])

df_perf[['Short_ID', 'Metal', 'Has_OMS', 'PLD']] = df_perf.apply(extract_metadata, axis=1)
df_perf = df_perf.sort_values('Selectivity', ascending=False).drop_duplicates(subset=['Short_ID'])

# 🌟 [수정 완료] 영구적인 색상 동기화 (Tier 1 필터링 전 '전체 금속' 기준으로 팔레트 고정)
set1_hex = ['#e41a1c', '#377eb8', '#4daf4a', '#984ea3', '#ff7f00', '#ffff33', '#a65628', '#f781bf', '#999999', '#a6cee3', '#1f78b4', '#b2df8a']
unique_all_metals = sorted(df_perf['Metal'].unique())
universal_metal_colors = {metal: set1_hex[i % len(set1_hex)] for i, metal in enumerate(unique_all_metals)}

# 3. Tier 1 생존자 필터링
tier1_condition = (
    df_perf['Metal'].isin(['Zn', 'Co']) & 
    (df_perf['Has_OMS'] == 'No') & 
    (df_perf['PLD'] >= 3.2) & (df_perf['PLD'] <= 3.8)
)
df_tier1 = df_perf[tier1_condition].copy()

print(f"🎯 최종 Tier 1 생존 타겟: {len(df_tier1)}개 추출 완료!\n")

# ==========================================
# 4. 초고해상도 플롯 시각화 (색상 동기화 적용)
# ==========================================
print("📈 2. 초고해상도(600DPI) 산점도 그래프 생성을 시작합니다...")

plt.style.use('default')
sns.set_context("talk", font_scale=2.0) 
sns.set_style("ticks")

fig, ax = plt.subplots(figsize=(24, 18))

x_col = 'CO2_Affinity' if 'CO2_Affinity' in df_tier1.columns else 'Working_Capacity'

# 🌟 앞서 세팅한 universal_metal_colors를 그대로 주입
sns.scatterplot(
    data=df_tier1, x=x_col, y='Selectivity', 
    hue='Metal', palette=universal_metal_colors,
    s=800, alpha=0.85, edgecolor='black', ax=ax, zorder=5
)

top_5 = df_tier1.nlargest(5, 'Selectivity')
for _, row in top_5.iterrows():
    ax.text(
        row[x_col] * 1.05, row['Selectivity'], 
        row['Short_ID'], 
        fontsize=18, fontweight='bold', color='black'
    )

ax.set_title('Tier 1 Ultimate Targets: High-Humidity Survival Frontier', fontsize=36, fontweight='bold', pad=25)
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('CO$_2$ Working Capacity (mmol/g)', fontsize=28, fontweight='bold')
ax.set_ylabel('CO$_2$/N$_2$ Selectivity', fontsize=28, fontweight='bold')
ax.grid(True, which="both", ls="--", alpha=0.4)
ax.tick_params(labelsize=22)

ax.legend(
    bbox_to_anchor=(1.02, 1), loc='upper left', borderaxespad=0., 
    title='Core Metal (No OMS)', title_fontsize=28, fontsize=24, 
    markerscale=1.2, labelspacing=1.5, handletextpad=0.8, 
    borderpad=1.2, frameon=True, edgecolor='black'
)

plt.tight_layout()
filename = 'Tier1_Humidity_Survival_Frontier_ColorSynced.pdf'
plt.savefig(filename.replace('.pdf', '.png'), dpi=600, bbox_inches='tight')
plt.savefig(filename, dpi=600, bbox_inches='tight')
plt.close()

df_tier1.to_excel("Tier1_Ultimate_Targets_ColorSynced.xlsx", index=False)
print(f"🎉 성공! 모든 논문 피규어와 100% 색상이 일치하는 그래프가 출력되었습니다: {filename}")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import ast
import glob

print("🚀 1. 최종 성능 지표 데이터 로드 및 Tier 1+2 병합 중...")

# 1. 성능 지표(Metrics)가 모두 계산된 최종 파일 로드
# 엑셀 또는 CSV 포맷에 모두 대응하도록 유연하게 로드
hf_files = glob.glob("*High_Flux_Final_Metrics*")
n2_files = glob.glob("*N2_Sieving_Final_Metrics*")

try:
    df_hf = pd.read_csv(hf_files[0]) if hf_files[0].endswith('.csv') else pd.read_excel(hf_files[0])
    df_n2 = pd.read_csv(n2_files[0]) if n2_files[0].endswith('.csv') else pd.read_excel(n2_files[0])
    df_perf = pd.concat([df_hf, df_n2], ignore_index=True)
except IndexError:
    print("❌ 최종 지표 파일(Final_Metrics)을 찾을 수 없습니다. 파일명을 확인해주세요!")
    raise

# 2. Chemistry 열에서 금속(Metal)과 OMS 여부 추출
def parse_chemistry(chem_str):
    chem_str = str(chem_str)
    # 텍스트 마이닝 방식으로 안전하게 추출
    metal = 'Zn' if "'metal_type': 'Zn'" in chem_str else ('Co' if "'metal_type': 'Co'" in chem_str else 'Other')
    has_oms = 'No' if "'has_OMS': 'No'" in chem_str else 'Yes'
    return pd.Series([metal, has_oms])

df_perf[['Metal', 'Has_OMS']] = df_perf['Chemistry (Metal)'].apply(parse_chemistry)

# 3. 🌟 Tier 1 & Tier 2 조건 적용 (Zn/Co 이면서 No OMS 인 모든 타겟)
# PLD 조건(매직 윈도우)을 해제하여 Tier 2까지 모두 흡수합니다.
tier1_2_cond = df_perf['Metal'].isin(['Zn', 'Co']) & (df_perf['Has_OMS'] == 'No')
df_survivors = df_perf[tier1_2_cond].copy()

# 4. 🧹 중복 검증 및 제거 (Deduplication)
# Readable_MOF_ID에서 앞부분의 6자리 식별자(Short ID)만 추출하여 동일 뼈대 중복을 찾습니다.
df_survivors['Short_ID'] = df_survivors['Readable_MOF_ID'].apply(lambda x: str(x).split(' ')[0])

# 동일 뼈대(Short ID) 중복 시, Performance Score(종합 성능)가 가장 높은 단 1개만 남김
df_survivors = df_survivors.sort_values('Performance Score', ascending=False).drop_duplicates(subset=['Short_ID'])

print(f"🎯 중복 제거 후 최종 Tier 1 & 2 생존 타겟: {len(df_survivors)}개 확보 완료!\n")

# 5. 요청하신 정확한 열(Column) 구성으로 엑셀 추출
# 열 이름은 기존에 만들어진 파일의 형태를 따릅니다.
export_columns = [
    'Readable_MOF_ID', 'Chemistry (Metal)', 'Zeopp GPV (cm3/g)', 
    'Physical q_max (mmol/g)', 'Henry Constant (kH)', 'q_ads at 0.15 bar', 
    'q_des at 0.01 bar', 'Working Capacity (mmol/g)', 'Selectivity', 'Performance Score'
]

# 누락된 열이 발생하지 않도록 교차 검증 후 저장
available_cols = [col for col in export_columns if col in df_survivors.columns]
df_export = df_survivors[available_cols]

excel_filename = "Tier1_Tier2_Anti_Humidity_Targets.xlsx"
df_export.to_excel(excel_filename, index=False)
print(f"📄 세부 열역학 지표 엑셀 파일 저장 완료: {excel_filename}")


# ==========================================
# 6. 초고해상도 플롯 시각화 (색상 동기화 완벽 적용)
# ==========================================
print("📈 2. 초고해상도(600DPI) 산점도 그래프 생성을 시작합니다...")

plt.style.use('default')
sns.set_context("talk", font_scale=2.0) 
sns.set_style("ticks")
fig, ax = plt.subplots(figsize=(24, 18))

# 이전 115개 그래프와 100% 동일한 고유 색상 매핑
set1_hex = ['#e41a1c', '#377eb8', '#4daf4a', '#984ea3', '#ff7f00', '#ffff33', '#a65628', '#f781bf', '#999999', '#a6cee3', '#1f78b4', '#b2df8a']
# 원본 데이터의 금속 알파벳 순 정렬에 맞춰 색상 고정
metal_colors = {'Zn': set1_hex[11], 'Co': set1_hex[2]} # 이전 정렬 기준에 맞춘 수동/자동 매핑 적용 가능. 여기서는 범용 Set1 사용
# 안전을 위해 df_perf 전체 기준으로 팔레트 재구성
universal_metals = sorted(df_perf['Metal'].unique())
metal_palette = {m: set1_hex[i % len(set1_hex)] for i, m in enumerate(universal_metals)}

# 산점도 렌더링
sns.scatterplot(
    data=df_survivors, x='Working Capacity (mmol/g)', y='Selectivity', 
    hue='Metal', palette=metal_palette,
    s=800, alpha=0.85, edgecolor='black', ax=ax, zorder=5
)

# 상위 5개 이름표 달기
top_5 = df_survivors.nlargest(5, 'Performance Score')
for _, row in top_5.iterrows():
    ax.text(
        row['Working Capacity (mmol/g)'] * 1.05, row['Selectivity'], 
        row['Short_ID'], 
        fontsize=18, fontweight='bold', color='black'
    )

ax.set_title('Tier 1 & 2: High-Humidity Survival Frontier (No OMS)', fontsize=36, fontweight='bold', pad=25)
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('CO$_2$ Working Capacity (mmol/g)', fontsize=28, fontweight='bold')
ax.set_ylabel('CO$_2$/N$_2$ Selectivity', fontsize=28, fontweight='bold')
ax.grid(True, which="both", ls="--", alpha=0.4)
ax.tick_params(labelsize=22)

ax.legend(
    bbox_to_anchor=(1.02, 1), loc='upper left', borderaxespad=0., 
    title='Core Metal (No OMS)', title_fontsize=28, fontsize=24, 
    markerscale=1.2, labelspacing=1.5, handletextpad=0.8, 
    borderpad=1.2, frameon=True, edgecolor='black'
)

plt.tight_layout()
plot_filename = 'Tier1_Tier2_Survival_Frontier.pdf'
plt.savefig(plot_filename.replace('.pdf', '.png'), dpi=600, bbox_inches='tight')
plt.savefig(plot_filename, dpi=600, bbox_inches='tight')
plt.close()

print(f"🎉 성공! Tier 1 & 2 결합 그래프 출력 완료: {plot_filename}")

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

print("🔍 1. 전체 2,737개 메타데이터 로드 및 Tier 1 스크리닝 중...")

file_path = "CR_meta_data_SI (1).json"

with open(file_path, 'r', encoding='utf-8') as f:
    data = json.load(f)

# 메타데이터 추출 및 리스트화
mof_list = []
for key, val in data.items():
    mofid_v1 = val.get("id", {}).get("mofid-v1", "Unknown")
    short_id = mofid_v1.split(';')[1][:6].upper() if ';' in mofid_v1 else mofid_v1[:15]
    
    zeopp = val.get("Zeopp", {})
    pld = zeopp.get("PLD", 0.0)
    lcd = zeopp.get("LCD", 0.0)
    
    metal_info = val.get("metal", {})
    metal_type = metal_info.get("metal_type", "Unknown")
    has_oms = metal_info.get("has_OMS", "Unknown")
    
    topology = val.get("CrystalNets", {}).get("all_nodes", "Unknown")
    doi = val.get("reference", {}).get("DOI", "Unknown")
    
    mof_list.append({
        "Short_ID": short_id,
        "Metal": metal_type,
        "Has_OMS": has_oms,
        "PLD (Limiting Pore)": pld,
        "LCD (Largest Cavity)": lcd,
        "Topology": topology,
        "DOI": doi
    })

df_all = pd.DataFrame(mof_list)

# 🌟 Tier 1 순수 스크리닝 (Zn/Co + No OMS + 매직 윈도우)
tier1_cond = (
    df_all['Metal'].isin(['Zn', 'Co']) & 
    (df_all['Has_OMS'] == 'No') & 
    (df_all['PLD (Limiting Pore)'] >= 3.2) & 
    (df_all['PLD (Limiting Pore)'] <= 3.8)
)
df_tier1_full = df_all[tier1_cond].copy()

# 중복 뼈대(Short_ID 기준) 제거 (선택사항이나 깔끔한 결과를 위해 적용)
df_tier1_unique = df_tier1_full.drop_duplicates(subset=['Short_ID'])

print(f"🎯 최종 Tier 1 고유 원석: {len(df_tier1_unique)}개 추출 완료!\n")

# 엑셀 파일로 저장
excel_filename = "All_Database_Tier1_Screening_Results.xlsx"
df_tier1_unique.to_excel(excel_filename, index=False)
print(f"📄 전체 리스트 엑셀 저장 완료: {excel_filename}")

# ==========================================
# 기하학적 스크리닝 결과 시각화 (PLD vs LCD)
# ==========================================
print("📈 2. 기하학적 분포(PLD vs LCD) 산점도 그래프 생성을 시작합니다...")

plt.style.use('default')
sns.set_context("talk", font_scale=1.5) 
sns.set_style("ticks")
fig, ax = plt.subplots(figsize=(14, 10))

# 금속 색상 지정 (통일성 유지)
set1_hex = ['#e41a1c', '#377eb8', '#4daf4a', '#984ea3', '#ff7f00', '#ffff33', '#a65628', '#f781bf']
metal_colors = {'Zn': set1_hex[1], 'Co': set1_hex[0]} # Zn 파랑, Co 빨강

sns.scatterplot(
    data=df_tier1_unique, 
    x='PLD (Limiting Pore)', y='LCD (Largest Cavity)', 
    hue='Metal', palette=metal_colors,
    s=400, alpha=0.8, edgecolor='black', ax=ax
)

# 매직 윈도우 영역 강조 표시 (배경색 칠하기)
ax.axvspan(3.2, 3.8, color='green', alpha=0.1, label='Magic Window Zone')

ax.set_title('Tier 1 Screening Results (2,737 MOFs Database)', fontsize=24, fontweight='bold', pad=20)
ax.set_xlabel('Limiting Pore Diameter (PLD, $\AA$)', fontsize=18, fontweight='bold')
ax.set_ylabel('Largest Cavity Diameter (LCD, $\AA$)', fontsize=18, fontweight='bold')
ax.grid(True, which="major", ls="--", alpha=0.5)

# 범례 설정
handles, labels = ax.get_legend_handles_labels()
ax.legend(
    handles=handles, labels=labels,
    bbox_to_anchor=(1.02, 1), loc='upper left', 
    title='Properties', title_fontsize=16, fontsize=14, 
    frameon=True, edgecolor='black'
)

plt.tight_layout()
plot_filename = 'Tier1_Screening_Geometric_Distribution.pdf'
plt.savefig(plot_filename.replace('.pdf', '.png'), dpi=600, bbox_inches='tight')
plt.savefig(plot_filename, dpi=600, bbox_inches='tight')
plt.close()

print(f"🎉 성공! 스크리닝 분포 그래프 출력 완료: {plot_filename}")

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

print("🔍 1. 전체 2,737개 메타데이터 로드 및 Tier 1 스크리닝 중...")

file_path = "CR_meta_data_SI .json"

with open(file_path, 'r', encoding='utf-8') as f:
    data = json.load(f)

# 메타데이터 추출 및 리스트화
mof_list = []
for key, val in data.items():
    mofid_v1 = val.get("id", {}).get("mofid-v1", "Unknown")
    short_id = mofid_v1.split(';')[1][:6].upper() if ';' in mofid_v1 else mofid_v1[:15]
    
    zeopp = val.get("Zeopp", {})
    pld = zeopp.get("PLD", 0.0)
    lcd = zeopp.get("LCD", 0.0)
    
    metal_info = val.get("metal", {})
    metal_type = metal_info.get("metal_type", "Unknown")
    has_oms = metal_info.get("has_OMS", "Unknown")
    
    topology = val.get("CrystalNets", {}).get("all_nodes", "Unknown")
    doi = val.get("reference", {}).get("DOI", "Unknown")
    
    mof_list.append({
        "Short_ID": short_id,
        "Metal": metal_type,
        "Has_OMS": has_oms,
        "PLD (Limiting Pore)": pld,
        "LCD (Largest Cavity)": lcd,
        "Topology": topology,
        "DOI": doi
    })

df_all = pd.DataFrame(mof_list)

# 🌟 Tier 1 순수 스크리닝 (Zn/Co + No OMS + 매직 윈도우)
tier1_cond = (
    df_all['Metal'].isin(['Zn', 'Co']) & 
    (df_all['Has_OMS'] == 'No') & 
    (df_all['PLD (Limiting Pore)'] >= 3.2) & 
    (df_all['PLD (Limiting Pore)'] <= 3.8)
)
df_tier1_full = df_all[tier1_cond].copy()

# 중복 뼈대(Short_ID 기준) 제거 (선택사항이나 깔끔한 결과를 위해 적용)
df_tier1_unique = df_tier1_full.drop_duplicates(subset=['Short_ID'])

print(f"🎯 최종 Tier 1 고유 원석: {len(df_tier1_unique)}개 추출 완료!\n")

# 엑셀 파일로 저장
excel_filename = "All_Database_Tier1_Screening_Results.xlsx"
df_tier1_unique.to_excel(excel_filename, index=False)
print(f"📄 전체 리스트 엑셀 저장 완료: {excel_filename}")

# ==========================================
# 기하학적 스크리닝 결과 시각화 (PLD vs LCD)
# ==========================================
print("📈 2. 기하학적 분포(PLD vs LCD) 산점도 그래프 생성을 시작합니다...")

plt.style.use('default')
sns.set_context("talk", font_scale=1.5) 
sns.set_style("ticks")
fig, ax = plt.subplots(figsize=(14, 10))

# 금속 색상 지정 (통일성 유지)
set1_hex = ['#e41a1c', '#377eb8', '#4daf4a', '#984ea3', '#ff7f00', '#ffff33', '#a65628', '#f781bf']
metal_colors = {'Zn': set1_hex[1], 'Co': set1_hex[0]} # Zn 파랑, Co 빨강

sns.scatterplot(
    data=df_tier1_unique, 
    x='PLD (Limiting Pore)', y='LCD (Largest Cavity)', 
    hue='Metal', palette=metal_colors,
    s=400, alpha=0.8, edgecolor='black', ax=ax
)

# 매직 윈도우 영역 강조 표시 (배경색 칠하기)
ax.axvspan(3.2, 3.8, color='green', alpha=0.1, label='Magic Window Zone')

ax.set_title('Tier 1 Screening Results (2,737 MOFs Database)', fontsize=24, fontweight='bold', pad=20)
ax.set_xlabel('Limiting Pore Diameter (PLD, $\AA$)', fontsize=18, fontweight='bold')
ax.set_ylabel('Largest Cavity Diameter (LCD, $\AA$)', fontsize=18, fontweight='bold')
ax.grid(True, which="major", ls="--", alpha=0.5)

# 범례 설정
handles, labels = ax.get_legend_handles_labels()
ax.legend(
    handles=handles, labels=labels,
    bbox_to_anchor=(1.02, 1), loc='upper left', 
    title='Properties', title_fontsize=16, fontsize=14, 
    frameon=True, edgecolor='black'
)

plt.tight_layout()
plot_filename = 'Tier1_Screening_Geometric_Distribution.pdf'
plt.savefig(plot_filename.replace('.pdf', '.png'), dpi=600, bbox_inches='tight')
plt.savefig(plot_filename, dpi=600, bbox_inches='tight')
plt.close()

print(f"🎉 성공! 스크리닝 분포 그래프 출력 완료: {plot_filename}")

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

print("🔍 1. 전체 2,737개 메타데이터 로드 및 Tier 1 스크리닝 중...")

# 파일명을 로컬 환경에 맞게 수정 완료!
file_path = "CR_meta_data_SI.json"

with open(file_path, 'r', encoding='utf-8') as f:
    data = json.load(f)

# 메타데이터 추출 및 리스트화
mof_list = []
for key, val in data.items():
    mofid_v1 = val.get("id", {}).get("mofid-v1", "Unknown")
    short_id = mofid_v1.split(';')[1][:6].upper() if ';' in mofid_v1 else mofid_v1[:15]
    
    zeopp = val.get("Zeopp", {})
    pld = zeopp.get("PLD", 0.0)
    lcd = zeopp.get("LCD", 0.0)
    
    metal_info = val.get("metal", {})
    metal_type = metal_info.get("metal_type", "Unknown")
    has_oms = metal_info.get("has_OMS", "Unknown")
    
    topology = val.get("CrystalNets", {}).get("all_nodes", "Unknown")
    doi = val.get("reference", {}).get("DOI", "Unknown")
    
    mof_list.append({
        "Short_ID": short_id,
        "Metal": metal_type,
        "Has_OMS": has_oms,
        "PLD (Limiting Pore)": pld,
        "LCD (Largest Cavity)": lcd,
        "Topology": topology,
        "DOI": doi
    })

df_all = pd.DataFrame(mof_list)

# 🌟 Tier 1 순수 스크리닝 (Zn/Co + No OMS + 매직 윈도우)
tier1_cond = (
    df_all['Metal'].isin(['Zn', 'Co']) & 
    (df_all['Has_OMS'] == 'No') & 
    (df_all['PLD (Limiting Pore)'] >= 3.2) & 
    (df_all['PLD (Limiting Pore)'] <= 3.8)
)
df_tier1_full = df_all[tier1_cond].copy()

# 중복 뼈대(Short_ID 기준) 제거
df_tier1_unique = df_tier1_full.drop_duplicates(subset=['Short_ID'])

print(f"🎯 최종 Tier 1 고유 원석: {len(df_tier1_unique)}개 추출 완료!\n")

# 엑셀 파일로 저장
excel_filename = "All_Database_Tier1_Screening_Results.xlsx"
df_tier1_unique.to_excel(excel_filename, index=False)
print(f"📄 전체 리스트 엑셀 저장 완료: {excel_filename}")

# ==========================================
# 기하학적 스크리닝 결과 시각화 (PLD vs LCD)
# ==========================================
print("📈 2. 기하학적 분포(PLD vs LCD) 산점도 그래프 생성을 시작합니다...")

plt.style.use('default')
sns.set_context("talk", font_scale=1.5) 
sns.set_style("ticks")
fig, ax = plt.subplots(figsize=(14, 10))

# 금속 색상 지정 (통일성 유지)
set1_hex = ['#e41a1c', '#377eb8', '#4daf4a', '#984ea3', '#ff7f00', '#ffff33', '#a65628', '#f781bf']
metal_colors = {'Zn': set1_hex[1], 'Co': set1_hex[0]} # Zn 파랑, Co 빨강

sns.scatterplot(
    data=df_tier1_unique, 
    x='PLD (Limiting Pore)', y='LCD (Largest Cavity)', 
    hue='Metal', palette=metal_colors,
    s=400, alpha=0.8, edgecolor='black', ax=ax
)

# 매직 윈도우 영역 강조 표시 (배경색 칠하기)
ax.axvspan(3.2, 3.8, color='green', alpha=0.1, label='Magic Window Zone')

ax.set_title('Tier 1 Screening Results (2,737 MOFs Database)', fontsize=24, fontweight='bold', pad=20)
ax.set_xlabel('Limiting Pore Diameter (PLD, $\AA$)', fontsize=18, fontweight='bold')
ax.set_ylabel('Largest Cavity Diameter (LCD, $\AA$)', fontsize=18, fontweight='bold')
ax.grid(True, which="major", ls="--", alpha=0.5)

# 범례 설정
handles, labels = ax.get_legend_handles_labels()
ax.legend(
    handles=handles, labels=labels,
    bbox_to_anchor=(1.02, 1), loc='upper left', 
    title='Properties', title_fontsize=16, fontsize=14, 
    frameon=True, edgecolor='black'
)

plt.tight_layout()
plot_filename = 'Tier1_Screening_Geometric_Distribution.pdf'
plt.savefig(plot_filename.replace('.pdf', '.png'), dpi=600, bbox_inches='tight')
plt.savefig(plot_filename, dpi=600, bbox_inches='tight')
plt.close()

print(f"🎉 성공! 스크리닝 분포 그래프 출력 완료: {plot_filename}")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import ast
import os

print("🚀 1. 세 그룹의 데이터 로드 및 색상 영구 동기화 시작...")

# 1. 파일명 설정 (환경에 맞게 확장자 수정 필요 시 .xlsx로 변경)
file_hf = "High_Flux_Final_Metrics.csv" 
file_n2 = "N2_Sieving_Final_Metrics.csv"
file_tier = "Tier1_Tier2_Anti_Humidity_Targets.csv"

# 데이터 로드
try:
    hf = pd.read_csv(file_hf)
    n2 = pd.read_csv(file_n2)
    tier = pd.read_csv(file_tier)
except FileNotFoundError:
    # CSV가 없을 경우 엑셀 포맷으로 재시도
    hf = pd.read_excel(file_hf.replace('.csv', '.xlsx'))
    n2 = pd.read_excel(file_n2.replace('.csv', '.xlsx'))
    tier = pd.read_excel(file_tier.replace('.csv', '.xlsx'))

hf['Group'] = 'High Flux'
n2['Group'] = 'N2 Sieving'
tier['Group'] = 'Tier 1 & 2'

# 2. 금속 추출 함수
def extract_metal(chem_str):
    chem_str = str(chem_str)
    metals = ['Zn', 'Co', 'Cu', 'Cd', 'Ni', 'Gd', 'Mn', 'Dy', 'Eu', 'Ho', 'Tb']
    for m in metals:
        if f"'metal_type': '{m}'" in chem_str: return m
    return 'Other'

hf['Metal'] = hf['Chemistry (Metal)'].apply(extract_metal)
n2['Metal'] = n2['Chemistry (Metal)'].apply(extract_metal)
tier['Metal'] = tier['Chemistry (Metal)'].apply(extract_metal)

# 🌟 3. [핵심 수정] 색상 영구 박제 (연구자님 코드 로직 100% 반영)
# Tier 그룹이 합쳐지기 전, 순수 115개(HF + N2)의 금속 배열을 기준으로 색상을 먼저 고정합니다!
df_base115 = pd.concat([hf, n2], ignore_index=True)
set1_hex = ['#e41a1c', '#377eb8', '#4daf4a', '#984ea3', '#ff7f00', '#ffff33', '#a65628', '#f781bf', '#999999', '#a6cee3', '#1f78b4', '#b2df8a']
unique_all_metals = sorted(df_base115['Metal'].unique())

# 이전 그래프와 100% 동일한 색상 사전(Dictionary) 생성
universal_metal_colors = {metal: set1_hex[i % len(set1_hex)] for i, metal in enumerate(unique_all_metals)}
universal_metal_colors['Other'] = '#000000' # 혹시 모를 예외 금속은 검은색 처리

# 4. 전체 데이터 병합 및 중복 뼈대 통제 (우선순위: Tier 그룹 > High Flux > N2 Sieving)
df_all = pd.concat([hf, n2, tier], ignore_index=True)
df_all['Short_ID'] = df_all['Readable_MOF_ID'].apply(lambda x: str(x).split(' ')[0].replace('MOF-', ''))

priority = {'Tier 1 & 2': 1, 'High Flux': 2, 'N2 Sieving': 3}
df_all['Priority'] = df_all['Group'].map(priority)
# 중복 시 Tier 그룹 최우선 생존
df_all = df_all.sort_values(['Priority', 'Performance Score'], ascending=[True, False]).drop_duplicates(subset=['Short_ID'])

print(f"✅ 중복 제거 완료. 색상 동기화 패치 완료!")

# ==========================================
# 5. 초고해상도 플롯 생성 함수 (주피터 도출 + 저장)
# ==========================================
def plot_frontier(df, title, filename):
    if len(df) == 0: return

    plt.style.use('default')
    sns.set_context("talk", font_scale=1.5)
    sns.set_style("ticks")
    fig, ax = plt.subplots(figsize=(16, 12))

    # 🌟 앞서 박제해둔 universal_metal_colors 주입
    sns.scatterplot(
        data=df, x='Working Capacity (mmol/g)', y='Selectivity',
        hue='Metal', palette=universal_metal_colors, 
        s=600, alpha=0.85, edgecolor='black', ax=ax, zorder=5
    )

    # 상위 랭커 이름표 라벨링 (종합 그래프는 5개, 개별 그래프는 3개)
    n_labels = 5 if "All" in title else 3
    top_targets = df.nlargest(n_labels, 'Performance Score')
    for _, row in top_targets.iterrows():
        ax.text(
            row['Working Capacity (mmol/g)'] * 1.05, row['Selectivity'], 
            row['Short_ID'], 
            fontsize=14, fontweight='bold', color='black', zorder=10
        )

    # 축 및 레이아웃 설정
    ax.set_title(title, fontsize=32, fontweight='bold', pad=20)
    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.set_xlabel('CO$_2$ Working Capacity (mmol/g)', fontsize=24, fontweight='bold')
    ax.set_ylabel('CO$_2$/N$_2$ Selectivity', fontsize=24, fontweight='bold')
    ax.grid(True, which="both", ls="--", alpha=0.4)
    ax.tick_params(labelsize=18)

    # 범례 위치 고정
    handles, labels = ax.get_legend_handles_labels()
    if handles:
        ax.legend(
            handles=handles, labels=labels, bbox_to_anchor=(1.02, 1), loc='upper left', 
            title='Core Metal', title_fontsize=20, fontsize=18, frameon=True, edgecolor='black'
        )
    
    plt.tight_layout()
    
    # 파일로 자동 저장 (PNG/PDF)
    plt.savefig(filename.replace('.png', '.pdf'), dpi=600, bbox_inches='tight')
    plt.savefig(filename, dpi=600, bbox_inches='tight')
    
    # 🌟 주피터 노트북 화면에 띄우기 (이 부분이 핵심!)
    plt.show()

# 4종류 그래프 순차적으로 도출
print("📈 1/4. 종합 프론티어 (All Groups)")
plot_frontier(df_all, 'Comprehensive Frontier (All 3 Groups)', 'Final_Comprehensive_Frontier.png')

print("📈 2/4. High Flux 전용 프론티어")
plot_frontier(df_all[df_all['Group'] == 'High Flux'], 'High Flux Group Frontier', 'Final_HighFlux_Frontier.png')

print("📈 3/4. N2 Sieving 전용 프론티어")
plot_frontier(df_all[df_all['Group'] == 'N2 Sieving'], 'N2 Sieving Group Frontier', 'Final_N2Sieving_Frontier.png')

print("📈 4/4. Tier 1 & 2 (고습도 생존) 프론티어")
plot_frontier(df_all[df_all['Group'] == 'Tier 1 & 2'], 'Tier 1 & 2 (Anti-Humidity) Frontier', 'Final_Tier1_2_Frontier.png')

print("🎉 모든 시각화 및 파일 저장이 완료되었습니다!")

In [ ]:
import os

print("현재 작업 폴더 경로:", os.getcwd())
print("\n[현재 폴더에 있는 CSV/Excel 파일 목록]")
for file in os.listdir():
    if file.endswith(".csv") or file.endswith(".xlsx"):
        print("-", file)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import ast

print("🚀 1. 데이터 로드 및 완벽한 색상/마커 동기화 시작...")

# 1. 데이터 로드 (파일명은 로컬 환경에 맞게 조정해 주세요)
try:
    hf = pd.read_csv("High_Flux_Final_Metrics_For_Python.csv")
    n2 = pd.read_csv("N2_Sieving_Final_Metrics_For_Python.csv")
    tier = pd.read_csv("Tier1_Tier2_Anti_Humidity_Targets_For_Python.csv")
except FileNotFoundError:
    hf = pd.read_csv("High_Flux_Final_Metrics.xlsx - High Flux Group.csv")
    n2 = pd.read_csv("N2_Sieving_Final_Metrics.xlsx - N2 Sieving Group.csv")
    tier = pd.read_csv("Tier1_Tier2_Anti_Humidity_Targets.xlsx - Sheet1.csv")

hf['Group'] = 'High-Flux (Wide)'
n2['Group'] = 'N2-Sieving (Narrow)'
tier['Group'] = 'Tier 1 & 2'

# 전체 통합 및 식별자 추출
df_all = pd.concat([hf, n2, tier], ignore_index=True)
df_all['Short_ID'] = df_all['Readable_MOF_ID'].apply(lambda x: str(x).split(' ')[0].replace('MOF-', ''))

# 🌟 2. 금속(Metal) 및 OMS 상태 정밀 추출
def parse_chem(chem_str):
    try:
        d = ast.literal_eval(chem_str)
        return pd.Series([d.get('metal_type', 'Unknown'), d.get('has_OMS', 'Unknown')])
    except:
        return pd.Series(["Unknown", "Unknown"])

df_all[['Metal', 'Has_OMS']] = df_all['Chemistry (Metal)'].apply(parse_chem)

# 중복 제거 로직 (우선순위: Tier > High Flux > N2 Sieving)
priority = {'Tier 1 & 2': 1, 'High-Flux (Wide)': 2, 'N2-Sieving (Narrow)': 3}
df_all['Priority'] = df_all['Group'].map(priority)
df_all = df_all.sort_values(['Priority', 'Performance Score'], ascending=[True, False]).drop_duplicates(subset=['Short_ID'])

# 🌟 3. [복원 완료] 이전과 100% 동일한 색상 팔레트 하드코딩
# 기존 115개(High Flux + N2)의 금속 배열을 기준으로 색상을 매핑하여 이전 그래프와 완벽 일치시킵니다.
df_base115 = df_all[df_all['Group'].isin(['High-Flux (Wide)', 'N2-Sieving (Narrow)'])]
unique_metals = sorted(df_base115['Metal'].unique())

set1_hex = ['#e41a1c', '#377eb8', '#4daf4a', '#984ea3', '#ff7f00', '#ffff33', '#a65628', '#f781bf', '#999999', '#a6cee3', '#1f78b4', '#b2df8a']
metal_colors = {metal: set1_hex[i % len(set1_hex)] for i, metal in enumerate(unique_metals)}

# 만약 Tier 그룹에 새로운 금속이 있을 경우 예외 처리
for m in df_all['Metal'].unique():
    if m not in metal_colors:
        metal_colors[m] = '#000000'

# 🌟 [복원 완료] OMS 유무에 따른 마커 스타일 구분 (동그라미 vs X표시)
oms_markers = {'No': 'o', 'Yes': 'X', 'Unknown': 's'}


# ==========================================
# 4. 공통 그래프 생성 및 개별 PDF 저장 함수 (스케일업 버전)
# ==========================================
def save_scatter_pdf(df, title, filename):
    if len(df) == 0: return
    
    plt.style.use('default')
    sns.set_context("talk", font_scale=2.0) 
    sns.set_style("ticks")

    # 캔버스 24x18 확대
    fig, ax = plt.subplots(figsize=(24, 18))

    # 마커 크기 600, hue=Metal, style=Has_OMS
    sns.scatterplot(
        data=df, x='Working Capacity (mmol/g)', y='Selectivity', 
        hue='Metal', style='Has_OMS', markers=oms_markers,
        palette=metal_colors,
        s=600, alpha=0.85, edgecolor='black', ax=ax, zorder=5
    )

    # 텍스트 라벨링 (종합 그래프는 상위 5개, 개별은 3개)
    n_labels = 5 if "Combined" in title else 3
    top_targets = df.nlargest(n_labels, 'Performance Score')
    for _, row in top_targets.iterrows():
        ax.text(
            row['Working Capacity (mmol/g)'] * 1.05, row['Selectivity'], 
            row['Short_ID'], 
            fontsize=20, fontweight='bold', color='black', zorder=10
        )

    # 폰트 스케일 업 적용
    ax.set_title(title, fontsize=36, fontweight='bold', pad=25)
    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.set_xlabel('CO$_2$ Working Capacity (mmol/g)', fontsize=28, fontweight='bold')
    ax.set_ylabel('CO$_2$/N$_2$ Selectivity (Purity)', fontsize=28, fontweight='bold')
    ax.grid(True, which="both", ls="--", alpha=0.4)
    ax.tick_params(labelsize=22)

    # 범례 설정 및 markerscale 확대
    handles, labels = ax.get_legend_handles_labels()
    if handles:
        ax.legend(
            handles=handles, labels=labels,
            bbox_to_anchor=(1.02, 1), loc='upper left', borderaxespad=0., 
            title='Legend (Metal / OMS)', title_fontsize=28, fontsize=24, markerscale=2.0
        )

    plt.tight_layout()
    plt.savefig(filename.replace('.pdf', '.png'), dpi=600, bbox_inches='tight')
    plt.savefig(filename, dpi=600, bbox_inches='tight')
    plt.show()

# ==========================================
# 5. 각각의 개별 대형 PDF/PNG 그래프 생성
# ==========================================
print("📈 개별 대형 PDF/PNG 그래프 생성을 시작합니다...\n")

save_scatter_pdf(df_all[df_all['Group'] == 'N2-Sieving (Narrow)'], '(a) N2-Sieving Group Only', 'Frontier_a_N2_Sieving_Only_Large.pdf')
save_scatter_pdf(df_all[df_all['Group'] == 'High-Flux (Wide)'], '(b) High-Flux Bottleneck Group Only', 'Frontier_b_High_Flux_Only_Large.pdf')
save_scatter_pdf(df_all[df_all['Group'] == 'Tier 1 & 2'], '(c) Tier 1 & 2 (Anti-Humidity) Only', 'Frontier_c_Tier1_2_Only_Large.pdf')
save_scatter_pdf(df_all, '(d) Combined View (All Groups)', 'Frontier_d_Combined_Large.pdf')

print("\n🎉 네이처(Nature) 급 저널에 제출해도 손색없는 600DPI 초고해상도 출력이 완료되었습니다!")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import ast
import os

print("🚀 금속 색상 고정 및 범례 레이아웃 최적화 작업을 시작합니다...")

# 1. 파일 로드 (연구자님 환경의 실제 파일명으로 수정해 주세요)
# 아래 리스트는 연구자님이 업로드해주신 파일명 기준입니다.
files = {
    'High Flux': "High_Flux_Final_Metrics_For_Python.csv",
    'N2 Sieving': "N2_Sieving_Final_Metrics_For_Python.csv",
    'Tier 1 & 2': "Tier1_Tier2_Anti_Humidity_Targets_For_Python.csv"
}

data_frames = {}
for name, path in files.items():
    if os.path.exists(path):
        data_frames[name] = pd.read_csv(path)
    else:
        print(f"⚠️ 파일을 찾을 수 없습니다: {path}")

# 데이터 통합 준비
hf = data_frames.get('High Flux', pd.DataFrame())
n2 = data_frames.get('N2 Sieving', pd.DataFrame())
tier = data_frames.get('Tier 1 & 2', pd.DataFrame())

hf['Group'] = 'High-Flux (Wide)'
n2['Group'] = 'N2-Sieving (Narrow)'
tier['Group'] = 'Tier 1 & 2'

# 2. 금속(Metal) 및 OMS 상태 정밀 추출 함수
def parse_chem(chem_str):
    try:
        # 딕셔너리 형태의 문자열을 파싱
        d = ast.literal_eval(str(chem_str))
        return pd.Series([d.get('metal_type', 'Unknown'), d.get('has_OMS', 'Unknown')])
    except:
        # 텍스트에서 직접 금속 기호만 추출 (MOF-Zn... 형태일 경우 대비)
        if pd.isna(chem_str): return pd.Series(["Unknown", "Unknown"])
        metal = str(chem_str).split(' ')[0]
        return pd.Series([metal, "Unknown"])

# 모든 데이터 병합 전, 전체 금속 리스트를 확보하여 색상 고정
df_all_raw = pd.concat([hf, n2, tier], ignore_index=True)
df_all_raw[['Metal', 'Has_OMS']] = df_all_raw['Chemistry (Metal)'].apply(parse_chem)

# 🌟 [해결책 1] 색상 팔레트 영구 박제 (알파벳 순서로 고정)
set1_hex = ['#e41a1c', '#377eb8', '#4daf4a', '#984ea3', '#ff7f00', '#ffff33', '#a65628', '#f781bf', '#999999', '#a6cee3', '#1f78b4', '#b2df8a']
all_unique_metals = sorted([m for m in df_all_raw['Metal'].unique() if m != 'Unknown'])
metal_colors = {metal: set1_hex[i % len(set1_hex)] for i, metal in enumerate(all_unique_metals)}
metal_colors['Unknown'] = '#000000'

# 마커 스타일 고정
oms_markers = {'No': 'o', 'Yes': 'X', 'Unknown': 's'}

# 3. 중복 제거 (Tier > HF > N2 우선순위)
df_all_raw['Short_ID'] = df_all_raw['Readable_MOF_ID'].apply(lambda x: str(x).split(' ')[0].replace('MOF-', ''))
priority = {'Tier 1 & 2': 1, 'High-Flux (Wide)': 2, 'N2-Sieving (Narrow)': 3}
df_all_raw['Priority'] = df_all_raw['Group'].map(priority)
df_final = df_all_raw.sort_values(['Priority', 'Performance Score'], ascending=[True, False]).drop_duplicates(subset=['Short_ID'])

# ==========================================
# 4. 개선된 그래프 생성 함수 (범례 영역 문제 해결)
# ==========================================
def save_optimized_plot(df, title, filename):
    if len(df) == 0: return
    
    plt.style.use('default')
    sns.set_context("talk", font_scale=2.0) 
    sns.set_style("ticks")

    # 24x18 대형 캔버스
    fig, ax = plt.subplots(figsize=(24, 18))

    # 산점도 그리기
    scatter = sns.scatterplot(
        data=df, x='Working Capacity (mmol/g)', y='Selectivity', 
        hue='Metal', style='Has_OMS', markers=oms_markers,
        palette=metal_colors,
        s=700, alpha=0.85, edgecolor='black', ax=ax, zorder=5
    )

    # 텍스트 라벨링
    n_labels = 5 if "Combined" in title else 3
    top_targets = df.nlargest(n_labels, 'Performance Score')
    for _, row in top_targets.iterrows():
        ax.text(
            row['Working Capacity (mmol/g)'] * 1.05, row['Selectivity'], 
            row['Short_ID'], 
            fontsize=22, fontweight='bold', color='black', zorder=10
        )

    # 차트 디자인
    ax.set_title(title, fontsize=40, fontweight='bold', pad=30)
    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.set_xlabel('CO$_2$ Working Capacity (mmol/g)', fontsize=32, fontweight='bold')
    ax.set_ylabel('CO$_2$/N$_2$ Selectivity (Purity)', fontsize=32, fontweight='bold')
    ax.grid(True, which="both", ls="--", alpha=0.4)
    ax.tick_params(labelsize=24)

    # 🌟 [해결책 2] 범례 영역 침범 문제 해결
    # 범례 항목 간의 간격(labelspacing), 마커와 텍스트 사이 간격(handletextpad) 조정
    # 범례 박스를 그래프 바깥쪽으로 더 밀어내고(1.05), 항목들을 더 여유 있게 배치
    handles, labels = ax.get_legend_handles_labels()
    if handles:
        legend = ax.legend(
            handles=handles, labels=labels,
            bbox_to_anchor=(1.02, 1), loc='upper left', 
            borderaxespad=0., 
            title='Metal / Open Metal Site', 
            title_fontsize=30, 
            fontsize=24, 
            markerscale=2.5,
            labelspacing=1.2,   # 항목 간 세로 간격 확대
            handletextpad=1.0    # 마커와 글자 사이 가로 간격 확대
        )
        legend.get_frame().set_edgecolor('black')
        legend.get_frame().set_linewidth(1.5)

    plt.tight_layout()
    
    # 초고해상도 저장
    plt.savefig(filename.replace('.pdf', '.png'), dpi=600, bbox_inches='tight')
    plt.savefig(filename, dpi=600, bbox_inches='tight')
    plt.show()

# ==========================================
# 5. 최종 시각화 실행
# ==========================================
print("📈 최종 필터링된 그래프 생성을 시작합니다...\n")

save_optimized_plot(df_final[df_final['Group'] == 'N2-Sieving (Narrow)'], '(a) N2-Sieving Group', 'Frontier_Optimized_N2_Sieving.pdf')
save_optimized_plot(df_final[df_final['Group'] == 'High-Flux (Wide)'], '(b) High-Flux Group', 'Frontier_Optimized_High_Flux.pdf')
save_optimized_plot(df_final[df_final['Group'] == 'Tier 1 & 2'], '(c) Tier 1 & 2 (Anti-Humidity)', 'Frontier_Optimized_Tier1_2.pdf')
save_optimized_plot(df_final, '(d) Combined View (All Targets)', 'Frontier_Optimized_Combined.pdf')

print("\n🎉 모든 그래프의 색상 동기화 및 범례 교정이 완료되었습니다!")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import ast
import os

print("🚀 축 레이블 수정 및 범례 겹침 현상 해결을 위한 최종 시각화 시작...")

# 1. 데이터 로드 (실제 파일명으로 확인 필요)
files = {
    'High Flux': "High_Flux_Final_Metrics_For_Python.csv",
    'N2 Sieving': "N2_Sieving_Final_Metrics_For_Python.csv",
    'Tier 1 & 2': "Tier1_Tier2_Anti_Humidity_Targets_For_Python.csv"
}

dfs = []
for name, path in files.items():
    if os.path.exists(path):
        temp_df = pd.read_csv(path)
        temp_df['Group'] = name
        dfs.append(temp_df)
    else:
        print(f"⚠️ 파일 확인 필요: {path}")

if not dfs:
    raise FileNotFoundError("분석할 데이터 파일이 하나도 없습니다.")

df_all = pd.concat(dfs, ignore_index=True)
df_all['Short_ID'] = df_all['Readable_MOF_ID'].apply(lambda x: str(x).split(' ')[0].replace('MOF-', ''))

# 2. 금속 및 OMS 추출
def parse_chem(chem_str):
    try:
        d = ast.literal_eval(str(chem_str))
        return pd.Series([d.get('metal_type', 'Unknown'), d.get('has_OMS', 'Unknown')])
    except:
        return pd.Series(["Unknown", "Unknown"])

df_all[['Metal', 'Has_OMS']] = df_all['Chemistry (Metal)'].apply(parse_chem)

# 3. 색상 및 마커 스타일 고정 (이전과 동일한 색상 유지)
set1_hex = ['#e41a1c', '#377eb8', '#4daf4a', '#984ea3', '#ff7f00', '#ffff33', '#a65628', '#f781bf', '#999999', '#a6cee3', '#1f78b4', '#b2df8a']
unique_metals = sorted([m for m in df_all['Metal'].unique() if m != 'Unknown'])
metal_colors = {m: set1_hex[i % len(set1_hex)] for i, m in enumerate(unique_metals)}
metal_colors['Unknown'] = '#000000'

oms_markers = {'No': 'o', 'Yes': 'X', 'Unknown': 's'}

# 중복 제거 (우선순위 적용)
priority = {'Tier 1 & 2': 1, 'High Flux': 2, 'N2 Sieving': 3}
df_all['Priority'] = df_all['Group'].map(priority)
df_final = df_all.sort_values(['Priority', 'Performance Score'], ascending=[True, False]).drop_duplicates(subset=['Short_ID'])

# ==========================================
# 4. 개선된 그래프 생성 함수 (축 레이블 및 범례 교정)
# ==========================================
def save_corrected_plot(df, title, filename):
    plt.style.use('default')
    sns.set_context("talk", font_scale=2.0) 
    sns.set_style("ticks")

    fig, ax = plt.subplots(figsize=(24, 18))

    # 산점도: s=700으로 고해상도 시인성 확보
    scatter = sns.scatterplot(
        data=df, x='Working Capacity (mmol/g)', y='Selectivity', 
        hue='Metal', style='Has_OMS', markers=oms_markers,
        palette=metal_colors,
        s=700, alpha=0.85, edgecolor='black', ax=ax, zorder=5
    )

    # 텍스트 라벨링
    top_targets = df.nlargest(5, 'Performance Score')
    for _, row in top_targets.iterrows():
        ax.text(
            row['Working Capacity (mmol/g)'] * 1.05, row['Selectivity'], 
            row['Short_ID'], 
            fontsize=22, fontweight='bold', color='black', zorder=10
        )

    # 축 레이블 수정 (요청하신 명칭으로 변경)
    ax.set_title(title, fontsize=42, fontweight='bold', pad=35)
    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.set_xlabel('CO$_2$ Affinity (Capacity)', fontsize=32, fontweight='bold')
    ax.set_ylabel('CO$_2$/N$_2$ Selectivity (Purity)', fontsize=32, fontweight='bold')
    ax.grid(True, which="both", ls="--", alpha=0.4)
    ax.tick_params(labelsize=26)

    # 🌟 범례 겹침 문제 해결을 위한 정밀 세팅
    handles, labels = ax.get_legend_handles_labels()
    if handles:
        # 범례 항목 간의 수직 간격(labelspacing)과 가로 간격(handletextpad)을 대폭 확대
        legend = ax.legend(
            handles=handles, labels=labels,
            bbox_to_anchor=(1.02, 1), loc='upper left', 
            borderaxespad=0., 
            title='Metal / OMS Status', 
            title_fontsize=32, 
            fontsize=26, 
            markerscale=2.2,
            labelspacing=1.8,   # 수직 간격 확대 (겹침 방지 핵심)
            handletextpad=1.5,  # 기호와 글자 사이 간격 확대
            columnspacing=2.0   # 열 간격 확대
        )
        legend.get_frame().set_edgecolor('black')
        legend.get_frame().set_linewidth(2.0)

    plt.tight_layout()
    
    # 초고해상도 저장 (딱 한 쌍의 파일만 생성)
    plt.savefig(filename.replace('.pdf', '.png'), dpi=600, bbox_inches='tight')
    plt.savefig(filename, dpi=600, bbox_inches='tight')
    plt.show()

# ==========================================
# 5. 최종 시각화 실행
# ==========================================
print("📈 최종 필터링된 그래프 생성을 시작합니다...\n")

save_corrected_plot(df_final, 'Combined View: CO$_2$ Capture Performance Frontier', 'Final_Frontier_Corrected.pdf')

print("\n🎉 축 레이블 및 범례 교정이 완료된 초고해상도 그래프가 도출되었습니다!")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import ast
import os
import numpy as np

print("🚀 [최종 교정] SI 단위 환산 및 범례 레이아웃 최적화 시작...")

# 1. 파일 로드 (For Python 파일 사용)
files = {
    'High Flux': "High_Flux_Final_Metrics_For_Python.csv",
    'N2 Sieving': "N2_Sieving_Final_Metrics_For_Python.csv",
    'Tier 1 & 2': "Tier1_Tier2_Anti_Humidity_Targets_For_Python.csv"
}

dfs = []
for name, path in files.items():
    if os.path.exists(path):
        temp_df = pd.read_csv(path)
        temp_df['Group'] = name
        dfs.append(temp_df)

df_all = pd.concat(dfs, ignore_index=True)
df_all['Short_ID'] = df_all['Readable_MOF_ID'].apply(lambda x: str(x).split(' ')[0].replace('MOF-', ''))

# 금속 및 OMS 추출
def parse_chem(chem_str):
    try:
        d = ast.literal_eval(str(chem_str))
        return pd.Series([d.get('metal_type', 'Unknown'), d.get('has_OMS', 'Unknown')])
    except: return pd.Series(["Unknown", "Unknown"])

df_all[['Metal', 'Has_OMS']] = df_all['Chemistry (Metal)'].apply(parse_chem)

# 🌟 2. SI 단위 환산 (mmol/g -> mol/kg·Pa)
# PDF 스케일($10^{-5}$)에 맞추기 위해 10^-5을 곱해줍니다.
df_all['SI_Affinity'] = df_all['Working Capacity (mmol/g)'] * 1e-5

# 3. 색상 및 마커 스타일 고정
set1_hex = ['#e41a1c', '#377eb8', '#4daf4a', '#984ea3', '#ff7f00', '#ffff33', '#a65628', '#f781bf', '#999999', '#a6cee3', '#1f78b4', '#b2df8a']
unique_metals = sorted([m for m in df_all['Metal'].unique() if m != 'Unknown'])
metal_colors = {m: set1_hex[i % len(set1_hex)] for i, m in enumerate(unique_metals)}
metal_colors['Unknown'] = '#000000'
oms_markers = {'No': 'o', 'Yes': 'X', 'Unknown': 's'}

# 중복 제거 (Tier 우선순위)
priority = {'Tier 1 & 2': 1, 'High Flux': 2, 'N2 Sieving': 3}
df_all['Priority'] = df_all['Group'].map(priority)
df_final = df_all.sort_values(['Priority', 'Performance Score'], ascending=[True, False]).drop_duplicates(subset=['Short_ID'])

# ==========================================
# 4. 시각화 (PDF와 동일한 x축 스케일 및 범례 교정)
# ==========================================
def save_nature_plot(df, title, filename):
    plt.style.use('default')
    sns.set_context("talk", font_scale=2.0) 
    sns.set_style("ticks")

    fig, ax = plt.subplots(figsize=(24, 18))

    # 스케일링된 SI_Affinity 사용
    sns.scatterplot(
        data=df, x='SI_Affinity', y='Selectivity', 
        hue='Metal', style='Has_OMS', markers=oms_markers,
        palette=metal_colors, s=800, alpha=0.85, edgecolor='black', ax=ax, zorder=5
    )

    # 상위 타겟 라벨링
    top_targets = df.nlargest(5, 'Performance Score')
    for _, row in top_targets.iterrows():
        ax.text(
            row['SI_Affinity'] * 1.1, row['Selectivity'], 
            row['Short_ID'], fontsize=22, fontweight='bold', color='black', zorder=10
        )

    # 축 설정 (PDF 사양과 100% 일치)
    ax.set_title(title, fontsize=42, fontweight='bold', pad=35)
    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.set_xlabel('CO$_2$ Affinity (Capacity) [mol/kg$\cdot$Pa]', fontsize=32, fontweight='bold') # 단위 명시
    ax.set_ylabel('CO$_2$/N$_2$ Selectivity (Purity)', fontsize=32, fontweight='bold')
    
    # 🌟 x축 범위를 PDF와 동일하게 고정 (10^-7 ~ 10^-3)
    ax.set_xlim(1e-7, 1e-3)
    ax.grid(True, which="both", ls="--", alpha=0.4)
    ax.tick_params(labelsize=26)

    # 🌟 범례 겹침 문제 최종 해결 (간격 및 위치 극대화)
    handles, labels = ax.get_legend_handles_labels()
    if handles:
        legend = ax.legend(
            handles=handles, labels=labels,
            bbox_to_anchor=(1.05, 1), loc='upper left', # 그래프 밖으로 더 멀리 이동
            borderaxespad=0.2, 
            title='Metal / OMS Status', title_fontsize=32, fontsize=26, 
            markerscale=2.5,
            labelspacing=2.0,   # 수직 간격을 2.0으로 대폭 확대
            handletextpad=1.8   # 기호와 글자 사이 간격을 1.8로 확대
        )
        legend.get_frame().set_edgecolor('black')
        legend.get_frame().set_linewidth(2.0)

    plt.tight_layout()
    plt.savefig(filename, dpi=600, bbox_inches='tight')
    plt.show()

# 실행
save_nature_plot(df_final, 'Combined View: CO$_2$ Capture Performance Frontier (SI Units)', 'Final_Corrected_SI_Frontier.pdf')

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import ast
import os

print("🚀 금속 색상 영구 고정 및 물리량/범례 최적화 시각화를 시작합니다...")

# 1. 데이터 로드 (실제 파일명으로 로드)
files = {
    'High Flux': "High_Flux_Final_Metrics_For_Python.csv",
    'N2 Sieving': "N2_Sieving_Final_Metrics_For_Python.csv",
    'Tier 1 & 2': "Tier1_Tier2_Anti_Humidity_Targets_For_Python.csv"
}

dfs = []
for name, path in files.items():
    if os.path.exists(path):
        temp_df = pd.read_csv(path)
        temp_df['Group'] = name
        dfs.append(temp_df)

df_all = pd.concat(dfs, ignore_index=True)
df_all['Short_ID'] = df_all['Readable_MOF_ID'].apply(lambda x: str(x).split(' ')[0].replace('MOF-', ''))

# 금속 및 OMS 추출 함수
def parse_chem(chem_str):
    try:
        d = ast.literal_eval(str(chem_str))
        return pd.Series([d.get('metal_type', 'Unknown'), d.get('has_OMS', 'Unknown')])
    except:
        return pd.Series(["Unknown", "Unknown"])

df_all[['Metal', 'Has_OMS']] = df_all['Chemistry (Metal)'].apply(parse_chem)

# 🌟 2. 물리량 단위 환산 및 스케일링 (PDF 기준 10^-5 배 적용)
df_all['SI_Affinity'] = df_all['Working Capacity (mmol/g)'] * 1e-5

# 🌟 3. 금속 색상 영구 박제 (Key Error 방지를 위해 전체 금속 리스트 사전 확보)
# 데이터에 존재하는 모든 금속(Zn, Co, V, Ni, Cu 등)을 알파벳 순으로 정렬하여 색상을 고정합니다.
set1_hex = ['#e41a1c', '#377eb8', '#4daf4a', '#984ea3', '#ff7f00', '#ffff33', '#a65628', '#f781bf', '#999999', '#a6cee3', '#1f78b4', '#b2df8a']
all_metals_in_data = sorted(df_all['Metal'].unique())
universal_palette = {m: set1_hex[i % len(set1_hex)] for i, m in enumerate(all_metals_in_data)}

oms_markers = {'No': 'o', 'Yes': 'X', 'Unknown': 's'}

# 중복 제거 (Tier 우선순위)
priority = {'Tier 1 & 2': 1, 'High Flux': 2, 'N2 Sieving': 3}
df_all['Priority'] = df_all['Group'].map(priority)
df_final = df_all.sort_values(['Priority', 'Performance Score'], ascending=[True, False]).drop_duplicates(subset=['Short_ID'])

# ==========================================
# 4. 시각화 함수 (범위 최적화 및 범례 박스 스케일업)
# ==========================================
def save_perfect_plot(df, title, filename):
    plt.style.use('default')
    sns.set_context("talk", font_scale=2.0) 
    sns.set_style("ticks")

    fig, ax = plt.subplots(figsize=(24, 18))

    # 산점도 그리기 (영구 박제된 universal_palette 사용)
    sns.scatterplot(
        data=df, x='SI_Affinity', y='Selectivity', 
        hue='Metal', style='Has_OMS', markers=oms_markers,
        palette=universal_palette, s=800, alpha=0.85, edgecolor='black', ax=ax, zorder=5
    )

    # 상위 타겟 라벨링
    top_n = 5 if "Combined" in title else 3
    top_targets = df.nlargest(top_n, 'Performance Score')
    for _, row in top_targets.iterrows():
        ax.text(
            row['SI_Affinity'] * 1.08, row['Selectivity'], 
            row['Short_ID'], fontsize=22, fontweight='bold', color='black', zorder=10
        )

    # 축 설정
    ax.set_title(title, fontsize=42, fontweight='bold', pad=35)
    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.set_xlabel('CO$_2$ Affinity (Capacity) [mol/kg$\cdot$Pa]', fontsize=32, fontweight='bold')
    ax.set_ylabel('CO$_2$/N$_2$ Selectivity (Purity)', fontsize=32, fontweight='bold')
    
    # 🌟 [요구사항] X축 범위를 유의미한 구간으로 최적화 (5e-7 ~ 5e-4)
    ax.set_xlim(5e-7, 5e-4)
    ax.grid(True, which="both", ls="--", alpha=0.4)
    ax.tick_params(labelsize=26)

    # 🌟 [요구사항] 범례 박스 15% 이상 확대 및 'X' 기호 초과 방지
    handles, labels = ax.get_legend_handles_labels()
    if handles:
        legend = ax.legend(
            handles=handles, labels=labels,
            bbox_to_anchor=(1.02, 1), loc='upper left', 
            borderaxespad=0.5, 
            title='Metal / OMS Status', title_fontsize=32, fontsize=26, 
            markerscale=2.5,
            labelspacing=2.2,   # 항목 간 수직 간격 확대
            handletextpad=1.8,  # 기호와 글자 사이 간격 확대
            borderpad=2.0       # 범례 박스 내부 여백 대폭 확대 (기호 초과 방지)
        )
        legend.get_frame().set_edgecolor('black')
        legend.get_frame().set_linewidth(2.5)

    plt.tight_layout()
    plt.savefig(filename, dpi=600, bbox_inches='tight')
    plt.show()

# ==========================================
# 5. 4종 그래프 도출
# ==========================================
print("📈 개별 그룹 및 종합 프론티어 도출을 시작합니다...")

save_perfect_plot(df_final[df_final['Group'] == 'N2 Sieving'], '(a) N2-Sieving Group Frontier', 'Frontier_a_N2Sieving_Final.pdf')
save_perfect_plot(df_final[df_final['Group'] == 'High Flux'], '(b) High-Flux Group Frontier', 'Frontier_b_HighFlux_Final.pdf')
save_perfect_plot(df_final[df_final['Group'] == 'Tier 1 & 2'], '(c) Tier 1 & 2 Anti-Humidity Frontier', 'Frontier_c_Tier1_2_Final.pdf')
save_perfect_plot(df_final, '(d) Combined Performance Frontier', 'Frontier_d_Combined_Final.pdf')

print("\n🎉 모든 요구사항이 반영된 고해상도 시각화가 완료되었습니다!")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import ast
import os

print("🚀 금속 색상 고정 및 물리량/범례 최적화 최종 시각화 시작...")

# 1. 데이터 로드 (For Python 파일 사용)
files = {
    'High Flux': "High_Flux_Final_Metrics_For_Python.csv",
    'N2 Sieving': "N2_Sieving_Final_Metrics_For_Python.csv",
    'Tier 1 & 2': "Tier1_Tier2_Anti_Humidity_Targets_For_Python.csv"
}

dfs = []
for name, path in files.items():
    if os.path.exists(path):
        temp_df = pd.read_csv(path)
        temp_df['Group'] = name
        dfs.append(temp_df)

df_all = pd.concat(dfs, ignore_index=True)
df_all['Short_ID'] = df_all['Readable_MOF_ID'].apply(lambda x: str(x).split(' ')[0].replace('MOF-', ''))

# 금속 및 OMS 추출 함수
def parse_chem(chem_str):
    try:
        d = ast.literal_eval(str(chem_str))
        return pd.Series([d.get('metal_type', 'Unknown'), d.get('has_OMS', 'Unknown')])
    except:
        return pd.Series(["Unknown", "Unknown"])

df_all[['Metal', 'Has_OMS']] = df_all['Chemistry (Metal)'].apply(parse_chem)

# 🌟 물리량 단위 환산 (CSV의 mmol/g를 PDF 기준 mol/kg·Pa로 변환)
df_all['SI_Affinity'] = df_all['Working Capacity (mmol/g)'] * 1e-5

# 🌟 [요구사항] 금속 색상 영구 박제 (모든 금속에 대해 색상 지정)
set1_hex = ['#e41a1c', '#377eb8', '#4daf4a', '#984ea3', '#ff7f00', '#ffff33', '#a65628', '#f781bf', '#999999', '#a6cee3', '#1f78b4', '#b2df8a']
# 전체 데이터의 모든 금속 리스트를 먼저 뽑아 정렬한 후 색상을 고정합니다.
all_metals = sorted(df_all['Metal'].unique())
universal_palette = {m: set1_hex[i % len(set1_hex)] for i, m in enumerate(all_metals)}

oms_markers = {'No': 'o', 'Yes': 'X', 'Unknown': 's'}

# 중복 제거 (Tier 우선순위)
priority = {'Tier 1 & 2': 1, 'High Flux': 2, 'N2 Sieving': 3}
df_all['Priority'] = df_all['Group'].map(priority)
df_final = df_all.sort_values(['Priority', 'Performance Score'], ascending=[True, False]).drop_duplicates(subset=['Short_ID'])

# ==========================================
# 2. 시각화 함수 (범위 최적화 및 범례 박스 스케일업)
# ==========================================
def save_perfect_plot(df, title, filename):
    plt.style.use('default')
    sns.set_context("talk", font_scale=2.0) 
    sns.set_style("ticks")

    fig, ax = plt.subplots(figsize=(24, 18))

    # 산점도: 영구 박제된 universal_palette 적용
    sns.scatterplot(
        data=df, x='SI_Affinity', y='Selectivity', 
        hue='Metal', style='Has_OMS', markers=oms_markers,
        palette=universal_palette, s=800, alpha=0.85, edgecolor='black', ax=ax, zorder=5
    )

    # 텍스트 라벨링
    top_n = 5 if "Combined" in title else 3
    top_targets = df.nlargest(top_n, 'Performance Score')
    for _, row in top_targets.iterrows():
        ax.text(
            row['SI_Affinity'] * 1.08, row['Selectivity'], 
            row['Short_ID'], fontsize=22, fontweight='bold', color='black', zorder=10
        )

    # 축 설정
    ax.set_title(title, fontsize=42, fontweight='bold', pad=35)
    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.set_xlabel('CO$_2$ Affinity (Capacity) [mol/kg$\cdot$Pa]', fontsize=32, fontweight='bold')
    ax.set_ylabel('CO$_2$/N$_2$ Selectivity (Purity)', fontsize=32, fontweight='bold')
    
    # 🌟 [요구사항] X축 범위를 유의미한 구간으로 타이트하게 조정 (5e-7 ~ 5e-4)
    ax.set_xlim(5e-7, 5e-4)
    ax.grid(True, which="both", ls="--", alpha=0.4)
    ax.tick_params(labelsize=26)

    # 🌟 [요구사항] 범례 박스 15% 이상 대폭 확대 및 기호 초과 방지
    handles, labels = ax.get_legend_handles_labels()
    if handles:
        legend = ax.legend(
            handles=handles, labels=labels,
            bbox_to_anchor=(1.02, 1), loc='upper left', 
            borderaxespad=0.5, 
            title='Metal / OMS Status', title_fontsize=32, fontsize=26, 
            markerscale=2.5,
            labelspacing=2.5,   # 항목 간 수직 간격 대폭 확대
            handletextpad=2.0,  # 기호와 텍스트 사이 간격 확대
            borderpad=2.5       # 범례 박스 내부 여백을 2.5로 키워 박스 크기 확대
        )
        legend.get_frame().set_edgecolor('black')
        legend.get_frame().set_linewidth(2.5)

    plt.tight_layout()
    plt.savefig(filename, dpi=600, bbox_inches='tight')
    plt.show()

# ==========================================
# 3. 4종 그래프 도출
# ==========================================
save_perfect_plot(df_final[df_final['Group'] == 'N2 Sieving'], '(a) N2-Sieving Group Frontier', 'Frontier_a_N2Sieving_Final.pdf')
save_perfect_plot(df_final[df_final['Group'] == 'High Flux'], '(b) High-Flux Group Frontier', 'Frontier_b_HighFlux_Final.pdf')
save_perfect_plot(df_final[df_final['Group'] == 'Tier 1 & 2'], '(c) Tier 1 & 2 Anti-Humidity Frontier', 'Frontier_c_Tier1_2_Final.pdf')
save_perfect_plot(df_final, '(d) Combined Performance Frontier', 'Frontier_d_Combined_Final.pdf')

print("\n🎉 모든 요구사항이 반영된 최종 시각화가 완료되었습니다!")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np
import ast
import seaborn as sns

# 1. 환경 설정
mpl.rcParams['font.family'] = 'sans-serif'
mpl.rcParams['font.sans-serif'] = ['Arial', 'DejaVu Sans', 'Liberation Sans']
mpl.rcParams['axes.unicode_minus'] = False 

# 2. 기존 데이터 처리 (N2-Sieving, High-Flux)
# df는 주피터 메모리에 이미 로드되어 있다고 가정합니다.
df['metal_info'] = df.iloc[:, 6] 
df['GEMC_data'] = df.iloc[:, 9]  

df['PLD_value'] = df['Zeopp'].apply(lambda x: x.get('PLD', 0) if isinstance(x, dict) else 0)
df['LCD_value'] = df['Zeopp'].apply(lambda x: x.get('LCD', 0) if isinstance(x, dict) else 0)
df['water_class'] = df['water'].apply(lambda x: x.get('water_classification', 'unknown') if isinstance(x, dict) else 'unknown')

n2_sieving = df[
    (df['PLD_value'] >= 3.3) & (df['PLD_value'] <= 3.6) & 
    (df['water_class'] == 'weak')
].copy()
n2_sieving['Filter_Type'] = 'N2-Sieving (Narrow)'

high_flux = df[
    (df['PLD_value'] >= 3.7) & (df['PLD_value'] <= 4.2) & 
    (df['LCD_value'] >= 5.5) & 
    (df['water_class'] == 'weak')
].copy()
high_flux['Filter_Type'] = 'High-Flux (Wide)'

combined_df = pd.concat([n2_sieving, high_flux], ignore_index=True)

# 3. 기존 데이터용 추출 함수
def extract_metrics(gemc_data):
    try:
        if isinstance(gemc_data, str):
            gemc_data = ast.literal_eval(gemc_data)
        if isinstance(gemc_data, dict) and 'Widom' in gemc_data:
            widom = gemc_data['Widom']
            return pd.Series([widom[0], widom[0]/widom[1] if widom[1]>0 else 0])
    except: pass
    return pd.Series([np.nan, np.nan])

def extract_metal_category(metal_data):
    try:
        if isinstance(metal_data, str):
            metal_data = ast.literal_eval(metal_data)
        if isinstance(metal_data, dict):
            return f"{metal_data.get('metal_type', 'Unknown')} (OMS: {metal_data.get('has_OMS', 'Unknown')})"
    except: pass
    return "Unknown"

combined_df[['CO2_Affinity', 'Selectivity']] = combined_df['GEMC_data'].apply(extract_metrics)
combined_df['Chemistry'] = combined_df['metal_info'].apply(extract_metal_category)
final_clean_df = combined_df.dropna(subset=['CO2_Affinity', 'Selectivity'])

# =================================================================
# 🌟 [수정] Tier 1 전용 로드 및 합산 (KeyError 방지 로직)
# =================================================================
try:
    # 1. 파일 로드
    tier_path = 'Tier1_Ultimate_Targets_ColorSynced.csv'
    t1_raw = pd.read_csv(tier_path)
    
    # 2. Tier 1 조건 필터링 (구조 데이터 기반)
    # 파일 내에 PLD_value, Metal, Has_OMS 컬럼이 있는지 확인하며 필터링
    t1_filtered = t1_raw[
        (t1_raw['Metal'].isin(['Zn', 'Co'])) & 
        (t1_raw['Has_OMS'] == 'No') & 
        (t1_raw['PLD_value'] >= 3.2) & (t1_raw['PLD_value'] <= 3.8)
    ].copy()
    
    # 3. ⚠️ 중요: 기존 데이터프레임과 컬럼명 및 형식 강제 통일
    # Tier 1 파일은 이미 Affinity와 Selectivity가 계산되어 있으므로 그대로 가져옵니다.
    t1_to_add = pd.DataFrame()
    t1_to_add['CO2_Affinity'] = t1_filtered['CO2_Affinity']
    t1_to_add['Selectivity'] = t1_filtered['Selectivity']
    t1_to_add['Chemistry'] = t1_filtered.apply(lambda x: f"{x['Metal']} (OMS: {x['Has_OMS']})", axis=1)
    t1_to_add['Filter_Type'] = 'Tier 1 (Ultimate Target)'
    
    # 4. 데이터 합치기
    final_clean_df = pd.concat([final_clean_df, t1_to_add], ignore_index=True)
    print(f"✅ 주피터 로드 성공! Tier 1 타겟 {len(t1_to_add)}개가 그래프에 추가되었습니다.")

except Exception as e:
    print(f"❌ 데이터 로드 에러: {e}")
    print("현재 파일의 컬럼명 확인:", t1_raw.columns.tolist()) # 에러 시 컬럼명 출력
# =================================================================

# 5. 시각화
plt.figure(figsize=(14, 9))
sns.scatterplot(
    data=final_clean_df, x='CO2_Affinity', y='Selectivity', 
    hue='Chemistry', style='Filter_Type', s=200, alpha=0.85, edgecolor='black', palette='tab20'
)

plt.xscale('log')
plt.yscale('log')
plt.title('MOF CCUS Frontier: Integrated Analysis (Tier 1 Included)', fontsize=18, fontweight='bold', pad=15)
plt.xlabel('CO2 Affinity (Henry\'s Constant)', fontsize=14)
plt.ylabel('Selectivity (CO2 / N2)', fontsize=14)
plt.grid(True, which="both", ls="--", alpha=0.4)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=11, title='Material Properties')
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np
import ast
import seaborn as sns

# 🌟 1. 철통 방어: 폰트 및 마이너스 기호 강제 설정
mpl.rcParams['font.family'] = 'sans-serif'
mpl.rcParams['font.sans-serif'] = ['Arial', 'DejaVu Sans', 'Liberation Sans']
mpl.rcParams['axes.unicode_minus'] = False # 마이너스 기호 깨짐 최종 방어선

# 2. 필요한 열 타겟팅 (G열=인덱스 6, J열=인덱스 9)
df['metal_info'] = df.iloc[:, 6] # G열: 금속 및 OMS 정보
df['GEMC_data'] = df.iloc[:, 9]  # J열: Widom 시뮬레이션 성적표

# 기본 물리량 세팅
df['PLD_value'] = df['Zeopp'].apply(lambda x: x.get('PLD', 0) if isinstance(x, dict) else 0)
df['LCD_value'] = df['Zeopp'].apply(lambda x: x.get('LCD', 0) if isinstance(x, dict) else 0)
df['water_class'] = df['water'].apply(lambda x: x.get('water_classification', 'unknown') if isinstance(x, dict) else 'unknown')

# 3. 그룹 분리 (기술적 명칭 유지)
n2_sieving = df[
    (df['PLD_value'] >= 3.3) & (df['PLD_value'] <= 3.6) & 
    (df['water_class'] == 'weak')
].copy()
n2_sieving['Filter_Type'] = 'N2-Sieving (Narrow)'

high_flux = df[
    (df['PLD_value'] >= 3.7) & (df['PLD_value'] <= 4.2) & 
    (df['LCD_value'] >= 5.5) & 
    (df['water_class'] == 'weak')
].copy()
high_flux['Filter_Type'] = 'High-Flux (Wide)'

# 두 그룹을 하나의 데이터프레임으로 합치기 (Seaborn 처리를 위해)
combined_df = pd.concat([n2_sieving, high_flux], ignore_index=True)

# 4. 데이터 추출 함수들
def extract_metrics(gemc_data):
    try:
        if isinstance(gemc_data, str):
            gemc_data = ast.literal_eval(gemc_data)
        if isinstance(gemc_data, dict) and 'Widom' in gemc_data:
            widom = gemc_data['Widom']
            if isinstance(widom, list) and len(widom) >= 2:
                co2_affinity = widom[0]
                n2_affinity = widom[1]
                selectivity = co2_affinity / n2_affinity if n2_affinity > 0 else 0
                return pd.Series([co2_affinity, selectivity])
    except:
        pass
    return pd.Series([np.nan, np.nan])

def extract_metal_category(metal_data):
    try:
        if isinstance(metal_data, str):
            metal_data = ast.literal_eval(metal_data)
        if isinstance(metal_data, dict):
            metal = metal_data.get('metal_type', 'Unknown')
            oms = metal_data.get('has_OMS', 'Unknown')
            return f"{metal} (OMS: {oms})"
    except:
        pass
    return "Unknown"

# 추출 실행
print("🔄 Extracting Multi-dimensional Data...")
combined_df[['CO2_Affinity', 'Selectivity']] = combined_df['GEMC_data'].apply(extract_metrics)
combined_df['Chemistry'] = combined_df['metal_info'].apply(extract_metal_category)

# 결측치 제거 및 데이터 정리
final_clean_df = combined_df.dropna(subset=['CO2_Affinity', 'Selectivity'])
# Unknown 등 분석에 방해되는 찌꺼기 데이터 필터링
final_clean_df = final_clean_df[final_clean_df['Chemistry'] != 'Unknown (OMS: Unknown)']

# 5. 궁극의 다차원 산점도 (Seaborn)
plt.figure(figsize=(14, 9))

# hue: 색상은 금속 화학종, style: 점의 모양은 N2-Sieving/High-Flux 그룹
ax = sns.scatterplot(
    data=final_clean_df, 
    x='CO2_Affinity', 
    y='Selectivity', 
    hue='Chemistry', 
    style='Filter_Type',
    s=150,           # 점 크기
    alpha=0.85,      # 투명도
    edgecolor='black',
    palette='tab20'  # 색상 팔레트 (다양한 색상 지원)
)

# 축 설정 (로그 스케일)
plt.xscale('log')
plt.yscale('log')

# 디자인 다듬기
plt.title('MOF CCUS Frontier: Chemistry & Structural Analysis', fontsize=18, fontweight='bold', pad=15)
plt.xlabel('CO2 Affinity (Henry\'s Constant) -> Higher Capacity', fontsize=14)
plt.ylabel('Selectivity (CO2 / N2) -> Higher Purity', fontsize=14)
plt.grid(True, which="both", ls="--", alpha=0.4)

# 범례(Legend)를 보기 좋게 바깥으로 빼기
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', borderaxespad=0., fontsize=11, title='Material Properties', title_fontsize=12)

plt.tight_layout()
plt.show()

print(f"📊 Visualization Complete! Total valid MOFs plotted: {len(final_clean_df)}")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np
import ast
import seaborn as sns

# 🌟 1. 철통 방어: 폰트 및 마이너스 기호 강제 설정
mpl.rcParams['font.family'] = 'sans-serif'
mpl.rcParams['font.sans-serif'] = ['Arial', 'DejaVu Sans', 'Liberation Sans']
mpl.rcParams['axes.unicode_minus'] = False 

# =================================================================
# 🌟 2. [오류 해결] 파일 직접 로드 및 그룹 지정
# 복잡한 Zeopp 파싱이나 불안정한 iloc 인덱싱(열 번호 찾기)을 제거하고, 
# 올려주신 3개의 파일을 직접 읽어 그룹(Filter_Type)을 명확히 부여합니다.
# =================================================================
n2_sieving = pd.read_csv("N2_Sieving_Group_63_MOFs.csv")
n2_sieving['Filter_Type'] = 'N2-Sieving (Narrow)'

high_flux = pd.read_csv("High_Flux_Bottleneck_52_MOFs.csv")
high_flux['Filter_Type'] = 'High-Flux (Wide)'

tier1 = pd.read_csv("Tier1_Ultimate_Targets_ColorSynced.csv")
tier1['Filter_Type'] = 'Tier 1 (Ultimate Target)'

# 3개의 데이터 프레임을 하나로 합치기
# 순서: Tier 1 -> High Flux -> N2 Sieving (Tier 1이 중복 제거 시 살아남도록 우선순위 배치)
combined_df = pd.concat([tier1, high_flux, n2_sieving], ignore_index=True)

# 3. 데이터 추출 함수 (유저 코드 완벽 유지)
def extract_metrics(gemc_data):
    try:
        if isinstance(gemc_data, str):
            gemc_data = ast.literal_eval(gemc_data)
        if isinstance(gemc_data, dict) and 'Widom' in gemc_data:
            widom = gemc_data['Widom']
            if isinstance(widom, list) and len(widom) >= 2:
                co2_affinity = widom[0]
                n2_affinity = widom[1]
                selectivity = co2_affinity / n2_affinity if n2_affinity > 0 else 0
                return pd.Series([co2_affinity, selectivity])
    except:
        pass
    return pd.Series([np.nan, np.nan])

def extract_metal_category(metal_data):
    try:
        if isinstance(metal_data, str):
            metal_data = ast.literal_eval(metal_data)
        if isinstance(metal_data, dict):
            metal = metal_data.get('metal_type', 'Unknown')
            oms = metal_data.get('has_OMS', 'Unknown')
            return f"{metal} (OMS: {oms})"
    except:
        pass
    return "Unknown"

print("🔄 Extracting Multi-dimensional Data...")
# ⚠️ 핵심 수정: iloc 대신 명시적으로 'GEMC_data'와 'metal' 컬럼 사용
combined_df[['CO2_Affinity', 'Selectivity']] = combined_df['GEMC_data'].apply(extract_metrics)
combined_df['Chemistry'] = combined_df['metal'].apply(extract_metal_category)

# 결측치 제거 및 데이터 정리
final_clean_df = combined_df.dropna(subset=['CO2_Affinity', 'Selectivity'])
final_clean_df = final_clean_df[final_clean_df['Chemistry'] != 'Unknown (OMS: Unknown)']

# 🌟 4. 식별자(Short ID) 생성 및 중복 제거
# Tier 1 타겟들이 다른 두 그룹에 중복해서 찍히는 것을 방지
def get_short_id(row):
    try:
        mofid = ast.literal_eval(row['id']).get('mofid-v1', '')
        return mofid.split(';')[1][:6].upper() if ';' in mofid else "XXXXXX"
    except: return "XXXXXX"

final_clean_df['Short_ID'] = final_clean_df.apply(get_short_id, axis=1)
final_clean_df = final_clean_df.drop_duplicates(subset=['Short_ID'], keep='first')

# 5. 궁극의 다차원 산점도 (Seaborn)
plt.figure(figsize=(14, 9))

# hue: 색상은 금속 화학종, style: 점의 모양은 N2-Sieving/High-Flux/Tier 1 그룹
ax = sns.scatterplot(
    data=final_clean_df, 
    x='CO2_Affinity', 
    y='Selectivity', 
    hue='Chemistry', 
    style='Filter_Type',
    s=250,           # 점 크기 (가시성을 위해 살짝 키움)
    alpha=0.85,      # 투명도
    edgecolor='black',
    palette='tab20'  # 색상 팔레트
)

# 축 설정 (로그 스케일)
plt.xscale('log')
plt.yscale('log')

# 디자인 다듬기
plt.title('MOF CCUS Frontier: Chemistry & Structural Analysis', fontsize=20, fontweight='bold', pad=15)
plt.xlabel('CO$_2$ Affinity (Henry\'s Constant) -> Higher Capacity', fontsize=16)
plt.ylabel('Selectivity (CO$_2$ / N$_2$) -> Higher Purity', fontsize=16)
plt.grid(True, which="both", ls="--", alpha=0.4)
plt.tick_params(labelsize=14)

# 범례(Legend)를 보기 좋게 바깥으로 빼기
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left', borderaxespad=0., fontsize=12, title='Material Properties', title_fontsize=14)

plt.tight_layout()
plt.savefig('Final_Preferred_Frontier.pdf', dpi=600, bbox_inches='tight')
plt.show()

print(f"📊 Visualization Complete! Total unique valid MOFs plotted: {len(final_clean_df)}")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import ast
import seaborn as sns

# 🌟 1. 철통 방어: 폰트 및 마이너스 기호 강제 설정
mpl.rcParams['font.family'] = 'sans-serif'
mpl.rcParams['font.sans-serif'] = ['Arial', 'DejaVu Sans', 'Liberation Sans']
mpl.rcParams['axes.unicode_minus'] = False 

# =================================================================
# 2. 파일 로드 및 그룹 지정
# =================================================================
n2_sieving = pd.read_csv("N2_Sieving_Group_63_MOFs.csv")
n2_sieving['Filter_Type'] = 'N2-Sieving (Narrow)'

high_flux = pd.read_csv("High_Flux_Bottleneck_52_MOFs.csv")
high_flux['Filter_Type'] = 'High-Flux (Wide)'

tier1 = pd.read_csv("Tier1_Ultimate_Targets_ColorSynced.csv")
tier1['Filter_Type'] = 'Tier 1 (Ultimate Target)'

# 통합
combined_df = pd.concat([tier1, high_flux, n2_sieving], ignore_index=True)

# =================================================================
# 3. [오류 해결] 데이터 추출 및 정밀한 중복 제거
# =================================================================
def extract_metal_category(metal_data):
    try:
        if isinstance(metal_data, str):
            metal_data = ast.literal_eval(metal_data)
        if isinstance(metal_data, dict):
            metal = metal_data.get('metal_type', 'Unknown')
            oms = metal_data.get('has_OMS', 'Unknown')
            return f"{metal} (OMS: {oms})"
    except:
        pass
    return "Unknown"

# 금속 정보 추출
combined_df['Chemistry'] = combined_df['metal'].apply(extract_metal_category)
final_clean_df = combined_df[combined_df['Chemistry'] != 'Unknown (OMS: Unknown)'].copy()

# ⚠️ 결측치 제거 방지: GEMC_data를 다시 파싱하지 않고, CSV에 이미 있는 CO2_Affinity/Selectivity 값을 그대로 사용합니다.
final_clean_df = final_clean_df.dropna(subset=['CO2_Affinity', 'Selectivity'])

# ⚠️ 중복 제거 오류 수정: 6자리 Short_ID가 아닌, 완벽한 고유 식별자(id)를 기준으로 하여 억울하게 지워지는 재료를 구출합니다.
final_clean_df = final_clean_df.drop_duplicates(subset=['id'], keep='first')

# =================================================================
# 4. 일관성을 위한 색상 및 스타일 고정
# 개별 그래프를 그릴 때 카테고리가 빠져도 색상이 뒤죽박죽 변하지 않도록 못을 박습니다.
# =================================================================
unique_chems = sorted(final_clean_df['Chemistry'].unique())
palette_colors = sns.color_palette('tab20', len(unique_chems))
fixed_palette = dict(zip(unique_chems, palette_colors))
style_order = ['Tier 1 (Ultimate Target)', 'High-Flux (Wide)', 'N2-Sieving (Narrow)']

# =================================================================
# 5. 초고해상도 다차원 산점도 출력 함수
# =================================================================
def draw_frontier_plot(df, title, filename):
    plt.figure(figsize=(14, 9))
    
    ax = sns.scatterplot(
        data=df, 
        x='CO2_Affinity', 
        y='Selectivity', 
        hue='Chemistry', 
        style='Filter_Type',
        hue_order=unique_chems, # 누락된 금속이 있어도 범례 순서 유지
        style_order=style_order,
        s=250,           
        alpha=0.85,      
        edgecolor='black',
        palette=fixed_palette
    )

    # 축 설정 (로그 스케일)
    plt.xscale('log')
    plt.yscale('log')
    
    # x축, y축 범위를 전체 데이터 기준으로 고정하여, 그래프가 바뀌어도 비율이 유지되게 함
    plt.xlim(final_clean_df['CO2_Affinity'].min() * 0.5, final_clean_df['CO2_Affinity'].max() * 2)
    plt.ylim(final_clean_df['Selectivity'].min() * 0.5, final_clean_df['Selectivity'].max() * 2)

    # 디자인
    plt.title(title, fontsize=20, fontweight='bold', pad=15)
    plt.xlabel('CO$_2$ Affinity (Henry\'s Constant) -> Higher Capacity', fontsize=16)
    plt.ylabel('Selectivity (CO$_2$ / N$_2$) -> Higher Purity', fontsize=16)
    plt.grid(True, which="both", ls="--", alpha=0.4)
    plt.tick_params(labelsize=14)

    # 범례 설정
    plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left', borderaxespad=0., fontsize=12, title='Material Properties', title_fontsize=14)

    plt.tight_layout()
    # 논문용 PDF와 확인용 PNG 동시 저장 (600 DPI)
    plt.savefig(filename.replace('.png', '.pdf'), dpi=600, bbox_inches='tight')
    plt.savefig(filename, dpi=600, bbox_inches='tight')
    plt.show()

# =================================================================
# 6. 한 번의 실행으로 4종 그래프 자동 도출
# =================================================================
print(f"✅ 로직 수정 완료! 총 확보된 유효 타겟 개수: {len(final_clean_df)}개\n")

print("📈 1/4. [종합 프론티어] 모든 데이터 출력 중...")
draw_frontier_plot(final_clean_df, 'MOF CCUS Frontier: Integrated Analysis (All Groups)', 'Final_Frontier_Combined.png')

print("📈 2/4. [Tier 1 전용] Ultimate Target 출력 중...")
draw_frontier_plot(final_clean_df[final_clean_df['Filter_Type'] == 'Tier 1 (Ultimate Target)'], 'MOF CCUS Frontier: Tier 1 Only', 'Final_Frontier_Tier1.png')

print("📈 3/4. [High-Flux 전용] Wide Bottleneck 출력 중...")
draw_frontier_plot(final_clean_df[final_clean_df['Filter_Type'] == 'High-Flux (Wide)'], 'MOF CCUS Frontier: High-Flux Only', 'Final_Frontier_HighFlux.png')

print("📈 4/4. [N2-Sieving 전용] Narrow Bottleneck 출력 중...")
draw_frontier_plot(final_clean_df[final_clean_df['Filter_Type'] == 'N2-Sieving (Narrow)'], 'MOF CCUS Frontier: N2-Sieving Only', 'Final_Frontier_N2Sieving.png')

print("🎉 모든 개별 및 종합 그래프 도출 완료! (작업 폴더 내 600DPI 파일 확인)")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np
import ast
import seaborn as sns

# 1. 환경 설정
mpl.rcParams['font.family'] = 'sans-serif'
mpl.rcParams['axes.unicode_minus'] = False 

# 2. 3개 파일 직접 로드 및 그룹 지정
n2_sieving = pd.read_csv("N2_Sieving_Group_63_MOFs.csv")
n2_sieving['Filter_Type'] = 'N2-Sieving (Narrow)'

high_flux = pd.read_csv("High_Flux_Bottleneck_52_MOFs.csv")
high_flux['Filter_Type'] = 'High-Flux (Wide)'

tier1 = pd.read_csv("Tier1_Ultimate_Targets_ColorSynced.csv")
tier1['Filter_Type'] = 'Tier 1 (Ultimate Target)'

# 전체 데이터 통합
combined_df = pd.concat([tier1, high_flux, n2_sieving], ignore_index=True)

# =================================================================
# 🌟 3. 데이터 추출 로직 통일 (모든 그룹 동일 적용)
# =================================================================
def extract_metrics(gemc_data):
    try:
        if isinstance(gemc_data, str):
            gemc_data = ast.literal_eval(gemc_data)
        if isinstance(gemc_data, dict) and 'Widom' in gemc_data:
            widom = gemc_data['Widom']
            if isinstance(widom, list) and len(widom) >= 2:
                co2_affinity = widom[0] # Henry's Constant
                n2_affinity = widom[1]
                selectivity = co2_affinity / n2_affinity if n2_affinity > 0 else 0
                return pd.Series([co2_affinity, selectivity])
    except: pass
    return pd.Series([np.nan, np.nan])

def extract_metal_category(metal_data):
    try:
        if isinstance(metal_data, str):
            metal_data = ast.literal_eval(metal_data)
        if isinstance(metal_data, dict):
            return f"{metal_data.get('metal_type', 'Unknown')} (OMS: {metal_data.get('has_OMS', 'Unknown')})"
    except: pass
    return "Unknown"

print("🔄 모든 데이터를 GEMC_data(Widom) 기준으로 통일 추출 중...")
# 모든 그룹에 대해 동일한 파싱 로직 적용
combined_df[['CO2_Affinity', 'Selectivity']] = combined_df['GEMC_data'].apply(extract_metrics)
combined_df['Chemistry'] = combined_df['metal'].apply(extract_metal_category)

# 결측치 제거 및 데이터 정리
final_clean_df = combined_df.dropna(subset=['CO2_Affinity', 'Selectivity'])
final_clean_df = final_clean_df[final_clean_df['Chemistry'] != 'Unknown (OMS: Unknown)']

# 중복 제거 (Tier 1을 최우선으로 남김)
# 동일한 id라면 이제 CO2_Affinity 좌표가 완벽히 일치하게 됩니다.
final_clean_df = final_clean_df.drop_duplicates(subset=['id'], keep='first')

# 4. 색상 및 스타일 설정 박제
unique_chems = sorted(final_clean_df['Chemistry'].unique())
fixed_palette = dict(zip(unique_chems, sns.color_palette('tab20', len(unique_chems))))
style_order = ['Tier 1 (Ultimate Target)', 'High-Flux (Wide)', 'N2-Sieving (Narrow)']

# =================================================================
# 5. 그래프 출력 함수 (600DPI & 개별 도출)
# =================================================================
def draw_frontier(df, title, filename):
    plt.figure(figsize=(14, 9))
    sns.scatterplot(
        data=df, x='CO2_Affinity', y='Selectivity', 
        hue='Chemistry', style='Filter_Type',
        hue_order=unique_chems, style_order=style_order,
        s=250, alpha=0.85, edgecolor='black', palette=fixed_palette
    )
    plt.xscale('log')
    plt.yscale('log')
    
    # 전체 데이터 기준 축 범위 고정
    plt.xlim(final_clean_df['CO2_Affinity'].min()*0.5, final_clean_df['CO2_Affinity'].max()*2)
    plt.ylim(final_clean_df['Selectivity'].min()*0.5, final_clean_df['Selectivity'].max()*2)

    plt.title(title, fontsize=18, fontweight='bold', pad=15)
    plt.xlabel('CO$_2$ Affinity (Henry\'s Constant) -> Higher Capacity', fontsize=14)
    plt.ylabel('Selectivity (CO$_2$ / N$_2$) -> Higher Purity', fontsize=14)
    plt.grid(True, which="both", ls="--", alpha=0.4)
    plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=11, title='Material Properties')
    plt.tight_layout()
    plt.savefig(filename, dpi=600, bbox_inches='tight')
    plt.show()

# 실행
print(f"📊 최종 유효 타겟 개수: {len(final_clean_df)} (중복 제외 고유 재료 수)")
draw_frontier(final_clean_df, 'Integrated Frontier (All Groups Unified by Widom)', 'Combined_Widom_Frontier.png')
draw_frontier(final_clean_df[final_clean_df['Filter_Type'] == 'Tier 1 (Ultimate Target)'], 'Tier 1 Only', 'Tier1_Widom_Frontier.png')

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np
import ast
import seaborn as sns

# 🌟 1. 철통 방어: 폰트 및 마이너스 기호 강제 설정
mpl.rcParams['font.family'] = 'sans-serif'
mpl.rcParams['font.sans-serif'] = ['Arial', 'DejaVu Sans', 'Liberation Sans']
mpl.rcParams['axes.unicode_minus'] = False 

# =================================================================
# 2. 파일 로드 및 그룹 지정
# =================================================================
n2_sieving = pd.read_csv("N2_Sieving_Group_63_MOFs.csv")
n2_sieving['Filter_Type'] = 'N2-Sieving (Narrow)'

high_flux = pd.read_csv("High_Flux_Bottleneck_52_MOFs.csv")
high_flux['Filter_Type'] = 'High-Flux (Wide)'

tier1 = pd.read_csv("Tier1_Ultimate_Targets_ColorSynced.csv")
tier1['Filter_Type'] = 'Tier 1 (Ultimate Target)'

# 통합 (우선순위: Tier 1 -> High Flux -> N2 Sieving)
combined_df = pd.concat([tier1, high_flux, n2_sieving], ignore_index=True)

# =================================================================
# 3. [핵심 수정] 모든 데이터에 대해 동일한 GEMC_data(Widom) 파싱 적용
# =================================================================
def extract_metrics(gemc_data):
    try:
        if isinstance(gemc_data, str):
            gemc_data = ast.literal_eval(gemc_data)
        if isinstance(gemc_data, dict) and 'Widom' in gemc_data:
            widom = gemc_data['Widom']
            if isinstance(widom, list) and len(widom) >= 2:
                co2_affinity = widom[0]  # Henry's Constant
                n2_affinity = widom[1]
                selectivity = co2_affinity / n2_affinity if n2_affinity > 0 else 0
                return pd.Series([co2_affinity, selectivity])
    except:
        pass
    return pd.Series([np.nan, np.nan])

def extract_metal_category(metal_data):
    try:
        if isinstance(metal_data, str):
            metal_data = ast.literal_eval(metal_data)
        if isinstance(metal_data, dict):
            metal = metal_data.get('metal_type', 'Unknown')
            oms = metal_data.get('has_OMS', 'Unknown')
            return f"{metal} (OMS: {oms})"
    except:
        pass
    return "Unknown"

print("🔄 모든 데이터를 GEMC_data(Widom) 기준으로 통일 추출 중...")

# ⚠️ 모든 데이터가 공평하게 Widom 원시 데이터로부터 좌표를 계산합니다.
combined_df[['CO2_Affinity', 'Selectivity']] = combined_df['GEMC_data'].apply(extract_metrics)
combined_df['Chemistry'] = combined_df['metal'].apply(extract_metal_category)

# 결측치(파싱 실패 등) 제거 및 찌꺼기 데이터 필터링
final_clean_df = combined_df.dropna(subset=['CO2_Affinity', 'Selectivity'])
final_clean_df = final_clean_df[final_clean_df['Chemistry'] != 'Unknown (OMS: Unknown)'].copy()

# ⚠️ 완벽한 고유 식별자(id) 기준 중복 제거
# Widom 값을 통일했기 때문에, id가 같으면 좌표도 100% 일치합니다.
final_clean_df = final_clean_df.drop_duplicates(subset=['id'], keep='first')

# =================================================================
# 4. 일관성을 위한 색상 및 스타일 고정
# =================================================================
unique_chems = sorted(final_clean_df['Chemistry'].unique())
palette_colors = sns.color_palette('tab20', len(unique_chems))
fixed_palette = dict(zip(unique_chems, palette_colors))
style_order = ['Tier 1 (Ultimate Target)', 'High-Flux (Wide)', 'N2-Sieving (Narrow)']

# =================================================================
# 5. 초고해상도 다차원 산점도 출력 함수 (캔버스 20% 확대 적용)
# =================================================================
def draw_frontier_plot(df, title, filename):
    # 🌟 기존 14 x 9 비율 유지하며 크기 20% 증가 -> 16.8 x 10.8
    plt.figure(figsize=(16.8, 10.8))
    
    ax = sns.scatterplot(
        data=df, 
        x='CO2_Affinity', 
        y='Selectivity', 
        hue='Chemistry', 
        style='Filter_Type',
        hue_order=unique_chems, 
        style_order=style_order,
        s=300,           # 캔버스가 커진 만큼 마커 크기도 살짝(250->300) 상향 조정
        alpha=0.85,      
        edgecolor='black',
        palette=fixed_palette
    )

    # 축 설정 (로그 스케일)
    plt.xscale('log')
    plt.yscale('log')
    
    # x축, y축 범위를 전체 데이터 기준으로 고정 (그래프가 바뀌어도 비율 유지)
    plt.xlim(final_clean_df['CO2_Affinity'].min() * 0.5, final_clean_df['CO2_Affinity'].max() * 2)
    plt.ylim(final_clean_df['Selectivity'].min() * 0.5, final_clean_df['Selectivity'].max() * 2)

    # 디자인 (폰트 크기도 캔버스 확대에 맞춰 약간 상향 조정)
    plt.title(title, fontsize=24, fontweight='bold', pad=18)
    plt.xlabel('CO$_2$ Affinity (Henry\'s Constant) -> Higher Capacity', fontsize=18)
    plt.ylabel('Selectivity (CO$_2$ / N$_2$) -> Higher Purity', fontsize=18)
    plt.grid(True, which="both", ls="--", alpha=0.4)
    plt.tick_params(labelsize=16)

    # 범례 설정
    plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left', borderaxespad=0., fontsize=14, title='Material Properties', title_fontsize=16)

    plt.tight_layout()
    # 논문용 PDF와 확인용 PNG 동시 저장 (600 DPI)
    plt.savefig(filename.replace('.png', '.pdf'), dpi=600, bbox_inches='tight')
    plt.savefig(filename, dpi=600, bbox_inches='tight')
    plt.show()

# =================================================================
# 6. 한 번의 실행으로 4종 그래프 자동 도출
# =================================================================
print(f"✅ 로직 수정 완료! 총 확보된 유효 타겟 개수: {len(final_clean_df)}개\n")

print("📈 1/4. [종합 프론티어] 모든 데이터 출력 중...")
draw_frontier_plot(final_clean_df, 'MOF CCUS Frontier: Integrated Analysis (All Groups)', 'Final_Frontier_Combined.png')

print("📈 2/4. [Tier 1 전용] Ultimate Target 출력 중...")
draw_frontier_plot(final_clean_df[final_clean_df['Filter_Type'] == 'Tier 1 (Ultimate Target)'], 'MOF CCUS Frontier: Tier 1 Only', 'Final_Frontier_Tier1.png')

print("📈 3/4. [High-Flux 전용] Wide Bottleneck 출력 중...")
draw_frontier_plot(final_clean_df[final_clean_df['Filter_Type'] == 'High-Flux (Wide)'], 'MOF CCUS Frontier: High-Flux Only', 'Final_Frontier_HighFlux.png')

print("📈 4/4. [N2-Sieving 전용] Narrow Bottleneck 출력 중...")
draw_frontier_plot(final_clean_df[final_clean_df['Filter_Type'] == 'N2-Sieving (Narrow)'], 'MOF CCUS Frontier: N2-Sieving Only', 'Final_Frontier_N2Sieving.png')

print("🎉 모든 개별 및 종합 그래프 도출 완료! (작업 폴더 내 600DPI 파일 확인)")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np
import ast
import seaborn as sns

# 1. 환경 설정
mpl.rcParams['font.family'] = 'sans-serif'
mpl.rcParams['axes.unicode_minus'] = False 

# 2. 3개 파일 직접 로드 및 그룹 지정
n2_sieving = pd.read_csv("N2_Sieving_Group_63_MOFs.csv")
n2_sieving['Filter_Type'] = 'N2-Sieving (Narrow)'

high_flux = pd.read_csv("High_Flux_Bottleneck_52_MOFs.csv")
high_flux['Filter_Type'] = 'High-Flux (Wide)'

tier1 = pd.read_csv("Tier1_Ultimate_Targets_ColorSynced.csv")
tier1['Filter_Type'] = 'Tier 1 (Ultimate Target)'

# 전체 데이터 통합
combined_df = pd.concat([tier1, high_flux, n2_sieving], ignore_index=True)

# =================================================================
# 🌟 3. 데이터 추출 로직 통일 (모든 그룹 동일 적용)
# =================================================================
def extract_metrics(gemc_data):
    try:
        if isinstance(gemc_data, str):
            gemc_data = ast.literal_eval(gemc_data)
        if isinstance(gemc_data, dict) and 'Widom' in gemc_data:
            widom = gemc_data['Widom']
            if isinstance(widom, list) and len(widom) >= 2:
                co2_affinity = widom[0] # Henry's Constant
                n2_affinity = widom[1]
                selectivity = co2_affinity / n2_affinity if n2_affinity > 0 else 0
                return pd.Series([co2_affinity, selectivity])
    except: pass
    return pd.Series([np.nan, np.nan])

def extract_metal_category(metal_data):
    try:
        if isinstance(metal_data, str):
            metal_data = ast.literal_eval(metal_data)
        if isinstance(metal_data, dict):
            return f"{metal_data.get('metal_type', 'Unknown')} (OMS: {metal_data.get('has_OMS', 'Unknown')})"
    except: pass
    return "Unknown"

print("🔄 모든 데이터를 GEMC_data(Widom) 기준으로 통일 추출 중...")
# 모든 그룹에 대해 동일한 파싱 로직 적용
combined_df[['CO2_Affinity', 'Selectivity']] = combined_df['GEMC_data'].apply(extract_metrics)
combined_df['Chemistry'] = combined_df['metal'].apply(extract_metal_category)

# 결측치 제거 및 데이터 정리
final_clean_df = combined_df.dropna(subset=['CO2_Affinity', 'Selectivity'])
final_clean_df = final_clean_df[final_clean_df['Chemistry'] != 'Unknown (OMS: Unknown)']

# 중복 제거 (Tier 1을 최우선으로 남김)
# 동일한 id라면 이제 CO2_Affinity 좌표가 완벽히 일치하게 됩니다.
final_clean_df = final_clean_df.drop_duplicates(subset=['id'], keep='first')

# 4. 색상 및 스타일 설정 박제
unique_chems = sorted(final_clean_df['Chemistry'].unique())
fixed_palette = dict(zip(unique_chems, sns.color_palette('tab20', len(unique_chems))))
style_order = ['Tier 1 (Ultimate Target)', 'High-Flux (Wide)', 'N2-Sieving (Narrow)']

# =================================================================
# 5. 그래프 출력 함수 (600DPI & 개별 도출)
# =================================================================
def draw_frontier(df, title, filename):
    plt.figure(figsize=(14, 9))
    sns.scatterplot(
        data=df, x='CO2_Affinity', y='Selectivity', 
        hue='Chemistry', style='Filter_Type',
        hue_order=unique_chems, style_order=style_order,
        s=250, alpha=0.85, edgecolor='black', palette=fixed_palette
    )
    plt.xscale('log')
    plt.yscale('log')
    
    # 전체 데이터 기준 축 범위 고정
    plt.xlim(final_clean_df['CO2_Affinity'].min()*0.5, final_clean_df['CO2_Affinity'].max()*2)
    plt.ylim(final_clean_df['Selectivity'].min()*0.5, final_clean_df['Selectivity'].max()*2)

    plt.title(title, fontsize=18, fontweight='bold', pad=15)
    plt.xlabel('CO$_2$ Affinity (Henry\'s Constant) -> Higher Capacity', fontsize=14)
    plt.ylabel('Selectivity (CO$_2$ / N$_2$) -> Higher Purity', fontsize=14)
    plt.grid(True, which="both", ls="--", alpha=0.4)
    plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=11, title='Material Properties')
    plt.tight_layout()
    plt.savefig(filename, dpi=600, bbox_inches='tight')
    plt.show()

# 실행
print(f"📊 최종 유효 타겟 개수: {len(final_clean_df)} (중복 제외 고유 재료 수)")
draw_frontier(final_clean_df, 'Integrated Frontier (All Groups Unified by Widom)', 'Combined_Widom_Frontier.png')
draw_frontier(final_clean_df[final_clean_df['Filter_Type'] == 'Tier 1 (Ultimate Target)'], 'Tier 1 Only', 'Tier1_Widom_Frontier.png')

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np
import ast
import seaborn as sns

# 🌟 1. 철통 방어: 폰트 및 마이너스 기호 강제 설정
mpl.rcParams['font.family'] = 'sans-serif'
mpl.rcParams['font.sans-serif'] = ['Arial', 'DejaVu Sans', 'Liberation Sans']
mpl.rcParams['axes.unicode_minus'] = False 

# =================================================================
# 2. 파일 로드 및 그룹 지정
# =================================================================
n2_sieving = pd.read_csv("N2_Sieving_Group_63_MOFs.csv")
n2_sieving['Filter_Type'] = 'N2-Sieving (Narrow)'

high_flux = pd.read_csv("High_Flux_Bottleneck_52_MOFs.csv")
high_flux['Filter_Type'] = 'High-Flux (Wide)'

tier1 = pd.read_csv("Tier1_Ultimate_Targets_ColorSynced.csv")
tier1['Filter_Type'] = 'Tier 1 (Ultimate Target)'

# 통합 (우선순위: Tier 1 -> High Flux -> N2 Sieving)
combined_df = pd.concat([tier1, high_flux, n2_sieving], ignore_index=True)

# =================================================================
# 3. [핵심 수정] 모든 데이터에 대해 동일한 GEMC_data(Widom) 파싱 적용
# =================================================================
def extract_metrics(gemc_data):
    try:
        if isinstance(gemc_data, str):
            gemc_data = ast.literal_eval(gemc_data)
        if isinstance(gemc_data, dict) and 'Widom' in gemc_data:
            widom = gemc_data['Widom']
            if isinstance(widom, list) and len(widom) >= 2:
                co2_affinity = widom[0]  # Henry's Constant
                n2_affinity = widom[1]
                selectivity = co2_affinity / n2_affinity if n2_affinity > 0 else 0
                return pd.Series([co2_affinity, selectivity])
    except:
        pass
    return pd.Series([np.nan, np.nan])

def extract_metal_category(metal_data):
    try:
        if isinstance(metal_data, str):
            metal_data = ast.literal_eval(metal_data)
        if isinstance(metal_data, dict):
            metal = metal_data.get('metal_type', 'Unknown')
            oms = metal_data.get('has_OMS', 'Unknown')
            return f"{metal} (OMS: {oms})"
    except:
        pass
    return "Unknown"

print("🔄 모든 데이터를 GEMC_data(Widom) 기준으로 통일 추출 중...")

# ⚠️ 모든 데이터가 공평하게 Widom 원시 데이터로부터 좌표를 계산합니다.
combined_df[['CO2_Affinity', 'Selectivity']] = combined_df['GEMC_data'].apply(extract_metrics)
combined_df['Chemistry'] = combined_df['metal'].apply(extract_metal_category)

# 결측치(파싱 실패 등) 제거 및 찌꺼기 데이터 필터링
final_clean_df = combined_df.dropna(subset=['CO2_Affinity', 'Selectivity'])
final_clean_df = final_clean_df[final_clean_df['Chemistry'] != 'Unknown (OMS: Unknown)'].copy()

# ⚠️ 완벽한 고유 식별자(id) 기준 중복 제거
# Widom 값을 통일했기 때문에, id가 같으면 좌표도 100% 일치합니다.
final_clean_df = final_clean_df.drop_duplicates(subset=['id'], keep='first')

# =================================================================
# 4. 일관성을 위한 색상 및 스타일 고정
# =================================================================
unique_chems = sorted(final_clean_df['Chemistry'].unique())
palette_colors = sns.color_palette('tab20', len(unique_chems))
fixed_palette = dict(zip(unique_chems, palette_colors))
style_order = ['Tier 1 (Ultimate Target)', 'High-Flux (Wide)', 'N2-Sieving (Narrow)']

# =================================================================
# 5. 초고해상도 다차원 산점도 출력 함수 (캔버스 20% 확대 적용)
# =================================================================
def draw_frontier_plot(df, title, filename):
    # 🌟 기존 14 x 9 비율 유지하며 크기 20% 증가 -> 16.8 x 10.8
    plt.figure(figsize=(16.8, 10.8))
    
    ax = sns.scatterplot(
        data=df, 
        x='CO2_Affinity', 
        y='Selectivity', 
        hue='Chemistry', 
        style='Filter_Type',
        hue_order=unique_chems, 
        style_order=style_order,
        s=300,           # 캔버스가 커진 만큼 마커 크기도 살짝(250->300) 상향 조정
        alpha=0.85,      
        edgecolor='black',
        palette=fixed_palette
    )

    # 축 설정 (로그 스케일)
    plt.xscale('log')
    plt.yscale('log')
    
    # x축, y축 범위를 전체 데이터 기준으로 고정 (그래프가 바뀌어도 비율 유지)
    plt.xlim(final_clean_df['CO2_Affinity'].min() * 0.5, final_clean_df['CO2_Affinity'].max() * 2)
    plt.ylim(final_clean_df['Selectivity'].min() * 0.5, final_clean_df['Selectivity'].max() * 2)

    # 디자인 (폰트 크기도 캔버스 확대에 맞춰 약간 상향 조정)
    plt.title(title, fontsize=24, fontweight='bold', pad=18)
    plt.xlabel('CO$_2$ Affinity (Henry\'s Constant) -> Higher Capacity', fontsize=18)
    plt.ylabel('Selectivity (CO$_2$ / N$_2$) -> Higher Purity', fontsize=18)
    plt.grid(True, which="both", ls="--", alpha=0.4)
    plt.tick_params(labelsize=16)

    # 범례 설정
    plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left', borderaxespad=0., fontsize=14, title='Material Properties', title_fontsize=16)

    plt.tight_layout()
    # 논문용 PDF와 확인용 PNG 동시 저장 (600 DPI)
    plt.savefig(filename.replace('.png', '.pdf'), dpi=600, bbox_inches='tight')
    plt.savefig(filename, dpi=600, bbox_inches='tight')
    plt.show()

# =================================================================
# 6. 한 번의 실행으로 4종 그래프 자동 도출
# =================================================================
print(f"✅ 로직 수정 완료! 총 확보된 유효 타겟 개수: {len(final_clean_df)}개\n")

print("📈 1/4. [종합 프론티어] 모든 데이터 출력 중...")
draw_frontier_plot(final_clean_df, 'MOF CCUS Frontier: Integrated Analysis (All Groups)', 'Final_Frontier_Combined.png')

print("📈 2/4. [Tier 1 전용] Ultimate Target 출력 중...")
draw_frontier_plot(final_clean_df[final_clean_df['Filter_Type'] == 'Tier 1 (Ultimate Target)'], 'MOF CCUS Frontier: Tier 1 Only', 'Final_Frontier_Tier1.png')

print("📈 3/4. [High-Flux 전용] Wide Bottleneck 출력 중...")
draw_frontier_plot(final_clean_df[final_clean_df['Filter_Type'] == 'High-Flux (Wide)'], 'MOF CCUS Frontier: High-Flux Only', 'Final_Frontier_HighFlux.png')

print("📈 4/4. [N2-Sieving 전용] Narrow Bottleneck 출력 중...")
draw_frontier_plot(final_clean_df[final_clean_df['Filter_Type'] == 'N2-Sieving (Narrow)'], 'MOF CCUS Frontier: N2-Sieving Only', 'Final_Frontier_N2Sieving.png')

print("🎉 모든 개별 및 종합 그래프 도출 완료! (작업 폴더 내 600DPI 파일 확인)")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np
import ast
import seaborn as sns

# 🌟 1. 철통 방어: 폰트 및 마이너스 기호 강제 설정
mpl.rcParams['font.family'] = 'sans-serif'
mpl.rcParams['font.sans-serif'] = ['Arial', 'DejaVu Sans', 'Liberation Sans']
mpl.rcParams['axes.unicode_minus'] = False 

# =================================================================
# 2. Base 데이터만 로드 (총 115개)
# =================================================================
n2_sieving = pd.read_csv("N2_Sieving_Group_63_MOFs.csv")
n2_sieving['Filter_Type'] = 'N2-Sieving (Narrow)'

high_flux = pd.read_csv("High_Flux_Bottleneck_52_MOFs.csv")
high_flux['Filter_Type'] = 'High-Flux (Wide)'

# Base 그룹만 먼저 완벽하게 병합
base_df = pd.concat([high_flux, n2_sieving], ignore_index=True)

# =================================================================
# 3. 물리량 완벽 추출 (Base 데이터를 기준으로만 계산)
# =================================================================
def extract_metrics(gemc_data):
    try:
        if isinstance(gemc_data, str):
            gemc_data = ast.literal_eval(gemc_data)
        if isinstance(gemc_data, dict) and 'Widom' in gemc_data:
            widom = gemc_data['Widom']
            if isinstance(widom, list) and len(widom) >= 2:
                co2_affinity = widom[0]  
                n2_affinity = widom[1]
                selectivity = co2_affinity / n2_affinity if n2_affinity > 0 else 0
                return pd.Series([co2_affinity, selectivity])
    except:
        pass
    return pd.Series([np.nan, np.nan])

def extract_metal_category(metal_data):
    try:
        if isinstance(metal_data, str):
            metal_data = ast.literal_eval(metal_data)
        if isinstance(metal_data, dict):
            metal = metal_data.get('metal_type', 'Unknown')
            oms = metal_data.get('has_OMS', 'Unknown')
            return f"{metal} (OMS: {oms})"
    except:
        pass
    return "Unknown"

print("🔄 Base 데이터 기준 물리량 계산 중...")
base_df[['CO2_Affinity', 'Selectivity']] = base_df['GEMC_data'].apply(extract_metrics)
base_df['Chemistry'] = base_df['metal'].apply(extract_metal_category)

final_clean_df = base_df.dropna(subset=['CO2_Affinity', 'Selectivity'])
final_clean_df = final_clean_df[final_clean_df['Chemistry'] != 'Unknown (OMS: Unknown)'].copy()

# 중복 제거 (Base 그룹 내 혹시 모를 중복 방어)
final_clean_df = final_clean_df.drop_duplicates(subset=['id'], keep='first')

# =================================================================
# 🌟 4. [궁극의 해결책] VIP 명단 태깅 (Tier 1 승급)
# Tier 1 파일은 오직 '고유 ID'를 가져오는 용도로만 사용합니다.
# =================================================================
try:
    tier1 = pd.read_csv("Tier1_Ultimate_Targets_ColorSynced.csv")
    # Tier 1에 속하는 타겟들의 고유 id 목록 추출
    tier1_vip_ids = tier1['id'].unique()
    
    # Base 데이터프레임에서 VIP 명단에 있는 id를 찾으면, Filter_Type을 Tier 1으로 변경!
    final_clean_df.loc[final_clean_df['id'].isin(tier1_vip_ids), 'Filter_Type'] = 'Tier 1 (Ultimate Target)'
    print("✅ Tier 1 VIP 태깅 완료! (좌표 어긋남 원천 차단)")
except Exception as e:
    print(f"⚠️ Tier 1 파일 로드 에러: {e}")

# =================================================================
# 5. 일관성을 위한 색상 및 스타일 고정
# =================================================================
unique_chems = sorted(final_clean_df['Chemistry'].unique())
palette_colors = sns.color_palette('tab20', len(unique_chems))
fixed_palette = dict(zip(unique_chems, palette_colors))
style_order = ['Tier 1 (Ultimate Target)', 'High-Flux (Wide)', 'N2-Sieving (Narrow)']

# =================================================================
# 6. 초고해상도 산점도 출력 함수 (캔버스 20% 확대: 16.8 x 10.8)
# =================================================================
def draw_frontier_plot(df, title, filename):
    plt.figure(figsize=(16.8, 10.8))
    
    ax = sns.scatterplot(
        data=df, 
        x='CO2_Affinity', 
        y='Selectivity', 
        hue='Chemistry', 
        style='Filter_Type',
        hue_order=unique_chems, 
        style_order=style_order,
        s=300,           
        alpha=0.85,      
        edgecolor='black',
        palette=fixed_palette
    )

    plt.xscale('log')
    plt.yscale('log')
    
    plt.xlim(final_clean_df['CO2_Affinity'].min() * 0.5, final_clean_df['CO2_Affinity'].max() * 2)
    plt.ylim(final_clean_df['Selectivity'].min() * 0.5, final_clean_df['Selectivity'].max() * 2)

    plt.title(title, fontsize=24, fontweight='bold', pad=18)
    plt.xlabel('CO$_2$ Affinity (Henry\'s Constant) -> Higher Capacity', fontsize=18)
    plt.ylabel('Selectivity (CO$_2$ / N$_2$) -> Higher Purity', fontsize=18)
    plt.grid(True, which="both", ls="--", alpha=0.4)
    plt.tick_params(labelsize=16)

    plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left', borderaxespad=0., fontsize=14, title='Material Properties', title_fontsize=16)

    plt.tight_layout()
    plt.savefig(filename.replace('.png', '.pdf'), dpi=600, bbox_inches='tight')
    plt.savefig(filename, dpi=600, bbox_inches='tight')
    plt.show()

# =================================================================
# 7. 실행부
# =================================================================
print(f"📊 최종 점 개수: {len(final_clean_df)}개 (115개 오차 없음)")

print("📈 1/4. 종합 프론티어 출력 중...")
draw_frontier_plot(final_clean_df, 'MOF CCUS Frontier: Integrated Analysis (All Groups)', 'Final_Frontier_Combined.png')

print("📈 2/4. Tier 1 전용 출력 중...")
draw_frontier_plot(final_clean_df[final_clean_df['Filter_Type'] == 'Tier 1 (Ultimate Target)'], 'MOF CCUS Frontier: Tier 1 Only', 'Final_Frontier_Tier1.png')

print("📈 3/4. High-Flux 전용 출력 중...")
draw_frontier_plot(final_clean_df[final_clean_df['Filter_Type'] == 'High-Flux (Wide)'], 'MOF CCUS Frontier: High-Flux Only', 'Final_Frontier_HighFlux.png')

print("📈 4/4. N2-Sieving 전용 출력 중...")
draw_frontier_plot(final_clean_df[final_clean_df['Filter_Type'] == 'N2-Sieving (Narrow)'], 'MOF CCUS Frontier: N2-Sieving Only', 'Final_Frontier_N2Sieving.png')

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np
import ast
import seaborn as sns

# 1. 폰트 및 마이너스 기호 방어
mpl.rcParams['font.family'] = 'sans-serif'
mpl.rcParams['axes.unicode_minus'] = False 

# =================================================================
# 2. 파일 로드 및 그룹 지정
# =================================================================
n2_sieving = pd.read_csv("N2_Sieving_Group_63_MOFs.csv")
n2_sieving['Filter_Type'] = 'N2-Sieving (Narrow)'

high_flux = pd.read_csv("High_Flux_Bottleneck_52_MOFs.csv")
high_flux['Filter_Type'] = 'High-Flux (Wide)'

tier1 = pd.read_csv("Tier1_Ultimate_Targets_ColorSynced.csv")
tier1['Filter_Type'] = 'Tier 1 (Ultimate Target)'

# =================================================================
# 3. 절대 지문(Unique Key) 및 물리량 추출 함수
# =================================================================
def parse_material_info(row):
    # 1. Short ID 추출 (앞 6자리)
    try:
        mofid = ast.literal_eval(row['id']).get('mofid-v1', '')
        short_id = mofid.split(';')[1][:6].upper() if ';' in mofid else "XXXXXX"
    except: short_id = "XXXXXX"

    # 2. 금속 및 OMS 추출
    try:
        metal_dict = ast.literal_eval(row['metal']) if isinstance(row['metal'], str) else row['metal']
        metal = metal_dict.get('metal_type', 'Unknown')
        oms = metal_dict.get('has_OMS', 'Unknown')
    except:
        metal, oms = "Unknown", "Unknown"
        
    # 3. Chemistry 문자열 및 절대 지문(Unique Key) 생성
    chemistry = f"{metal} (OMS: {oms})"
    unique_key = f"{short_id}_{metal}_{oms}"
    
    # 4. 물리량 추출 (엑셀에서 지우셨을 경우를 대비해 GEMC_data 원본에서 무조건 파싱)
    try:
        gemc_dict = ast.literal_eval(row['GEMC_data']) if isinstance(row['GEMC_data'], str) else row['GEMC_data']
        widom = gemc_dict.get('Widom', [np.nan, np.nan])
        co2_aff = widom[0]
        sel = widom[0] / widom[1] if len(widom)>1 and widom[1]>0 else 0
    except:
        co2_aff, sel = np.nan, np.nan

    return pd.Series([chemistry, unique_key, co2_aff, sel])

print("🔄 절대 지문 생성 및 물리량 추출 중...")

# 각 데이터프레임에 함수 적용
for df in [tier1, high_flux, n2_sieving]:
    df[['Chemistry', 'Unique_Key', 'CO2_Affinity', 'Selectivity']] = df.apply(parse_material_info, axis=1)
    df.dropna(subset=['CO2_Affinity', 'Selectivity'], inplace=True)
    df.drop(df[df['Chemistry'] == 'Unknown (OMS: Unknown)'].index, inplace=True)

# =================================================================
# 🌟 4. [핵심 로직] Tier 1 데이터 도려내기 (미스매치 원천 차단)
# =================================================================
# Tier 1의 절대 지문 목록 확보
tier1_keys = tier1['Unique_Key'].unique()

# High Flux와 N2 그룹에서, Tier 1 지문과 똑같은 녀석들을 아예 삭제해버립니다.
high_flux_filtered = high_flux[~high_flux['Unique_Key'].isin(tier1_keys)]
n2_sieving_filtered = n2_sieving[~n2_sieving['Unique_Key'].isin(tier1_keys)]

# 깨끗하게 분리된 3개의 그룹을 하나로 합칩니다.
final_clean_df = pd.concat([tier1, high_flux_filtered, n2_sieving_filtered], ignore_index=True)
final_clean_df = final_clean_df.drop_duplicates(subset=['Unique_Key'], keep='first')

# =================================================================
# 5. 스타일 및 마커 완벽 고정
# =================================================================
unique_chems = sorted(final_clean_df['Chemistry'].unique())
fixed_palette = dict(zip(unique_chems, sns.color_palette('tab20', len(unique_chems))))

# 요청하신 마커 강제 할당 (Tier1: 네모, N2: 동그라미, High Flux: X)
style_order = ['Tier 1 (Ultimate Target)', 'N2-Sieving (Narrow)', 'High-Flux (Wide)']
custom_markers = {
    'Tier 1 (Ultimate Target)': 's',  # Square (네모)
    'N2-Sieving (Narrow)': 'o',       # Circle (동그라미)
    'High-Flux (Wide)': 'X'           # Cross (X 기호)
}

# =================================================================
# 6. 초고해상도 산점도 (비율 20% 스케일업: 16.8 x 10.8)
# =================================================================
def draw_strict_frontier(df, title, filename):
    plt.figure(figsize=(16.8, 10.8))
    
    ax = sns.scatterplot(
        data=df, 
        x='CO2_Affinity', 
        y='Selectivity', 
        hue='Chemistry', 
        style='Filter_Type',
        hue_order=unique_chems, 
        style_order=style_order,
        markers=custom_markers,  # 커스텀 마커 강제 적용
        s=300,           
        alpha=0.85,      
        edgecolor='black',
        palette=fixed_palette
    )

    plt.xscale('log')
    plt.yscale('log')
    
    # 축 범위 고정
    plt.xlim(final_clean_df['CO2_Affinity'].min() * 0.5, final_clean_df['CO2_Affinity'].max() * 2)
    plt.ylim(final_clean_df['Selectivity'].min() * 0.5, final_clean_df['Selectivity'].max() * 2)

    # 폰트 사이즈 상향
    plt.title(title, fontsize=24, fontweight='bold', pad=18)
    plt.xlabel('CO$_2$ Affinity (Henry\'s Constant) -> Higher Capacity', fontsize=18)
    plt.ylabel('Selectivity (CO$_2$ / N$_2$) -> Higher Purity', fontsize=18)
    plt.grid(True, which="both", ls="--", alpha=0.4)
    plt.tick_params(labelsize=16)

    # 범례 정리
    plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left', borderaxespad=0., fontsize=14, title='Material Properties', title_fontsize=16)

    plt.tight_layout()
    plt.savefig(filename.replace('.png', '.pdf'), dpi=600, bbox_inches='tight')
    plt.savefig(filename, dpi=600, bbox_inches='tight')
    plt.show()

# =================================================================
# 7. 자동 도출
# =================================================================
print(f"📊 최종 점 개수: {len(final_clean_df)}개 (어긋남 절대 불가)")

print("📈 1/4. 종합 프론티어 출력 중...")
draw_strict_frontier(final_clean_df, 'MOF CCUS Frontier: Integrated Analysis (All Groups)', 'Final_Frontier_Combined_Fixed.png')

print("📈 2/4. Tier 1 전용 출력 중...")
draw_strict_frontier(final_clean_df[final_clean_df['Filter_Type'] == 'Tier 1 (Ultimate Target)'], 'MOF CCUS Frontier: Tier 1 Only', 'Final_Frontier_Tier1_Fixed.png')

print("📈 3/4. High-Flux 전용 출력 중...")
draw_strict_frontier(final_clean_df[final_clean_df['Filter_Type'] == 'High-Flux (Wide)'], 'MOF CCUS Frontier: High-Flux Only', 'Final_Frontier_HighFlux_Fixed.png')

print("📈 4/4. N2-Sieving 전용 출력 중...")
draw_strict_frontier(final_clean_df[final_clean_df['Filter_Type'] == 'N2-Sieving (Narrow)'], 'MOF CCUS Frontier: N2-Sieving Only', 'Final_Frontier_N2Sieving_Fixed.png')

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np
import ast
import seaborn as sns

# 1. 폰트 및 마이너스 기호 방어
mpl.rcParams['font.family'] = 'sans-serif'
mpl.rcParams['axes.unicode_minus'] = False 

# =================================================================
# 2. 파일 로드 (순서: Tier 1 -> High Flux -> N2)
# =================================================================
tier1 = pd.read_csv("Tier1_Ultimate_Targets_ColorSynced.csv")
tier1['Filter_Type'] = 'Tier 1 (Ultimate Target)'

high_flux = pd.read_csv("High_Flux_Bottleneck_52_MOFs.csv")
high_flux['Filter_Type'] = 'High-Flux (Wide)'

n2_sieving = pd.read_csv("N2_Sieving_Group_63_MOFs.csv")
n2_sieving['Filter_Type'] = 'N2-Sieving (Narrow)'

# 총 128개 (13 + 52 + 63)
combined_df = pd.concat([tier1, high_flux, n2_sieving], ignore_index=True)

# =================================================================
# 3. 완벽한 파싱 함수들 (엑셀 변형 원천 차단)
# =================================================================
def get_true_mofid(id_data):
    # 엑셀의 따옴표, 띄어쓰기 변형을 이겨내고 순수 mofid-v1 문자열만 추출
    try:
        d = ast.literal_eval(id_data) if isinstance(id_data, str) else id_data
        return d.get('mofid-v1', str(id_data))
    except:
        return str(id_data)

def extract_metrics(gemc_data):
    # 무조건 GEMC_data 원본의 Widom 값을 파싱하여 오차 원천 차단
    try:
        d = ast.literal_eval(gemc_data) if isinstance(gemc_data, str) else gemc_data
        widom = d.get('Widom', [])
        if len(widom) >= 2:
            return pd.Series([widom[0], widom[0]/widom[1] if widom[1]>0 else 0])
    except: pass
    return pd.Series([np.nan, np.nan])

def extract_metal_category(metal_data):
    try:
        d = ast.literal_eval(metal_data) if isinstance(metal_data, str) else metal_data
        return f"{d.get('metal_type', 'Unknown')} (OMS: {d.get('has_OMS', 'Unknown')})"
    except: pass
    return "Unknown"

print("🔄 데이터 정밀 파싱 및 115개 복원 중...")

# 파싱 일괄 적용
combined_df['True_MOFid'] = combined_df['id'].apply(get_true_mofid)
combined_df[['CO2_Affinity', 'Selectivity']] = combined_df['GEMC_data'].apply(extract_metrics)
combined_df['Chemistry'] = combined_df['metal'].apply(extract_metal_category)

# 결측치 및 알 수 없는 데이터 제거
final_clean_df = combined_df.dropna(subset=['CO2_Affinity', 'Selectivity'])
final_clean_df = final_clean_df[final_clean_df['Chemistry'] != 'Unknown (OMS: Unknown)']

# 🌟 핵심 로직: 진짜 분자식(mofid-v1)을 기준으로 중복 제거
# Tier 1이 concat 배열의 맨 앞에 있으므로, 중복 발생 시 Tier 1(네모)이 무조건 살아남습니다!
final_clean_df = final_clean_df.drop_duplicates(subset=['True_MOFid'], keep='first')

# =================================================================
# 4. 스타일 및 마커 완벽 고정
# =================================================================
unique_chems = sorted(final_clean_df['Chemistry'].unique())
fixed_palette = dict(zip(unique_chems, sns.color_palette('tab20', len(unique_chems))))

style_order = ['Tier 1 (Ultimate Target)', 'N2-Sieving (Narrow)', 'High-Flux (Wide)']

# 🌟 [요구사항 반영] 명확한 마커 구분 (네모, 동그라미, X)
custom_markers = {
    'Tier 1 (Ultimate Target)': 's',  # Square (네모)
    'N2-Sieving (Narrow)': 'o',       # Circle (동그라미)
    'High-Flux (Wide)': 'X'           # Cross (X 기호)
}

# =================================================================
# 5. 초고해상도 산점도 (비율 20% 스케일업: 16.8 x 10.8)
# =================================================================
def draw_ultimate_frontier(df, title, filename):
    plt.figure(figsize=(16.8, 10.8))
    
    sns.scatterplot(
        data=df, 
        x='CO2_Affinity', 
        y='Selectivity', 
        hue='Chemistry', 
        style='Filter_Type',
        hue_order=unique_chems, 
        style_order=style_order,
        markers=custom_markers, # 커스텀 마커 강제 적용
        s=350,  # 캔버스 크기에 맞춰 시인성 극대화
        alpha=0.85,      
        edgecolor='black',
        palette=fixed_palette
    )

    plt.xscale('log')
    plt.yscale('log')
    
    # 축 범위 고정 (그래프 비율 유지)
    plt.xlim(final_clean_df['CO2_Affinity'].min() * 0.5, final_clean_df['CO2_Affinity'].max() * 2)
    plt.ylim(final_clean_df['Selectivity'].min() * 0.5, final_clean_df['Selectivity'].max() * 2)

    plt.title(title, fontsize=24, fontweight='bold', pad=18)
    plt.xlabel('CO$_2$ Affinity (Henry\'s Constant) -> Higher Capacity', fontsize=18)
    plt.ylabel('Selectivity (CO$_2$ / N$_2$) -> Higher Purity', fontsize=18)
    plt.grid(True, which="both", ls="--", alpha=0.4)
    plt.tick_params(labelsize=16)

    plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left', borderaxespad=0., fontsize=14, title='Material Properties', title_fontsize=16)

    plt.tight_layout()
    plt.savefig(filename.replace('.png', '.pdf'), dpi=600, bbox_inches='tight')
    plt.savefig(filename, dpi=600, bbox_inches='tight')
    plt.show()

# =================================================================
# 6. 실행 및 출력
# =================================================================
print(f"📊 최종 점 개수: {len(final_clean_df)}개 (115개 복원 및 미스매치 차단 완료!)")

print("📈 1/4. 종합 프론티어 출력 중...")
draw_ultimate_frontier(final_clean_df, 'MOF CCUS Frontier: Integrated Analysis', 'Final_Frontier_Combined_115.png')

print("📈 2/4. Tier 1 전용 출력 중...")
draw_ultimate_frontier(final_clean_df[final_clean_df['Filter_Type'] == 'Tier 1 (Ultimate Target)'], 'MOF CCUS Frontier: Tier 1 Only', 'Final_Frontier_Tier1_115.png')

print("📈 3/4. High-Flux 전용 출력 중...")
draw_ultimate_frontier(final_clean_df[final_clean_df['Filter_Type'] == 'High-Flux (Wide)'], 'MOF CCUS Frontier: High-Flux Only', 'Final_Frontier_HighFlux_115.png')

print("📈 4/4. N2-Sieving 전용 출력 중...")
draw_ultimate_frontier(final_clean_df[final_clean_df['Filter_Type'] == 'N2-Sieving (Narrow)'], 'MOF CCUS Frontier: N2-Sieving Only', 'Final_Frontier_N2Sieving_115.png')

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np
import ast
import seaborn as sns

# 1. 폰트 및 마이너스 기호 방어
mpl.rcParams['font.family'] = 'sans-serif'
mpl.rcParams['axes.unicode_minus'] = False 

# =================================================================
# 2. [Base 데이터만 로드] - N2와 High-Flux만 합쳐서 115개의 기준점 생성
# =================================================================
n2_sieving = pd.read_csv("N2_Sieving_Group_63_MOFs.csv")
n2_sieving['Base_Group'] = 'N2-Sieving (Narrow)'

high_flux = pd.read_csv("High_Flux_Bottleneck_52_MOFs.csv")
high_flux['Base_Group'] = 'High-Flux (Wide)'

base_df = pd.concat([high_flux, n2_sieving], ignore_index=True)

# =================================================================
# 3. 파싱 함수 (오류 없는 고유 분자식 및 좌표 추출)
# =================================================================
def get_mofid(id_data):
    try:
        d = ast.literal_eval(id_data) if isinstance(id_data, str) else id_data
        return d.get('mofid-v1', str(id_data))
    except: return str(id_data)

def get_metrics(gemc_data):
    try:
        d = ast.literal_eval(gemc_data) if isinstance(gemc_data, str) else gemc_data
        widom = d.get('Widom', [])
        if len(widom) >= 2:
            return pd.Series([widom[0], widom[0]/widom[1] if widom[1]>0 else 0])
    except: pass
    return pd.Series([np.nan, np.nan])

def get_chem(metal_data):
    try:
        d = ast.literal_eval(metal_data) if isinstance(metal_data, str) else metal_data
        return f"{d.get('metal_type', 'Unknown')} (OMS: {d.get('has_OMS', 'Unknown')})"
    except: pass
    return "Unknown"

print("🔄 Base 데이터 기준 물리량 100% 동기화 중...")

# Base 데이터 좌표 및 정보 세팅
base_df['MOF_ID'] = base_df['id'].apply(get_mofid)
base_df[['CO2_Affinity', 'Selectivity']] = base_df['GEMC_data'].apply(get_metrics)
base_df['Chemistry'] = base_df['metal'].apply(get_chem)

# 결측치 제거 및 Base 내 동일 뼈대 중복 제거
base_df = base_df.dropna(subset=['CO2_Affinity', 'Selectivity'])
base_df = base_df[base_df['Chemistry'] != 'Unknown (OMS: Unknown)']
base_df = base_df.drop_duplicates(subset=['MOF_ID'], keep='first')

# =================================================================
# 🌟 4. [완벽한 분기 제어] Tier 1은 물리량 무시! 오직 '명단'으로만 사용
# =================================================================
try:
    tier1 = pd.read_csv("Tier1_Ultimate_Targets_ColorSynced.csv")
    tier1['MOF_ID'] = tier1['id'].apply(get_mofid)
    
    # VIP 명단 추출 (분자식 기준)
    vip_list = tier1['MOF_ID'].unique()
    
    # Base 그룹을 복사하여 Filter_Type 컬럼 생성
    base_df['Filter_Type'] = base_df['Base_Group']
    
    # VIP 명단에 있는 MOF는 Filter_Type을 'Tier 1'로 덮어씌움 (도형만 바뀜)
    base_df.loc[base_df['MOF_ID'].isin(vip_list), 'Filter_Type'] = 'Tier 1 (Ultimate Target)'
    
    print("✅ 분기 제어 성공! Tier 1 좌표를 병합하지 않고 마커 속성만 변경했습니다.")
except Exception as e:
    print(f"⚠️ Tier 1 파일 로드 오류: {e}")

# =================================================================
# 5. 스타일 및 마커 강제 고정
# =================================================================
unique_chems = sorted(base_df['Chemistry'].unique())
fixed_palette = dict(zip(unique_chems, sns.color_palette('tab20', len(unique_chems))))

style_order = ['Tier 1 (Ultimate Target)', 'N2-Sieving (Narrow)', 'High-Flux (Wide)']
custom_markers = {
    'Tier 1 (Ultimate Target)': 's',  # 네모
    'N2-Sieving (Narrow)': 'o',       # 동그라미
    'High-Flux (Wide)': 'X'           # X 표시
}

# =================================================================
# 6. 초고해상도 산점도 (비율 20% 스케일업: 16.8 x 10.8)
# =================================================================
def draw_strict_frontier(df, title, filename):
    plt.figure(figsize=(16.8, 10.8))
    
    sns.scatterplot(
        data=df, 
        x='CO2_Affinity', 
        y='Selectivity', 
        hue='Chemistry', 
        style='Filter_Type',
        hue_order=unique_chems, 
        style_order=style_order,
        markers=custom_markers,  # 지시하신 마커 완벽 적용
        s=350,           
        alpha=0.85,      
        edgecolor='black',
        palette=fixed_palette
    )

    plt.xscale('log')
    plt.yscale('log')
    
    plt.xlim(base_df['CO2_Affinity'].min() * 0.5, base_df['CO2_Affinity'].max() * 2)
    plt.ylim(base_df['Selectivity'].min() * 0.5, base_df['Selectivity'].max() * 2)

    plt.title(title, fontsize=24, fontweight='bold', pad=18)
    plt.xlabel('CO$_2$ Affinity (Henry\'s Constant) -> Higher Capacity', fontsize=18)
    plt.ylabel('Selectivity (CO$_2$ / N$_2$) -> Higher Purity', fontsize=18)
    plt.grid(True, which="both", ls="--", alpha=0.4)
    plt.tick_params(labelsize=16)

    plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left', borderaxespad=0., fontsize=14, title='Material Properties', title_fontsize=16)

    plt.tight_layout()
    plt.savefig(filename.replace('.png', '.pdf'), dpi=600, bbox_inches='tight')
    plt.savefig(filename, dpi=600, bbox_inches='tight')
    plt.show()

# =================================================================
# 7. 출력 실행
# =================================================================
print(f"📊 최종 확보된 유효 타겟 개수: {len(base_df)}개 (115개 유지)")

print("📈 종합 프론티어 출력 중...")
draw_strict_frontier(base_df, 'MOF CCUS Frontier: Integrated Analysis', 'Final_Frontier_Combined_Bulletproof.png')

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np
import re
import seaborn as sns

# 1. 폰트 및 마이너스 기호 방어
mpl.rcParams['font.family'] = 'sans-serif'
mpl.rcParams['axes.unicode_minus'] = False 

# =================================================================
# 2. Base 데이터 로드 (총합 115개 유지)
# =================================================================
n2_sieving = pd.read_csv("N2_Sieving_Group_63_MOFs.csv")
n2_sieving['Base_Group'] = 'N2-Sieving (Narrow)'

high_flux = pd.read_csv("High_Flux_Bottleneck_52_MOFs.csv")
high_flux['Base_Group'] = 'High-Flux (Wide)'

base_df = pd.concat([high_flux, n2_sieving], ignore_index=True)

# =================================================================
# 3. [핵심] Regex 추출 (엑셀 변형 원천 차단, 에러율 0%)
# =================================================================
def get_mofid_regex(id_data):
    # 'mofid-v1': '값' 형태에서 값만 정확히 파출
    match = re.search(r"'mofid-v1'\s*:\s*'([^']+)'", str(id_data))
    if match: return match.group(1)
    # 실패 시 공백/따옴표를 모두 제거한 절대 지문 생성
    return re.sub(r'[\s\'"]', '', str(id_data))

def get_metrics_regex(gemc_data):
    # 'Widom': [값1, 값2] 형태에서 숫자만 정확히 추출 (엑셀 따옴표 무시)
    match = re.search(r"'Widom'\s*:\s*\[\s*([-+]?\d*\.?\d+(?:[eE][-+]?\d+)?)\s*,\s*([-+]?\d*\.?\d+(?:[eE][-+]?\d+)?)\s*\]", str(gemc_data))
    if match:
        co2 = float(match.group(1))
        n2 = float(match.group(2))
        return pd.Series([co2, co2/n2 if n2 > 0 else 0])
    return pd.Series([np.nan, np.nan])

def get_chem_regex(metal_data):
    # metal_type과 has_OMS의 값을 각각 추출
    s = str(metal_data)
    m_match = re.search(r"'metal_type'\s*:\s*'([^']+)'", s)
    o_match = re.search(r"'has_OMS'\s*:\s*'([^']+)'", s)
    metal = m_match.group(1) if m_match else 'Unknown'
    oms = o_match.group(1) if o_match else 'Unknown'
    return f"{metal} (OMS: {oms})"

print("🔄 정규표현식(Regex)을 통한 데이터 100% 구출 중...")

base_df['MOF_ID'] = base_df['id'].apply(get_mofid_regex)
base_df[['CO2_Affinity', 'Selectivity']] = base_df['GEMC_data'].apply(get_metrics_regex)
base_df['Chemistry'] = base_df['metal'].apply(get_chem_regex)

# 결측치 제거 (이제 파싱 에러가 없으므로 실제 데이터가 없는 경우만 삭제됨)
base_df = base_df.dropna(subset=['CO2_Affinity', 'Selectivity'])
base_df = base_df[base_df['Chemistry'] != 'Unknown (OMS: Unknown)']
base_df = base_df.drop_duplicates(subset=['MOF_ID'], keep='first')

# =================================================================
# 🌟 4. [분기 제어] Tier 1은 오직 "마커 변경" 용도로만 사용
# =================================================================
try:
    tier1 = pd.read_csv("Tier1_Ultimate_Targets_ColorSynced.csv")
    tier1['MOF_ID'] = tier1['id'].apply(get_mofid_regex)
    vip_list = tier1['MOF_ID'].unique()
    
    base_df['Filter_Type'] = base_df['Base_Group']
    # VIP 명단과 일치하면 Filter_Type만 Tier 1으로 변경 (좌표 건드림 X)
    base_df.loc[base_df['MOF_ID'].isin(vip_list), 'Filter_Type'] = 'Tier 1 (Ultimate Target)'
    print(f"✅ Tier 1 VIP 태깅 완료! (총 {len(vip_list)}개 중 매칭 성공)")
except Exception as e:
    print(f"⚠️ Tier 1 로드 오류: {e}")

# =================================================================
# 5. 스타일 및 마커 강제 고정
# =================================================================
unique_chems = sorted(base_df['Chemistry'].unique())
fixed_palette = dict(zip(unique_chems, sns.color_palette('tab20', len(unique_chems))))

style_order = ['Tier 1 (Ultimate Target)', 'N2-Sieving (Narrow)', 'High-Flux (Wide)']
custom_markers = {
    'Tier 1 (Ultimate Target)': 's',  # 네모
    'N2-Sieving (Narrow)': 'o',       # 동그라미
    'High-Flux (Wide)': 'X'           # X 표시
}

# =================================================================
# 6. 초고해상도 산점도 (비율 유지 20% 스케일업: 16.8 x 10.8)
# =================================================================
def draw_strict_frontier(df, title, filename):
    plt.figure(figsize=(16.8, 10.8))
    
    sns.scatterplot(
        data=df, x='CO2_Affinity', y='Selectivity', hue='Chemistry', style='Filter_Type',
        hue_order=unique_chems, style_order=style_order, markers=custom_markers,
        s=350, alpha=0.85, edgecolor='black', palette=fixed_palette
    )

    plt.xscale('log')
    plt.yscale('log')
    
    # 축 범위 고정
    plt.xlim(base_df['CO2_Affinity'].min() * 0.5, base_df['CO2_Affinity'].max() * 2)
    plt.ylim(base_df['Selectivity'].min() * 0.5, base_df['Selectivity'].max() * 2)

    plt.title(title, fontsize=24, fontweight='bold', pad=18)
    plt.xlabel('CO$_2$ Affinity (Henry\'s Constant) -> Higher Capacity', fontsize=18)
    plt.ylabel('Selectivity (CO$_2$ / N$_2$) -> Higher Purity', fontsize=18)
    plt.grid(True, which="both", ls="--", alpha=0.4)
    plt.tick_params(labelsize=16)
    plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left', borderaxespad=0., fontsize=14, title='Material Properties', title_fontsize=16)

    plt.tight_layout()
    plt.savefig(filename.replace('.png', '.pdf'), dpi=600, bbox_inches='tight')
    plt.savefig(filename, dpi=600, bbox_inches='tight')
    plt.show()

# =================================================================
# 7. 출력 실행
# =================================================================
print(f"📊 최종 데이터 개수 검증: {len(base_df)}개 (115개 복원 완료!)")

draw_strict_frontier(base_df, 'MOF CCUS Frontier: Integrated Analysis', 'Final_Frontier_Regex_Fixed.png')

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np
import ast
import seaborn as sns

# 1. 폰트 및 마이너스 기호 방어
mpl.rcParams['font.family'] = 'sans-serif'
mpl.rcParams['axes.unicode_minus'] = False 

# =================================================================
# 2. [Base 데이터만 로드] - N2와 High-Flux만 합쳐서 115개의 기준점 생성
# =================================================================
n2_sieving = pd.read_csv("N2_Sieving_Group_63_MOFs.csv")
n2_sieving['Base_Group'] = 'N2-Sieving (Narrow)'

high_flux = pd.read_csv("High_Flux_Bottleneck_52_MOFs.csv")
high_flux['Base_Group'] = 'High-Flux (Wide)'

base_df = pd.concat([high_flux, n2_sieving], ignore_index=True)

# =================================================================
# 3. 파싱 함수 (오류 없는 고유 분자식 및 좌표 추출)
# =================================================================
def get_mofid(id_data):
    try:
        d = ast.literal_eval(id_data) if isinstance(id_data, str) else id_data
        return d.get('mofid-v1', str(id_data))
    except: return str(id_data)

def get_metrics(gemc_data):
    try:
        d = ast.literal_eval(gemc_data) if isinstance(gemc_data, str) else gemc_data
        widom = d.get('Widom', [])
        if len(widom) >= 2:
            return pd.Series([widom[0], widom[0]/widom[1] if widom[1]>0 else 0])
    except: pass
    return pd.Series([np.nan, np.nan])

def get_chem(metal_data):
    try:
        d = ast.literal_eval(metal_data) if isinstance(metal_data, str) else metal_data
        return f"{d.get('metal_type', 'Unknown')} (OMS: {d.get('has_OMS', 'Unknown')})"
    except: pass
    return "Unknown"

print("🔄 Base 데이터 기준 물리량 100% 동기화 중...")

base_df['MOF_ID'] = base_df['id'].apply(get_mofid)
base_df[['CO2_Affinity', 'Selectivity']] = base_df['GEMC_data'].apply(get_metrics)
base_df['Chemistry'] = base_df['metal'].apply(get_chem)

# 결측치 제거 및 Base 내 동일 뼈대 중복 제거 (정확히 115개 유지)
base_df = base_df.dropna(subset=['CO2_Affinity', 'Selectivity'])
base_df = base_df[base_df['Chemistry'] != 'Unknown (OMS: Unknown)']
base_df = base_df.drop_duplicates(subset=['MOF_ID'], keep='first')

# =================================================================
# 🌟 4. [완벽한 분기 제어] Tier 1은 물리량 무시! 오직 '명단'으로만 사용
# =================================================================
try:
    tier1 = pd.read_csv("Tier1_Ultimate_Targets_ColorSynced.csv")
    tier1['MOF_ID'] = tier1['id'].apply(get_mofid)
    
    vip_list = tier1['MOF_ID'].unique()
    
    base_df['Filter_Type'] = base_df['Base_Group']
    
    # VIP 명단에 있는 MOF는 마커 속성만 'Tier 1'로 덮어씌움 (좌표 물리량 간섭 0%)
    base_df.loc[base_df['MOF_ID'].isin(vip_list), 'Filter_Type'] = 'Tier 1 (Ultimate Target)'
    
    print(f"✅ 분기 제어 성공! {len(vip_list)}개의 타겟 마커를 네모로 변경했습니다.")
except Exception as e:
    print(f"⚠️ Tier 1 파일 로드 오류: {e}")

# =================================================================
# 5. 스타일 및 마커 강제 고정
# =================================================================
unique_chems = sorted(base_df['Chemistry'].unique())
fixed_palette = dict(zip(unique_chems, sns.color_palette('tab20', len(unique_chems))))

style_order = ['Tier 1 (Ultimate Target)', 'N2-Sieving (Narrow)', 'High-Flux (Wide)']
custom_markers = {
    'Tier 1 (Ultimate Target)': 's',  # 네모
    'N2-Sieving (Narrow)': 'o',       # 동그라미
    'High-Flux (Wide)': 'X'           # X 표시
}

# =================================================================
# 6. 초고해상도 산점도 함수 (비율 20% 스케일업: 16.8 x 10.8)
# =================================================================
def draw_strict_frontier(df, title, filename):
    plt.figure(figsize=(16.8, 10.8))
    
    sns.scatterplot(
        data=df, 
        x='CO2_Affinity', 
        y='Selectivity', 
        hue='Chemistry', 
        style='Filter_Type',
        hue_order=unique_chems, 
        style_order=style_order,
        markers=custom_markers,  
        s=350,           
        alpha=0.85,      
        edgecolor='black',
        palette=fixed_palette
    )

    plt.xscale('log')
    plt.yscale('log')
    
    # 축 고정
    plt.xlim(base_df['CO2_Affinity'].min() * 0.5, base_df['CO2_Affinity'].max() * 2)
    plt.ylim(base_df['Selectivity'].min() * 0.5, base_df['Selectivity'].max() * 2)

    plt.title(title, fontsize=24, fontweight='bold', pad=18)
    plt.xlabel('CO$_2$ Affinity (Henry\'s Constant) -> Higher Capacity', fontsize=18)
    plt.ylabel('Selectivity (CO$_2$ / N$_2$) -> Higher Purity', fontsize=18)
    plt.grid(True, which="both", ls="--", alpha=0.4)
    plt.tick_params(labelsize=16)

    plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left', borderaxespad=0., fontsize=14, title='Material Properties', title_fontsize=16)

    plt.tight_layout()
    plt.savefig(filename.replace('.png', '.pdf'), dpi=600, bbox_inches='tight')
    plt.savefig(filename, dpi=600, bbox_inches='tight')
    plt.show()

# =================================================================
# 7. 출력 실행
# =================================================================
print(f"📊 최종 확보된 유효 타겟 개수: {len(base_df)}개 (115개 유지 확인 완료!)")

print("📈 1/4. 종합 프론티어 출력 중...")
draw_strict_frontier(base_df, 'MOF CCUS Frontier: Integrated Analysis', 'Final_Verified_Combined.png')

print("📈 2/4. Tier 1 전용 출력 중...")
draw_strict_frontier(base_df[base_df['Filter_Type'] == 'Tier 1 (Ultimate Target)'], 'MOF CCUS Frontier: Tier 1 Only', 'Final_Verified_Tier1.png')

print("📈 3/4. High-Flux 전용 출력 중...")
draw_strict_frontier(base_df[base_df['Filter_Type'] == 'High-Flux (Wide)'], 'MOF CCUS Frontier: High-Flux Only', 'Final_Verified_HighFlux.png')

print("📈 4/4. N2-Sieving 전용 출력 중...")
draw_strict_frontier(base_df[base_df['Filter_Type'] == 'N2-Sieving (Narrow)'], 'MOF CCUS Frontier: N2-Sieving Only', 'Final_Verified_N2Sieving.png')

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np
import ast
import seaborn as sns

# 1. 폰트 및 마이너스 기호 방어
mpl.rcParams['font.family'] = 'sans-serif'
mpl.rcParams['axes.unicode_minus'] = False 

# =================================================================
# 2. [Base 데이터 로드]
# =================================================================
n2_sieving = pd.read_csv("N2_Sieving_Group_63_MOFs.csv")
n2_sieving['Base_Group'] = 'N2-Sieving (Narrow)'

high_flux = pd.read_csv("High_Flux_Bottleneck_52_MOFs.csv")
high_flux['Base_Group'] = 'High-Flux (Wide)'

base_df = pd.concat([high_flux, n2_sieving], ignore_index=True)

# =================================================================
# 3. 파싱 함수 (물리량 및 Zeo++ 필터용)
# =================================================================
def get_mofid(id_data):
    try:
        d = ast.literal_eval(id_data) if isinstance(id_data, str) else id_data
        return d.get('mofid-v1', str(id_data))
    except: return str(id_data)

def get_short_id(mofid_str):
    try:
        if ';' in mofid_str:
            return mofid_str.split(';')[1][:6].upper()
        return 'XXXXXX'
    except: return 'XXXXXX'

def get_metrics(gemc_data):
    try:
        d = ast.literal_eval(gemc_data) if isinstance(gemc_data, str) else gemc_data
        widom = d.get('Widom', [])
        if len(widom) >= 2:
            return pd.Series([widom[0], widom[0]/widom[1] if widom[1]>0 else 0])
    except: pass
    return pd.Series([np.nan, np.nan])

def get_chem(metal_data):
    try:
        d = ast.literal_eval(metal_data) if isinstance(metal_data, str) else metal_data
        return f"{d.get('metal_type', 'Unknown')} (OMS: {d.get('has_OMS', 'Unknown')})"
    except: pass
    return "Unknown"

def get_zeopp_filters(zeopp_str):
    try:
        d = ast.literal_eval(zeopp_str) if isinstance(zeopp_str, str) else zeopp_str
        dim = d.get('dimension', 0)
        pld = d.get('PLD', 0)
        vf = d.get('VF', 0)
        return pd.Series([dim, pld, vf])
    except:
        return pd.Series([0, 0, 0])

print("🔄 Base 데이터 기준 물리량 100% 동기화 중...")

base_df['MOF_ID'] = base_df['id'].apply(get_mofid)
base_df['Short_ID'] = base_df['MOF_ID'].apply(get_short_id)
base_df[['CO2_Affinity', 'Selectivity']] = base_df['GEMC_data'].apply(get_metrics)
base_df['Chemistry'] = base_df['metal'].apply(get_chem)
base_df[['Dimension', 'PLD', 'VF']] = base_df['Zeopp'].apply(get_zeopp_filters)

# 기본 결측치 및 오류 뼈대 제거
base_df = base_df.dropna(subset=['CO2_Affinity', 'Selectivity'])
base_df = base_df[base_df['Chemistry'] != 'Unknown (OMS: Unknown)']
base_df = base_df.drop_duplicates(subset=['MOF_ID'], keep='first')

print(f"🔹 1차 정제 후 데이터: {len(base_df)}개")

# =================================================================
# 🌟 4. [신규] 위양성(False Positive) 하드 필터링
# =================================================================
# 1D/2D 밀집상 제거 (dimension == 3) & 최소 확산 직경 보장 (PLD >= 3.4) & 최소 기공 보장 (VF >= 0.2)
base_df = base_df[(base_df['Dimension'] == 3) & (base_df['PLD'] >= 3.4) & (base_df['VF'] >= 0.2)].copy()
print(f"🛡️ 위양성(1D/2D 등) 필터링 후 진성 3D 다공성 타겟: {len(base_df)}개 생존!")

# =================================================================
# 🌟 5. [신규] Robust Scoring (아웃라이어 왜곡 방지)
# =================================================================
def robust_scoring(df, weight_sel=0.6, weight_cap=0.4):
    # 상위 5%의 극단값을 하향 조정(Clipping)하여 정상적인 물질들의 점수가 바닥에 깔리는 현상 방지
    sel_cap_value = df['Selectivity'].quantile(0.95)
    cap_cap_value = df['CO2_Affinity'].quantile(0.95)
    
    df['Sel_Clipped'] = df['Selectivity'].clip(upper=sel_cap_value)
    df['Cap_Clipped'] = df['CO2_Affinity'].clip(upper=cap_cap_value)
    
    sel_min, sel_max = df['Sel_Clipped'].min(), df['Sel_Clipped'].max()
    cap_min, cap_max = df['Cap_Clipped'].min(), df['Cap_Clipped'].max()
    
    # Min-Max 스케일링 (0~100점)
    df['Sel_Norm'] = (df['Sel_Clipped'] - sel_min) / (sel_max - sel_min) * 100 if sel_max > sel_min else 0
    df['Cap_Norm'] = (df['Cap_Clipped'] - cap_min) / (cap_max - cap_min) * 100 if cap_max > cap_min else 0
    
    df['Final_Score'] = (df['Sel_Norm'] * weight_sel) + (df['Cap_Norm'] * weight_cap)
    
    # 랭킹 부여
    df = df.sort_values(by='Final_Score', ascending=False).reset_index(drop=True)
    df['Rank'] = np.arange(1, len(df) + 1)
    return df

base_df = robust_scoring(base_df)
base_df[['Rank', 'Short_ID', 'Dimension', 'PLD', 'Final_Score']].to_csv("Robust_Verified_Rankings.csv", index=False)

# =================================================================
# 6. [완벽한 분기 제어] Tier 1은 물리량 무시! 오직 '명단'으로만 사용
# =================================================================
try:
    tier1 = pd.read_csv("Tier1_Ultimate_Targets_ColorSynced.csv")
    tier1['MOF_ID'] = tier1['id'].apply(get_mofid)
    
    vip_list = tier1['MOF_ID'].unique()
    
    base_df['Filter_Type'] = base_df['Base_Group']
    
    # VIP 명단에 있는 MOF는 마커 속성만 'Tier 1'로 덮어씌움
    base_df.loc[base_df['MOF_ID'].isin(vip_list), 'Filter_Type'] = 'Tier 1 (Ultimate Target)'
    
    print(f"✅ Tier 1 분기 제어 성공!")
except Exception as e:
    print(f"⚠️ Tier 1 파일 로드 오류: {e}")

# =================================================================
# 7. 스타일 및 마커 강제 고정
# =================================================================
unique_chems = sorted(base_df['Chemistry'].unique())
fixed_palette = dict(zip(unique_chems, sns.color_palette('tab20', len(unique_chems))))

style_order = ['Tier 1 (Ultimate Target)', 'N2-Sieving (Narrow)', 'High-Flux (Wide)']
custom_markers = {
    'Tier 1 (Ultimate Target)': 's',  # 네모
    'N2-Sieving (Narrow)': 'o',       # 동그라미
    'High-Flux (Wide)': 'X'           # X 표시
}

# =================================================================
# 8. 초고해상도 산점도 함수
# =================================================================
def draw_strict_frontier(df, title, filename):
    if len(df) == 0:
        print(f"⚠️ {title} 데이터가 없어 건너뜁니다.")
        return

    plt.figure(figsize=(16.8, 10.8))
    
    # 경고 방지를 위해 현재 존재하는 style만 필터링
    current_styles = df['Filter_Type'].unique()
    current_style_order = [s for s in style_order if s in current_styles]
    
    sns.scatterplot(
        data=df, 
        x='CO2_Affinity', 
        y='Selectivity', 
        hue='Chemistry', 
        style='Filter_Type',
        hue_order=unique_chems, 
        style_order=current_style_order,
        markers=custom_markers,  
        s=350,            
        alpha=0.85,      
        edgecolor='black',
        palette=fixed_palette
    )

    plt.xscale('log')
    plt.yscale('log')
    
    # 축 고정 (전체 데이터 베이스 기준)
    plt.xlim(base_df['CO2_Affinity'].min() * 0.5, base_df['CO2_Affinity'].max() * 2)
    plt.ylim(base_df['Selectivity'].min() * 0.5, base_df['Selectivity'].max() * 2)

    plt.title(title, fontsize=24, fontweight='bold', pad=18)
    plt.xlabel('CO$_2$ Affinity (Henry\'s Constant) -> Higher Capacity', fontsize=18)
    plt.ylabel('Selectivity (CO$_2$ / N$_2$) -> Higher Purity', fontsize=18)
    plt.grid(True, which="both", ls="--", alpha=0.4)
    plt.tick_params(labelsize=16)

    plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left', borderaxespad=0., fontsize=14, title='Material Properties', title_fontsize=16)

    plt.tight_layout()
    plt.savefig(filename.replace('.png', '.pdf'), dpi=600, bbox_inches='tight')
    plt.savefig(filename, dpi=600, bbox_inches='tight')
    plt.show()
    plt.close()

# =================================================================
# 9. 출력 실행
# =================================================================
print("📈 1/4. 종합 프론티어 출력 중...")
draw_strict_frontier(base_df, 'MOF CCUS Frontier (Robust 2.0): Integrated Analysis', 'Robust_Combined.png')

print("📈 2/4. Tier 1 전용 출력 중...")
draw_strict_frontier(base_df[base_df['Filter_Type'] == 'Tier 1 (Ultimate Target)'], 'MOF CCUS Frontier: Tier 1 Only', 'Robust_Tier1.png')

print("📈 3/4. High-Flux 전용 출력 중...")
draw_strict_frontier(base_df[base_df['Filter_Type'] == 'High-Flux (Wide)'], 'MOF CCUS Frontier: High-Flux Only', 'Robust_HighFlux.png')

print("📈 4/4. N2-Sieving 전용 출력 중...")
draw_strict_frontier(base_df[base_df['Filter_Type'] == 'N2-Sieving (Narrow)'], 'MOF CCUS Frontier: N2-Sieving Only', 'Robust_N2Sieving.png')

print("🏁 모든 작업이 성공적으로 완료되었습니다!")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np
import ast
import seaborn as sns

# 1. 폰트 및 마이너스 기호 방어
mpl.rcParams['font.family'] = 'sans-serif'
mpl.rcParams['axes.unicode_minus'] = False 

# =================================================================
# 2. [Base 데이터 로드]
# =================================================================
n2_sieving = pd.read_csv("N2_Sieving_Group_63_MOFs.csv")
n2_sieving['Base_Group'] = 'N2-Sieving (Narrow)'

high_flux = pd.read_csv("High_Flux_Bottleneck_52_MOFs.csv")
high_flux['Base_Group'] = 'High-Flux (Wide)'

base_df = pd.concat([high_flux, n2_sieving], ignore_index=True)

# =================================================================
# 3. 파싱 함수 (물리량 및 Zeo++ 필터용)
# =================================================================
def get_mofid(id_data):
    try:
        d = ast.literal_eval(id_data) if isinstance(id_data, str) else id_data
        return d.get('mofid-v1', str(id_data))
    except: return str(id_data)

def get_short_id(mofid_str):
    try:
        if ';' in mofid_str:
            return mofid_str.split(';')[1][:6].upper()
        return 'XXXXXX'
    except: return 'XXXXXX'

def get_metrics(gemc_data):
    try:
        d = ast.literal_eval(gemc_data) if isinstance(gemc_data, str) else gemc_data
        widom = d.get('Widom', [])
        if len(widom) >= 2:
            return pd.Series([widom[0], widom[0]/widom[1] if widom[1]>0 else 0])
    except: pass
    return pd.Series([np.nan, np.nan])

def get_chem(metal_data):
    try:
        d = ast.literal_eval(metal_data) if isinstance(metal_data, str) else metal_data
        return f"{d.get('metal_type', 'Unknown')} (OMS: {d.get('has_OMS', 'Unknown')})"
    except: pass
    return "Unknown"

def get_zeopp_filters(zeopp_str):
    try:
        d = ast.literal_eval(zeopp_str) if isinstance(zeopp_str, str) else zeopp_str
        dim = d.get('dimension', 0)
        pld = d.get('PLD', 0)
        vf = d.get('VF', 0)
        return pd.Series([dim, pld, vf])
    except:
        return pd.Series([0, 0, 0])

print("🔄 Base 데이터 기준 물리량 100% 동기화 중...")

base_df['MOF_ID'] = base_df['id'].apply(get_mofid)
base_df['Short_ID'] = base_df['MOF_ID'].apply(get_short_id)
base_df[['CO2_Affinity', 'Selectivity']] = base_df['GEMC_data'].apply(get_metrics)
base_df['Chemistry'] = base_df['metal'].apply(get_chem)
base_df[['Dimension', 'PLD', 'VF']] = base_df['Zeopp'].apply(get_zeopp_filters)

# 기본 결측치 및 오류 뼈대 제거
base_df = base_df.dropna(subset=['CO2_Affinity', 'Selectivity'])
base_df = base_df[base_df['Chemistry'] != 'Unknown (OMS: Unknown)']
base_df = base_df.drop_duplicates(subset=['MOF_ID'], keep='first')

print(f"🔹 1차 정제 후 데이터: {len(base_df)}개")

# =================================================================
# 🌟 4. [신규] 위양성(False Positive) 하드 필터링
# =================================================================
# 1D/2D 밀집상 제거 (dimension == 3) & 최소 확산 직경 보장 (PLD >= 3.4) & 최소 기공 보장 (VF >= 0.2)
base_df = base_df[(base_df['Dimension'] == 3) & (base_df['PLD'] >= 3.4) & (base_df['VF'] >= 0.2)].copy()
print(f"🛡️ 위양성(1D/2D 등) 필터링 후 진성 3D 다공성 타겟: {len(base_df)}개 생존!")

# -------------------------------------------------------------------
# [추가할 파싱 함수] 비열 및 내습성 데이터 추출
# -------------------------------------------------------------------
def get_cp_and_water(row):
    try:
        cp = float(row.get('heat_capacity', np.nan))
        water = str(row.get('water_class', 'unknown')).lower()
        
        # water_class가 명시적이지 않으면 stability dict에서 파싱 시도
        if water == 'unknown' or water == 'nan':
            stab_data = row.get('stability', '{}')
            stab_dict = ast.literal_eval(stab_data) if isinstance(stab_data, str) else stab_data
            water = str(stab_dict.get('water', 'unknown')).lower()
            
        return pd.Series([cp, water])
    except:
        return pd.Series([np.nan, 'unknown'])

print("💧 비열(Cp) 및 내습성(Water Stability) 데이터 동기화 중...")
base_df[['Cp', 'Water_Class']] = base_df.apply(get_cp_and_water, axis=1)

# -------------------------------------------------------------------
# [수정된 핵심 로직] 다차원 실전 스코어링 (Practical Robust Scoring)
# -------------------------------------------------------------------
def practical_scoring(df, weight_sel=0.4, weight_cap=0.4, weight_cp=0.2):
    # 1. 아웃라이어 클리핑
    sel_cap = df['Selectivity'].quantile(0.95)
    df['Sel_Clipped'] = df['Selectivity'].clip(upper=sel_cap)
    
    # 2. [신규] 초고친화력 패널티 적용 (Gaussian-like Peak)
    # 친화력이 너무 높으면 탈착 불가로 간주하여 점수 감소 (예: 상위 80% 지점을 Sweet Spot으로 설정)
    optimal_affinity = df['CO2_Affinity'].quantile(0.80)
    # Sweet spot과의 거리에 따른 가우시안 감쇠 적용
    df['Cap_Penalty_Score'] = np.exp(-0.5 * ((df['CO2_Affinity'] - optimal_affinity) / (optimal_affinity * 0.5))**2)
    
    # 3. [신규] 비열(재생 에너지) 점수화 (낮을수록 좋음 -> 역수 취하기)
    # Cp값이 결측치면 중간값으로 대치
    median_cp = df['Cp'].median()
    df['Cp'] = df['Cp'].fillna(median_cp)
    df['Cp_Score'] = 1 / df['Cp']
    
    # 4. 각 항목 Min-Max 정규화 (0~100점)
    def normalize(series):
        return (series - series.min()) / (series.max() - series.min()) * 100

    df['Sel_Norm'] = normalize(df['Sel_Clipped'])
    df['Cap_Norm'] = normalize(df['Cap_Penalty_Score'])
    df['Cp_Norm'] = normalize(df['Cp_Score'])
    
    # 5. [신규] 내습성 기반 하드 패널티 적용 (Multiplier)
    # 약함(weak) = 0.2 (80% 점수 삭감), 튼튼함(rob) = 1.0, 모름 = 0.7
    conditions = [
        df['Water_Class'].str.contains('rob', na=False),
        df['Water_Class'].str.contains('weak', na=False)
    ]
    choices = [1.0, 0.2]
    df['Water_Multiplier'] = np.select(conditions, choices, default=0.7)
    
    # 6. 최종 종합 수식 계산
    df['Final_Score'] = (
        (df['Sel_Norm'] * weight_sel) + 
        (df['Cap_Norm'] * weight_cap) + 
        (df['Cp_Norm'] * weight_cp)
    ) * df['Water_Multiplier']
    
    # 랭킹 부여
    df = df.sort_values(by='Final_Score', ascending=False).reset_index(drop=True)
    df['Rank'] = np.arange(1, len(df) + 1)
    
    return df

base_df = practical_scoring(base_df)
# =================================================================
# 6. [완벽한 분기 제어] Tier 1은 물리량 무시! 오직 '명단'으로만 사용
# =================================================================
try:
    tier1 = pd.read_csv("Tier1_Ultimate_Targets_ColorSynced.csv")
    tier1['MOF_ID'] = tier1['id'].apply(get_mofid)
    
    vip_list = tier1['MOF_ID'].unique()
    
    base_df['Filter_Type'] = base_df['Base_Group']
    
    # VIP 명단에 있는 MOF는 마커 속성만 'Tier 1'로 덮어씌움
    base_df.loc[base_df['MOF_ID'].isin(vip_list), 'Filter_Type'] = 'Tier 1 (Ultimate Target)'
    
    print(f"✅ Tier 1 분기 제어 성공!")
except Exception as e:
    print(f"⚠️ Tier 1 파일 로드 오류: {e}")

# =================================================================
# 7. 스타일 및 마커 강제 고정
# =================================================================
unique_chems = sorted(base_df['Chemistry'].unique())
fixed_palette = dict(zip(unique_chems, sns.color_palette('tab20', len(unique_chems))))

style_order = ['Tier 1 (Ultimate Target)', 'N2-Sieving (Narrow)', 'High-Flux (Wide)']
custom_markers = {
    'Tier 1 (Ultimate Target)': 's',  # 네모
    'N2-Sieving (Narrow)': 'o',       # 동그라미
    'High-Flux (Wide)': 'X'           # X 표시
}

# =================================================================
# 8. 초고해상도 산점도 함수
# =================================================================
def draw_strict_frontier(df, title, filename):
    if len(df) == 0:
        print(f"⚠️ {title} 데이터가 없어 건너뜁니다.")
        return

    plt.figure(figsize=(16.8, 10.8))
    
    # 경고 방지를 위해 현재 존재하는 style만 필터링
    current_styles = df['Filter_Type'].unique()
    current_style_order = [s for s in style_order if s in current_styles]
    
    sns.scatterplot(
        data=df, 
        x='CO2_Affinity', 
        y='Selectivity', 
        hue='Chemistry', 
        style='Filter_Type',
        hue_order=unique_chems, 
        style_order=current_style_order,
        markers=custom_markers,  
        s=350,            
        alpha=0.85,      
        edgecolor='black',
        palette=fixed_palette
    )

    plt.xscale('log')
    plt.yscale('log')
    
    # 축 고정 (전체 데이터 베이스 기준)
    plt.xlim(base_df['CO2_Affinity'].min() * 0.5, base_df['CO2_Affinity'].max() * 2)
    plt.ylim(base_df['Selectivity'].min() * 0.5, base_df['Selectivity'].max() * 2)

    plt.title(title, fontsize=24, fontweight='bold', pad=18)
    plt.xlabel('CO$_2$ Affinity (Henry\'s Constant) -> Higher Capacity', fontsize=18)
    plt.ylabel('Selectivity (CO$_2$ / N$_2$) -> Higher Purity', fontsize=18)
    plt.grid(True, which="both", ls="--", alpha=0.4)
    plt.tick_params(labelsize=16)

    plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left', borderaxespad=0., fontsize=14, title='Material Properties', title_fontsize=16)

    plt.tight_layout()
    plt.savefig(filename.replace('.png', '.pdf'), dpi=600, bbox_inches='tight')
    plt.savefig(filename, dpi=600, bbox_inches='tight')
    plt.show()
    plt.close()

# =================================================================
# 9. 출력 실행
# =================================================================
print("📈 1/4. 종합 프론티어 출력 중...")
draw_strict_frontier(base_df, 'MOF CCUS Frontier (Robust 2.0): Integrated Analysis', 'Robust_Combined.png')

print("📈 2/4. Tier 1 전용 출력 중...")
draw_strict_frontier(base_df[base_df['Filter_Type'] == 'Tier 1 (Ultimate Target)'], 'MOF CCUS Frontier: Tier 1 Only', 'Robust_Tier1.png')

print("📈 3/4. High-Flux 전용 출력 중...")
draw_strict_frontier(base_df[base_df['Filter_Type'] == 'High-Flux (Wide)'], 'MOF CCUS Frontier: High-Flux Only', 'Robust_HighFlux.png')

print("📈 4/4. N2-Sieving 전용 출력 중...")
draw_strict_frontier(base_df[base_df['Filter_Type'] == 'N2-Sieving (Narrow)'], 'MOF CCUS Frontier: N2-Sieving Only', 'Robust_N2Sieving.png')

print("🏁 모든 작업이 성공적으로 완료되었습니다!")

In [ ]:
import json
import numpy as np
import pandas as pd
from scipy.optimize import curve_fit
import warnings

# 최적화 과정에서 발생하는 자잘한 경고 숨김
warnings.filterwarnings('ignore')

# =================================================================
# 1. Sips (Langmuir-Freundlich) 등온흡착 방정식 정의
# =================================================================
def sips_equation(P, q_max, K, n):
    """
    P: 압력 (Pa)
    q_max: 최대 포화 흡착량 (기공 부피 한계)
    K: 랭뮤어 친화력 상수
    n: 상호작용 지수 (n이 1에서 벗어날수록 분자 간 인력/척력 등 비선형성 강함)
    """
    numerator = (K * P)**n
    return q_max * (numerator / (1 + numerator))

# =================================================================
# 2. JSON 데이터 로드 및 비선형 피팅 파이프라인
# =================================================================
file_path = "CR_meta_data_SI.json" # 실제 파일 경로에 맞게 수정
print(f"🔄 [{file_path}] 기반 Sips 피팅 및 극단적 재료 스크리닝을 시작합니다...\n")

with open(file_path, 'r') as f:
    raw_data = json.load(f)

results = []

for mof_id, data in raw_data.items():
    try:
        # --- [A] 기본 파라미터 추출 ---
        # 실제 JSON 구조에 맞게 키(Key) 값은 조정이 필요할 수 있습니다.
        pv = data.get("Zeopp", {}).get("PV", 0)
        widom_k = data.get("GEMC_data", {}).get("Widom", [0.001])[0] 
        
        # 구르비치 한계를 초기 q_max 추정치(Guess)로 사용 (액체 밀도 계수 약 23.6 적용)
        q_max_guess = (pv / 1000) * 23.6 if pv > 0 else 5.0
        
        # --- [B] 5개의 압력 포인트 실측치 추출 ---
        # JSON 내에 기록된 0.1, 1.0, 10.0, 100.0, 1000.0 Pa의 흡착량 배열
        # (데이터 구조에 따라 파싱 로직 조정 필요)
        # 예시: data['GEMC_data']['GEMC'] 배열에서 파싱했다고 가정
        P_obs = np.array([0.1, 1.0, 10.0, 100.0, 1000.0])
        # JSON 구조에 맞게 실측 q 배열을 가져와야 합니다. 여기서는 임시 변수로 처리합니다.
        # q_obs = np.array([q_0.1, q_1.0, q_10, q_100, q_1000]) 
        
        # ⚠️ [주의] 아래는 코드가 작동하도록 임의의 배열을 넣은 것입니다. 
        # 실제 적용 시 JSON에서 파싱한 리스트로 반드시 교체하십시오.
        q_obs = np.array([widom_k*0.1, widom_k*1.0, widom_k*15, widom_k*130, widom_k*1900])
        
        # --- [C] Scipy curve_fit을 이용한 기계학습(피팅) ---
        # 초기 추정치: [q_max, K, n] = [구르비치 용량, Widom 친화력, 1.0(Langmuir 가정)]
        p0 = [q_max_guess, widom_k, 1.0]
        # 하한 및 상한 설정 (K와 n은 음수가 될 수 없음, n은 보통 0.1~3.0 사이)
        bounds = (0, [np.inf, np.inf, 5.0])
        
        popt, pcov = curve_fit(sips_equation, P_obs, q_obs, p0=p0, bounds=bounds, maxfev=5000)
        
        fitted_q_max = popt[0]
        fitted_K = popt[1]
        fitted_n = popt[2]
        
        # --- [D] 공정 타겟 압력 외삽 (Extrapolation) ---
        # 피팅된 그 물질만의 고유한 수식에 DAC 및 배가스 조건을 대입
        cap_dac = sips_equation(260, fitted_q_max, fitted_K, fitted_n)        # 0.0026 bar (260 Pa)
        cap_flue = sips_equation(15000, fitted_q_max, fitted_K, fitted_n)     # 0.15 bar (15000 Pa)
        cap_regen = sips_equation(1000, fitted_q_max, fitted_K, fitted_n)     # 0.01 bar (1000 Pa)
        
        working_cap = cap_flue - cap_regen
        
        # --- [E] 메타데이터 (선택도, 내습성 등 - JSON에 없으면 CSV에서 Merge 요망) ---
        selectivity = data.get("GEMC_data", {}).get("Selectivity", 0) # 임시 파싱
        water_class = "rob" # JSON에 없다면 기본값, 추후 CSV와 병합
        
        results.append({
            "MOF_ID": mof_id,
            "Fitted_q_max": fitted_q_max,
            "Fitted_K": fitted_K,
            "Interaction_n": fitted_n,
            "Cap_DAC_0.0026bar": cap_dac,
            "Working_Cap_0.15bar": working_cap,
            "Selectivity": selectivity,
            "Water_Class": water_class
        })
        
    except Exception as e:
        # 5개 포인트의 거동이 너무 기형적이라 피팅이 실패하는 물질은 제외 (또는 별도 수동 검증군으로 분류)
        pass

df_sips = pd.DataFrame(results)

# =================================================================
# 3. Extremes Approach: 극단적 물성 스크리닝 보드
# =================================================================

print("🏆 [Sips Fitting 기반 극단적 물성 스크리닝 결과]\n")

# 1. 극단적 친화력 (Extreme Affinity for DAC)
# 상호작용 지수(n)나 워킹 카파시티 무시, 오직 0.0026 bar에서의 절대 포집량 1티어
extreme_affinity = df_sips.sort_values(by="Cap_DAC_0.0026bar", ascending=False).head(5)
print("🔥 [Group 1] 극단적 저압 친화력 (DAC 타겟 - 0.0026 bar 용량 기준)")
print(extreme_affinity[["MOF_ID", "Cap_DAC_0.0026bar", "Fitted_K", "Interaction_n"]].to_string(index=False))
print("-" * 70)

# 2. 극단적 선택도 (Extreme Selectivity)
# 용량은 작더라도 CO2/N2 선택도가 비정상적으로 높은 동역학적 분자 체 타겟
extreme_selectivity = df_sips.sort_values(by="Selectivity", ascending=False).head(5)
print("🎯 [Group 2] 극단적 선택도 (혼합 가스 분리막 타겟)")
print(extreme_selectivity[["MOF_ID", "Selectivity", "Working_Cap_0.15bar", "Interaction_n"]].to_string(index=False))
print("-" * 70)

# 3. 극단적 워킹 카파시티 (Extreme Working Capacity)
# 배가스(0.15 bar)와 진공(0.01 bar) 사이의 유효 탈부착량이 가장 거대한 스펀지 타겟
extreme_capacity = df_sips.sort_values(by="Working_Cap_0.15bar", ascending=False).head(5)
print("🏭 [Group 3] 극단적 워킹 카파시티 (일반 배가스 대량 포집 타겟)")
print(extreme_capacity[["MOF_ID", "Working_Cap_0.15bar", "Fitted_q_max", "Interaction_n"]].to_string(index=False))
print("-" * 70)

print("\n✅ Sips 피팅 완료. (Interaction_n 값이 1에서 크게 벗어난 물질은 분자 간 상호작용이 강한 특이 재료입니다.)")

In [ ]:
import pandas as pd
import numpy as np
import ast
import warnings
warnings.filterwarnings('ignore')

# =================================================================
# 1. Base 데이터 로드
# =================================================================
n2_sieving = pd.read_csv("N2_Sieving_Group_63_MOFs.csv")
high_flux = pd.read_csv("Target_Group_2_HighCapacity.csv")
base_df = pd.concat([high_flux, n2_sieving], ignore_index=True)

# =================================================================
# 2. 파싱 함수 모음
# =================================================================
def get_mofid(id_data):
    try:
        d = ast.literal_eval(id_data) if isinstance(id_data, str) else id_data
        return d.get('mofid-v1', str(id_data))
    except: return str(id_data)

def get_short_id(mofid_str):
    try:
        if ';' in mofid_str: return mofid_str.split(';')[1][:6].upper()
        return 'XXXXXX'
    except: return 'XXXXXX'

def get_metrics(gemc_data):
    try:
        d = ast.literal_eval(gemc_data) if isinstance(gemc_data, str) else gemc_data
        widom = d.get('Widom', [])
        if len(widom) >= 2: return pd.Series([widom[0], widom[0]/widom[1] if widom[1]>0 else 0])
    except: pass
    return pd.Series([np.nan, np.nan])

def get_zeopp_filters(zeopp_str):
    try:
        d = ast.literal_eval(zeopp_str) if isinstance(zeopp_str, str) else zeopp_str
        # 기압별 용량 계산을 위해 PV(Pore Volume) 추가 추출
        return pd.Series([d.get('dimension', 0), d.get('PLD', 0), d.get('VF', 0), d.get('PV', 0)])
    except:
        return pd.Series([0, 0, 0, 0])

# [추가된 파싱 함수] 비열 및 내습성 데이터 추출
def get_cp_and_water(row):
    try:
        cp = float(row.get('heat_capacity', np.nan))
        water = str(row.get('water_class', 'unknown')).lower()
        
        # water_class가 명시적이지 않으면 stability dict에서 파싱 시도
        if water == 'unknown' or water == 'nan':
            stab_data = row.get('stability', '{}')
            stab_dict = ast.literal_eval(stab_data) if isinstance(stab_data, str) else stab_data
            water = str(stab_dict.get('water', 'unknown')).lower()
            
        return pd.Series([cp, water])
    except:
        return pd.Series([np.nan, 'unknown'])

print("🔄 기본 물리량, 비열(Cp) 및 내습성(Water) 데이터 동기화 중...")

base_df['MOF_ID'] = base_df['id'].apply(get_mofid)
base_df['Short_ID'] = base_df['MOF_ID'].apply(get_short_id)
base_df[['CO2_Affinity', 'Selectivity']] = base_df['GEMC_data'].apply(get_metrics)
base_df[['Dimension', 'PLD', 'VF', 'PV']] = base_df['Zeopp'].apply(get_zeopp_filters)
base_df[['Cp', 'Water_Class']] = base_df.apply(get_cp_and_water, axis=1)

# 기초 결측치 및 중복 제거
base_df = base_df.dropna(subset=['CO2_Affinity', 'Selectivity'])
base_df = base_df.drop_duplicates(subset=['MOF_ID'], keep='first')

# =================================================================
# 3. [신규] 특정 기압 조건 Working Capacity 산출 함수 (Pseudo-Langmuir)
# =================================================================
def calculate_working_capacity_at_pressure(df, p_target_bar):
    col_name = f'Capacity_at_{p_target_bar}bar'
    K_H = df['CO2_Affinity'] 
    
    # 구르비치 상한선(q_max) 역산: PV(cm3/kg) -> (cm3/g) * 액체 CO2 밀도(23.6 mmol/cm3)
    CO2_LIQUID_DENSITY = 23.6 
    q_max = (df['PV'] / 1000) * CO2_LIQUID_DENSITY  
    
    # Pseudo-Langmuir 수식: q(P) = (K_H * P) / (1 + (K_H * P / q_max))
    numerator = K_H * p_target_bar
    denominator = 1 + (numerator / q_max)
    df[col_name] = numerator / denominator
    
    # 비다공성(PV=0) 예외 처리
    df.loc[df['PV'] <= 0, col_name] = 0.0
    return df

# 2.6 mbar(DAC 조건) 및 1.0 bar(상압 조건) 용량 산출
base_df = calculate_working_capacity_at_pressure(base_df, p_target_bar=0.0026)
base_df = calculate_working_capacity_at_pressure(base_df, p_target_bar=1.0)

# =================================================================
# 4. 하드 필터링 (위양성 철벽 방어)
# =================================================================
base_df = base_df[(base_df['Dimension'] == 3) & (base_df['PLD'] >= 3.4) & (base_df['VF'] >= 0.2)].copy()
print(f"🛡️ 1D/2D 밀집상 제거 완료. 생존 타겟: {len(base_df)}개")

# =================================================================
# 5. [수정된 핵심 로직] 다차원 실전 스코어링 (Practical Robust Scoring)
# =================================================================
def practical_scoring(df, weight_sel=0.4, weight_cap=0.4, weight_cp=0.2):
    # 1. 아웃라이어 클리핑
    sel_cap = df['Selectivity'].quantile(0.95)
    df['Sel_Clipped'] = df['Selectivity'].clip(upper=sel_cap)
    
    # 2. 초고친화력 패널티 적용 (Gaussian-like Peak)
    optimal_affinity = df['CO2_Affinity'].quantile(0.80)
    df['Cap_Penalty_Score'] = np.exp(-0.5 * ((df['CO2_Affinity'] - optimal_affinity) / (optimal_affinity * 0.5))**2)
    
    # 3. 비열(재생 에너지) 점수화 (결측치 중앙값 대치 후 역수)
    median_cp = df['Cp'].median()
    df['Cp'] = df['Cp'].fillna(median_cp)
    df['Cp_Score'] = 1 / df['Cp']
    
    # 4. 각 항목 Min-Max 정규화 (0~100점)
    def normalize(series):
        if series.max() == series.min(): return series * 0 # 분모 0 방지
        return (series - series.min()) / (series.max() - series.min()) * 100

    df['Sel_Norm'] = normalize(df['Sel_Clipped'])
    df['Cap_Norm'] = normalize(df['Cap_Penalty_Score'])
    df['Cp_Norm'] = normalize(df['Cp_Score'])
    
    # 5. 내습성 기반 하드 패널티 적용 (Multiplier)
    conditions = [
        df['Water_Class'].str.contains('rob', na=False),
        df['Water_Class'].str.contains('weak', na=False)
    ]
    choices = [1.0, 0.2]
    df['Water_Multiplier'] = np.select(conditions, choices, default=0.7)
    
    # 6. 최종 종합 수식 계산
    df['Final_Score'] = (
        (df['Sel_Norm'] * weight_sel) + 
        (df['Cap_Norm'] * weight_cap) + 
        (df['Cp_Norm'] * weight_cp)
    ) * df['Water_Multiplier']
    
    # 랭킹 부여
    df = df.sort_values(by='Final_Score', ascending=False).reset_index(drop=True)
    df['Rank'] = np.arange(1, len(df) + 1)
    
    return df

final_df = practical_scoring(base_df)

# =================================================================
# 6. 결과 출력 및 저장
# =================================================================
view_columns = ['Rank', 'Short_ID', 'Capacity_at_0.0026bar', 'Capacity_at_1.0bar', 'Final_Score', 'Water_Class', 'Cp']
print("\n🏆 [Practical Robust Scoring V3] 최종 Top 10 결과:")
print(final_df[view_columns].head(10).to_string(index=False))

final_df.to_csv("Practical_Ranked_Validation.csv", index=False)
print("\n✅ 모든 연산 완료! 'Practical_Ranked_Validation.csv' 파일이 생성되었습니다.")

In [ ]:
import pandas as pd
import numpy as np
import ast
import warnings
warnings.filterwarnings('ignore')

# =================================================================
# 1. 데이터 로드 (모든 후보군 통합)
# =================================================================
n2_sieving = pd.read_csv("N2_Sieving_Group_63_MOFs.csv")
high_cap = pd.read_csv("Target_Group_2_HighCapacity.csv")
base_df = pd.concat([high_cap, n2_sieving], ignore_index=True)

# =================================================================
# 2. 필수 데이터 파싱 (+ OMS 여부 추가)
# =================================================================
def get_short_id(id_str):
    try:
        d = ast.literal_eval(id_str) if isinstance(id_str, str) else id_str
        mofid = d.get('mofid-v1', str(id_str))
        if ';' in mofid: return mofid.split(';')[1][:6].upper()
    except: pass
    return 'XXXXXX'

def parse_features(row):
    try:
        # 열역학 지표 (Widom 헨리 상수)
        gemc = ast.literal_eval(row['GEMC_data']) if isinstance(row['GEMC_data'], str) else row['GEMC_data']
        widom_kh = gemc.get('Widom', ) if gemc.get('Widom') else 0
        
        # 물리적 한계 지표 (Zeo++)
        zeopp = ast.literal_eval(row['Zeopp']) if isinstance(row['Zeopp'], str) else row['Zeopp']
        dim = zeopp.get('dimension', 0)
        pld = zeopp.get('PLD', 0)
        vf = zeopp.get('VF', 0)
        pv = zeopp.get('PV', 0)
        
        # 화학/기하학 지표 (Metal, Topology, OMS)
        metal_dict = ast.literal_eval(row['metal']) if pd.notna(row['metal']) and isinstance(row['metal'], str) else {}
        metal_type = metal_dict.get('metal_type', 'unknown')
        node_stab = str(row.get('node_stability', 'unknown')).lower()
        
        # 개방형 금속 자리(OMS) 여부 추출
        oms_flag = metal_dict.get('has_OMS', False)
        
        return pd.Series([widom_kh, dim, pld, vf, pv, metal_type, node_stab, oms_flag])
    except:
        return pd.Series([0, 0, 0, 0, 0, 'unknown', 'unknown', False])

base_df = base_df['id'].apply(get_short_id)
base_df] = base_df.apply(parse_features, axis=1)

# =================================================================
# 3. 위양성(False Positive) 및 중복 차단 필터
# =================================================================
df_valid = base_df == 3) & (base_df >= 3.4) & (base_df['VF'] >= 0.2)].copy()

# Pymatgen의 StructureMatcher를 응용한 중복(Duplicate) 제거 로직 적용
df_valid = df_valid.drop_duplicates(subset=)

# (선택) MOFChecker를 통한 기하학적/전하적 오류 필터링 모의 적용
# 비정상적인 결합 원자가, 원자 겹침 등을 걸러내어 계산 가능한(CR) 구조만 남김
# df_valid = df_valid[df_valid['MOFChecker_Valid'] == True]

# =================================================================
# 4. [핵심] 토폴로지 & 금속 기반 내습성 교정 및 OMS 취약성 반영
# =================================================================
def correct_water_stability(row):
    metal = str(row['Metal'])
    topo = str(row)
    has_oms = row
    
    # 1. 화학적 방어 (강력한 금속-산소 결합: Hf 추가)
    robust_metals =
    # 2. 기하학적 방어 (12-c, 8-c, 차폐형 4-c, 입체 장력망: csq, ftw 등 추가)
    robust_topos = ['fcu', 'scu', 'sod', 'rob', 'csq', 'ftw']
    
    # OMS가 존재하면 이산화탄소 흡착에는 유리하나, 수분(H2O) 분자에 의한 가수분해 및 경쟁 흡착에 극도로 취약함
    if has_oms and metal not in robust_metals:
        return 'True_Weak'
        
    if metal in robust_metals or any(t in topo for t in robust_topos):
        return 'True_Robust' 
    else:
        return 'True_Weak'

df_valid = df_valid.apply(correct_water_stability, axis=1)

# =================================================================
# 5. DAC 및 Flue gas 조건별 흡착 평가 프로세스 보정
# =================================================================
df_valid['K_H_Norm'] = (df_valid - df_valid.min()) / (df_valid.max() - df_valid.min() + 1e-9) * 100
df_valid['PV_Norm'] = (df_valid['PV'] - df_valid['PV'].min()) / (df_valid['PV'].max() - df_valid['PV'].min() + 1e-9) * 100

def calculate_scores(row):
    kh = row['K_H_Norm']
    pv = row['PV_Norm']
    has_oms = row
    
    # Flue Gas 조건 (약 15% CO2): 기공 부피(작업 용량)와 친화력(선택도)의 균형이 중요
    flue_score = kh * 0.5 + pv * 0.5
    
    # DAC 조건 (400 ppm CO2): 극저농도이므로 결합 에너지(등량 흡착열) 등 강한 친화력이 절대적
    dac_score = kh * 0.8 + pv * 0.2
    
    # OMS 보정: 개방형 금속 자리가 있으면 저압(DAC) 환경에서 CO2 친화력이 폭발적으로 상승함
    if has_oms:
        dac_score *= 1.3 
        
    return pd.Series([flue_score, dac_score])

df_valid] = df_valid.apply(calculate_scores, axis=1)

# =================================================================
# 6. 결과 분리 및 랭킹 부여
# =================================================================
# 그룹 A: 자체 내습성이 완벽하여 Flue Gas 환경에 바로 투입 가능한 소재
df_robust = df_valid == 'True_Robust'].sort_values('FlueGas_Score', ascending=False)

# 그룹 B: DAC 점수는 압도적이나 수분에 무너지는 소재 -> Core-Shell 구조의 'Core'로 사용할 완벽한 타겟
df_weak = df_valid == 'True_Weak'].sort_values('DAC_Score', ascending=False)

view_cols =

print("🏆 [Group A] 자체 내습성 완벽 방어 소재 (Flue Gas 최적화 - True Robust Top 3)")
print(df_robust[view_cols].head(3).to_markdown(index=False))

print("\n⚠️ Core-Shell 코팅이 필수적인 OMS 포함 소재 (DAC 최적화 Core 타겟 - True Weak Top 3)")
print(df_weak[view_cols].head(3).to_markdown(index=False))

In [ ]:
import pandas as pd
import numpy as np
import ast
import warnings
warnings.filterwarnings('ignore')

# =================================================================
# 1. 데이터 로드 (모든 후보군 통합)
# =================================================================
n2_sieving = pd.read_csv("N2_Sieving_Group_63_MOFs.csv")
high_cap = pd.read_csv("Target_Group_2_HighCapacity.csv")
base_df = pd.concat([high_cap, n2_sieving], ignore_index=True)

# =================================================================
# 2. 필수 데이터 파싱 (+ OMS 여부 추가)
# =================================================================
def get_short_id(id_str):
    try:
        d = ast.literal_eval(id_str) if isinstance(id_str, str) else id_str
        mofid = d.get('mofid-v1', str(id_str))
        if ';' in mofid: return mofid.split(';')[1][:6].upper()
    except: pass
    return 'XXXXXX'

def parse_features(row):
    try:
        # 열역학 지표 (Widom 헨리 상수)
        gemc = ast.literal_eval(row['GEMC_data']) if isinstance(row['GEMC_data'], str) else row['GEMC_data']
        widom_kh = gemc.get('Widom', [0])[0] if gemc.get('Widom') else 0
        
        # 물리적 한계 지표 (Zeo++)
        zeopp = ast.literal_eval(row['Zeopp']) if isinstance(row['Zeopp'], str) else row['Zeopp']
        dim = zeopp.get('dimension', 0)
        pld = zeopp.get('PLD', 0)
        vf = zeopp.get('VF', 0)
        pv = zeopp.get('PV', 0)
        
        # 화학/기하학 지표 (Metal, Topology, OMS)
        metal_dict = ast.literal_eval(row['metal']) if pd.notna(row['metal']) and isinstance(row['metal'], str) else {}
        metal_type = metal_dict.get('metal_type', 'unknown')
        node_stab = str(row.get('node_stability', 'unknown')).lower()
        
        # 개방형 금속 자리(OMS) 여부 추출
        oms_flag = metal_dict.get('has_OMS', False)
        
        return pd.Series([widom_kh, dim, pld, vf, pv, metal_type, node_stab, oms_flag])
    except:
        return pd.Series([0, 0, 0, 0, 0, 'unknown', 'unknown', False])

# [수정됨] 컬럼명을 명확히 지정하여 덮어쓰기 방지
base_df['Short_ID'] = base_df['id'].apply(get_short_id)
base_df[['K_H_Widom', 'Dimension', 'PLD', 'VF', 'PV', 'Metal', 'Topology', 'OMS']] = base_df.apply(parse_features, axis=1)

# =================================================================
# 3. 위양성(False Positive) 및 중복 차단 필터
# =================================================================
# [수정됨] 누락되었던 컬럼명('Dimension', 'PLD') 완벽 복원
df_valid = base_df[(base_df['Dimension'] == 3) & (base_df['PLD'] >= 3.4) & (base_df['VF'] >= 0.2)].copy()

# [수정됨] drop_duplicates의 subset 지정
df_valid = df_valid.drop_duplicates(subset=['Short_ID'])

# =================================================================
# 4. [핵심] 토폴로지 & 금속 기반 내습성 교정 및 OMS 취약성 반영
# =================================================================
def correct_water_stability(row):
    metal = str(row['Metal'])
    topo = str(row['Topology'])
    has_oms = row['OMS']
    
    # 1. 화학적 방어 (강력한 금속-산소 결합: Hf 추가)
    robust_metals = ['Zr', 'Ce', 'Ti', 'Hf']
    # 2. 기하학적 방어 (12-c, 8-c, 차폐형 4-c, 입체 장력망: csq, ftw 등 추가)
    robust_topos = ['fcu', 'scu', 'sod', 'rob', 'csq', 'ftw']
    
    # OMS가 존재하면 이산화탄소 흡착에는 유리하나, 수분에 의한 가수분해에 극도로 취약함
    if has_oms and metal not in robust_metals:
        return 'True_Weak'
        
    if metal in robust_metals or any(t in topo for t in robust_topos):
        return 'True_Robust' 
    else:
        return 'True_Weak'

# [수정됨] 새로운 컬럼(Water_Correction)에 결과 할당
df_valid['Water_Correction'] = df_valid.apply(correct_water_stability, axis=1)

# =================================================================
# 5. DAC 및 Flue gas 조건별 흡착 평가 프로세스 보정
# =================================================================
# [수정됨] K_H_Widom 컬럼을 명확히 참조하여 스케일링
df_valid['K_H_Norm'] = (df_valid['K_H_Widom'] - df_valid['K_H_Widom'].min()) / (df_valid['K_H_Widom'].max() - df_valid['K_H_Widom'].min() + 1e-9) * 100
df_valid['PV_Norm'] = (df_valid['PV'] - df_valid['PV'].min()) / (df_valid['PV'].max() - df_valid['PV'].min() + 1e-9) * 100

def calculate_scores(row):
    kh = row['K_H_Norm']
    pv = row['PV_Norm']
    has_oms = row['OMS']
    
    # Flue Gas 조건 (약 15% CO2): 기공 부피(작업 용량)와 친화력의 균형
    flue_score = kh * 0.5 + pv * 0.5
    
    # DAC 조건 (400 ppm CO2): 극저농도이므로 결합 에너지(등량 흡착열) 등 강한 친화력이 절대적
    dac_score = kh * 0.8 + pv * 0.2
    
    # OMS 보정: 개방형 금속 자리가 있으면 저압(DAC) 환경에서 CO2 친화력이 폭발적으로 상승함
    if has_oms:
        dac_score *= 1.3 
        
    return pd.Series([flue_score, dac_score])

# [수정됨] 결과값을 받을 두 개의 컬럼 리스트 지정
df_valid[['FlueGas_Score', 'DAC_Score']] = df_valid.apply(calculate_scores, axis=1)

# =================================================================
# 6. 결과 분리 및 랭킹 부여
# =================================================================
# 그룹 A: 자체 내습성이 완벽하여 Flue Gas 환경에 바로 투입 가능한 소재
# [수정됨] 필터링 조건문 완벽 복원
df_robust = df_valid[df_valid['Water_Correction'] == 'True_Robust'].sort_values('FlueGas_Score', ascending=False)

# 그룹 B: DAC 점수는 압도적이나 수분에 무너지는 소재 -> Core-Shell 등의 코어 타겟
df_weak = df_valid[df_valid['Water_Correction'] == 'True_Weak'].sort_values('DAC_Score', ascending=False)

# [수정됨] 출력할 컬럼 리스트 명시
view_cols = ['Short_ID', 'Metal', 'Topology', 'OMS', 'K_H_Widom', 'PV', 'FlueGas_Score', 'DAC_Score']

print("🏆 [Group A] 자체 내습성 완벽 방어 소재 (Flue Gas 최적화 - True Robust Top 3)")
print(df_robust[view_cols].head(3).to_markdown(index=False))

print("\n⚠️ [Group B] 코어(Core) 타겟용 OMS 포함 소재 (DAC 최적화 - True Weak Top 3)")
print(df_weak[view_cols].head(3).to_markdown(index=False))

In [ ]:
import pandas as pd
import numpy as np
import ast
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# =================================================================
# 1. 데이터 로드
# =================================================================
n2_sieving = pd.read_csv("N2_Sieving_Group_63_MOFs.csv")
high_cap = pd.read_csv("Target_Group_2_HighCapacity.csv")
base_df = pd.concat([high_cap, n2_sieving], ignore_index=True)

# =================================================================
# 2. 필수 데이터 파싱 (Selectivity 복원 완료)
# =================================================================
def get_short_id(id_str):
    try:
        d = ast.literal_eval(id_str) if isinstance(id_str, str) else id_str
        mofid = d.get('mofid-v1', str(id_str))
        if ';' in mofid: return mofid.split(';')[1][:6].upper()
    except: pass
    return 'XXXXXX'

def parse_features(row):
    try:
        gemc = ast.literal_eval(row['GEMC_data']) if isinstance(row['GEMC_data'], str) else row['GEMC_data']
        widom = gemc.get('Widom', [0, 0])
        widom_kh = widom[0] if len(widom) > 0 else 0
        
        # [수정] 원시 데이터의 Widom 분압 비율을 통해 선택도(Selectivity) 산출
        selectivity = widom[0] / widom[1] if len(widom) >= 2 and widom[1] > 0 else 1.0
        
        zeopp = ast.literal_eval(row['Zeopp']) if isinstance(row['Zeopp'], str) else row['Zeopp']
        dim = zeopp.get('dimension', 0)
        pld = zeopp.get('PLD', 0)
        vf = zeopp.get('VF', 0)
        pv = zeopp.get('PV', 0)
        
        metal_dict = ast.literal_eval(row['metal']) if pd.notna(row['metal']) and isinstance(row['metal'], str) else {}
        metal_type = metal_dict.get('metal_type', 'unknown')
        node_stab = str(row.get('node_stability', 'unknown')).lower()
        oms_flag = metal_dict.get('has_OMS', False)
        
        return pd.Series([widom_kh, selectivity, dim, pld, vf, pv, metal_type, node_stab, oms_flag])
    except:
        return pd.Series([0, 1.0, 0, 0, 0, 0, 'unknown', 'unknown', False])

base_df['Short_ID'] = base_df['id'].apply(get_short_id)
base_df[['K_H_Widom', 'Selectivity', 'Dimension', 'PLD', 'VF', 'PV', 'Metal', 'Topology', 'OMS']] = base_df.apply(parse_features, axis=1)

# =================================================================
# 3. 위양성 차단 및 내습성 교정 (Hard Filter & Correction)
# =================================================================
df_valid = base_df[(base_df['Dimension'] == 3) & (base_df['PLD'] >= 3.4) & (base_df['VF'] >= 0.2)].copy()
df_valid = df_valid.drop_duplicates(subset=['Short_ID'])

def correct_water_stability(row):
    metal = str(row['Metal'])
    topo = str(row['Topology'])
    has_oms = row['OMS']
    
    robust_metals = ['Zr', 'Ce', 'Ti', 'Hf']
    robust_topos = ['fcu', 'scu', 'sod', 'rob', 'csq', 'ftw']
    
    if has_oms and metal not in robust_metals:
        return 'True_Weak (DAC Target)'
        
    if metal in robust_metals or any(t in topo for t in robust_topos):
        return 'True_Robust (Flue Gas Target)' 
    else:
        return 'True_Weak (DAC Target)'

df_valid['Target_Group'] = df_valid.apply(correct_water_stability, axis=1)

# 아웃라이어 방어용 로그 스케일링
df_valid['Log_K_H'] = np.log10(df_valid['K_H_Widom'] + 1e-9)
df_valid['Log_Selectivity'] = np.log10(df_valid['Selectivity'] + 1.0)

# =================================================================
# 4. 수정된 물리량 축 기준 시각화 (Affinity vs Selectivity)
# =================================================================
plt.figure(figsize=(11, 8))
sns.set_theme(style="whitegrid")

# X축: 친화력(Log K_H), Y축: 선택도(Log Selectivity) 배치
scatter = sns.scatterplot(
        data=df_valid, 
        x='Log_K_H', y='Log_Selectivity', 
        hue='Target_Group', 
        palette={'True_Robust (Flue Gas Target)': '#2ca02c', 'True_Weak (DAC Target)': '#d62728'},
        style='OMS', markers={True: '^', False: 'o'},
        s=120, alpha=0.75, edgecolor='k'
)

# 최상위 주요 물질 라벨링
top_targets = df_valid.nlargest(5, 'K_H_Widom')
for _, row in top_targets.iterrows():
    plt.annotate(row['Short_ID'], 
                 (row['Log_K_H'], row['Log_Selectivity']),
                 xytext=(6, 6), textcoords='offset points',
                 fontsize=10, fontweight='bold')

plt.title("MOF Mapping: Affinity vs Selectivity Landscape", fontsize=16, fontweight='bold')
plt.xlabel("CO2 Affinity [Log10(K_H_Widom)]", fontsize=12)
plt.ylabel("Selectivity [Log10(Selectivity)]", fontsize=12)

plt.legend(title='Target Group & OMS Status', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.savefig("MOF_Affinity_Selectivity_Landscape.png", dpi=300)
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np
import ast
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# 1. 폰트 및 마이너스 기호 방어
mpl.rcParams['font.family'] = 'sans-serif'
mpl.rcParams['axes.unicode_minus'] = False 

# =================================================================
# 2. [Base 데이터 로드]
# =================================================================
try:
    n2_sieving = pd.read_csv("N2_Sieving_Group_63_MOFs.csv")
    high_flux = pd.read_csv("High_Flux_Bottleneck_52_MOFs.csv")
    base_df = pd.concat([high_flux, n2_sieving], ignore_index=True)
except FileNotFoundError:
    print("⚠️ CSV 파일을 찾을 수 없습니다. 경로를 확인해주세요.")
    base_df = pd.DataFrame()

if not base_df.empty:
    # =================================================================
    # 3. 필수 데이터 파싱 (물리량, 기하학, 화학 속성)
    # =================================================================
    def get_short_id(id_str):
        try:
            d = ast.literal_eval(id_str) if isinstance(id_str, str) else id_str
            mofid = d.get('mofid-v1', str(id_str))
            if ';' in mofid: return mofid.split(';')[1][:6].upper()
        except: pass
        return 'XXXXXX'

    def parse_features(row):
        try:
            # 친화력 및 선택도 (Widom 기반)
            gemc = ast.literal_eval(row['GEMC_data']) if isinstance(row['GEMC_data'], str) else row['GEMC_data']
            widom = gemc.get('Widom', [0, 0])
            widom_kh = widom[0] if len(widom) > 0 else 0
            selectivity = (widom[0] / widom[1]) if len(widom) >= 2 and widom[1] > 0 else 1.0
            
            # 물리적 한계 지표 (Zeo++)
            zeopp = ast.literal_eval(row['Zeopp']) if isinstance(row['Zeopp'], str) else row['Zeopp']
            dim = zeopp.get('dimension', 0)
            pld = zeopp.get('PLD', 0)
            vf = zeopp.get('VF', 0)
            pv = zeopp.get('PV', 0)
            
            # 화학/기하학 지표 (Metal, Topology, OMS)
            metal_dict = ast.literal_eval(row['metal']) if pd.notna(row['metal']) and isinstance(row['metal'], str) else {}
            metal_type = metal_dict.get('metal_type', 'unknown')
            node_stab = str(row.get('node_stability', 'unknown')).lower()
            oms_flag = metal_dict.get('has_OMS', False)
            
            return pd.Series([widom_kh, selectivity, dim, pld, vf, pv, metal_type, node_stab, oms_flag])
        except:
            return pd.Series([0, 1.0, 0, 0, 0, 0, 'unknown', 'unknown', False])

    print("🔄 Base 데이터 기준 물리/화학량 파싱 중...")
    base_df['Short_ID'] = base_df['id'].apply(get_short_id)
    base_df[['CO2_Affinity', 'Selectivity', 'Dimension', 'PLD', 'VF', 'PV', 'Metal', 'Topology', 'OMS']] = base_df.apply(parse_features, axis=1)

    # 기본 결측치 제거
    base_df = base_df[base_df['CO2_Affinity'] > 0]

    # =================================================================
    # 🌟 4. 위양성 하드 필터링 & 토폴로지 내습성 교정 (Robust 2.0)
    # =================================================================
    df_valid = base_df[(base_df['Dimension'] == 3) & (base_df['PLD'] >= 3.4) & (base_df['VF'] >= 0.2)].copy()
    df_valid = df_valid.drop_duplicates(subset=['Short_ID'])
    print(f"🛡️ 1D/2D 병목 제거 후 진성 3D 다공성 타겟: {len(df_valid)}개 생존!")

    def correct_water_stability(row):
        metal = str(row['Metal'])
        topo = str(row['Topology'])
        has_oms = row['OMS']
        
        robust_metals = ['Zr', 'Ce', 'Ti', 'Hf']
        robust_topos = ['fcu', 'scu', 'sod', 'rob', 'csq', 'ftw']
        
        # OMS가 있으나 초강력 금속이 아니면 수분에 무너짐
        if has_oms and metal not in robust_metals:
            return 'True Weak (DAC Target)'
            
        # 강력한 금속이거나 튼튼한 방어 위상(토폴로지)을 가졌다면
        if metal in robust_metals or any(t in topo for t in robust_topos):
            return 'True Robust (Flue Gas Target)' 
        else:
            return 'True Weak (DAC Target)'

    df_valid['Water_Correction'] = df_valid.apply(correct_water_stability, axis=1)

    # =================================================================
    # 5. 아웃라이어 방어 및 스코어링 (Log & Winsorization)
    # =================================================================
    df_valid['Log_K_H'] = np.log10(df_valid['CO2_Affinity'] + 1e-9)
    pv_cap = df_valid['PV'].quantile(0.95)
    df_valid['Clipped_PV'] = df_valid['PV'].clip(upper=pv_cap)
    
    df_valid['K_H_Norm'] = (df_valid['Log_K_H'] - df_valid['Log_K_H'].min()) / (df_valid['Log_K_H'].max() - df_valid['Log_K_H'].min() + 1e-9) * 100
    df_valid['PV_Norm'] = (df_valid['Clipped_PV'] - df_valid['Clipped_PV'].min()) / (df_valid['Clipped_PV'].max() - df_valid['Clipped_PV'].min() + 1e-9) * 100

    def calculate_scores(row):
        kh = row['K_H_Norm']
        pv = row['PV_Norm']
        has_oms = row['OMS']
        
        flue_score = (kh * 0.5) + (pv * 0.5)
        dac_score = (kh * 0.8) + (pv * 0.2)
        if has_oms: dac_score *= 1.3 
            
        return pd.Series([flue_score, dac_score])

    df_valid[['FlueGas_Score', 'DAC_Score']] = df_valid.apply(calculate_scores, axis=1)

    # =================================================================
    # 6. 초고해상도 산점도 함수 (Affinity vs Selectivity)
    # =================================================================
    def draw_strict_frontier(df, title, filename):
        if len(df) == 0: return

        plt.figure(figsize=(14, 10))
        
        # 내습성 그룹에 따른 색상, OMS에 따른 마커 모양 지정
        palette = {'True Robust (Flue Gas Target)': '#2ca02c', 'True Weak (DAC Target)': '#d62728'}
        markers = {True: '^', False: 'o'} # OMS: True(세모), False(동그라미)
        
        sns.scatterplot(
            data=df, 
            x='CO2_Affinity', 
            y='Selectivity', 
            hue='Water_Correction', 
            style='OMS',
            palette=palette,
            markers=markers,  
            s=300,            
            alpha=0.85,      
            edgecolor='black',
            linewidth=1.2
        )

        # Log Scale 적용 (기존 연구자님 세팅 유지)
        plt.xscale('log')
        plt.yscale('log')
        
        # 주요 1티어 타겟 텍스트 라벨링
        top_targets = pd.concat([
            df[df['Water_Correction'].str.contains('Robust')].nlargest(3, 'FlueGas_Score'),
            df[df['Water_Correction'].str.contains('Weak')].nlargest(3, 'DAC_Score')
        ]).drop_duplicates(subset=['Short_ID'])

        for _, row in top_targets.iterrows():
            plt.annotate(row['Short_ID'], 
                         (row['CO2_Affinity'], row['Selectivity']),
                         xytext=(8, 8), textcoords='offset points',
                         fontsize=12, fontweight='bold')

        plt.title(title, fontsize=22, fontweight='bold', pad=18)
        plt.xlabel('CO$_2$ Affinity (Widom $K_H$) -> Higher Capture Force', fontsize=16)
        plt.ylabel('Selectivity (CO$_2$ / N$_2$) -> Higher Purity', fontsize=16)
        plt.grid(True, which="both", ls="--", alpha=0.4)
        plt.tick_params(labelsize=14)

        plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left', borderaxespad=0., fontsize=13, title='Material Properties', title_fontsize=15)
        plt.tight_layout()
        plt.savefig(filename, dpi=600, bbox_inches='tight')
        plt.show()
        plt.close()

    # 출력 실행
    print("📈 평가 스코어 기반 프론티어 맵 출력 중...")
    draw_strict_frontier(df_valid, 'MOF CCUS Frontier (Robust 2.0): Affinity vs Selectivity', 'Robust2.0_Frontier.png')
    print("🏁 모든 작업이 성공적으로 완료되었습니다!")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np
import ast
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# 1. 폰트 및 마이너스 기호 방어
mpl.rcParams['font.family'] = 'sans-serif'
mpl.rcParams['axes.unicode_minus'] = False 

# =================================================================
# 2. [Base 데이터 로드]
# =================================================================
try:
    n2_sieving = pd.read_csv("N2_Sieving_Group_63_MOFs.csv")
    high_flux = pd.read_csv("High_Flux_Bottleneck_52_MOFs.csv")
    base_df = pd.concat([high_flux, n2_sieving], ignore_index=True)
except FileNotFoundError:
    print("⚠️ CSV 파일을 찾을 수 없습니다. 경로를 확인해주세요.")
    base_df = pd.DataFrame()

if not base_df.empty:
    # =================================================================
    # 3. 필수 데이터 파싱 (물리량, 기하학, 화학 속성)
    # =================================================================
    def get_short_id(id_str):
        try:
            d = ast.literal_eval(id_str) if isinstance(id_str, str) else id_str
            mofid = d.get('mofid-v1', str(id_str))
            if ';' in mofid: return mofid.split(';')[1][:6].upper()
        except: pass
        return 'XXXXXX'

    def parse_features(row):
        try:
            # 친화력 및 선택도 (Widom 기반)
            gemc = ast.literal_eval(row['GEMC_data']) if isinstance(row['GEMC_data'], str) else row['GEMC_data']
            widom = gemc.get('Widom', [0, 0])
            widom_kh = widom[0] if len(widom) > 0 else 0
            selectivity = (widom[0] / widom[1]) if len(widom) >= 2 and widom[1] > 0 else 1.0
            
            # 물리적 한계 지표 (Zeo++)
            zeopp = ast.literal_eval(row['Zeopp']) if isinstance(row['Zeopp'], str) else row['Zeopp']
            dim = zeopp.get('dimension', 0)
            pld = zeopp.get('PLD', 0)
            vf = zeopp.get('VF', 0)
            pv = zeopp.get('PV', 0)
            
            # 화학/기하학 지표 (Metal, Topology, OMS)
            metal_dict = ast.literal_eval(row['metal']) if pd.notna(row['metal']) and isinstance(row['metal'], str) else {}
            metal_type = metal_dict.get('metal_type', 'unknown')
            node_stab = str(row.get('node_stability', 'unknown')).lower()
            oms_flag = metal_dict.get('has_OMS', False)
            
            return pd.Series([widom_kh, selectivity, dim, pld, vf, pv, metal_type, node_stab, oms_flag])
        except:
            return pd.Series([0, 1.0, 0, 0, 0, 0, 'unknown', 'unknown', False])

    print("🔄 Base 데이터 기준 물리/화학량 파싱 중...")
    base_df['Short_ID'] = base_df['id'].apply(get_short_id)
    base_df[['CO2_Affinity', 'Selectivity', 'Dimension', 'PLD', 'VF', 'PV', 'Metal', 'Topology', 'OMS']] = base_df.apply(parse_features, axis=1)

    # 기본 결측치 제거
    base_df = base_df[base_df['CO2_Affinity'] > 0]

    # =================================================================
    # 🌟 4. 위양성 하드 필터링 & 토폴로지 내습성 교정 (Robust 2.0)
    # =================================================================
    df_valid = base_df[(base_df['Dimension'] == 3) & (base_df['PLD'] >= 3.4) & (base_df['VF'] >= 0.2)].copy()
    df_valid = df_valid.drop_duplicates(subset=['Short_ID'])
    print(f"🛡️ 1D/2D 병목 제거 후 진성 3D 다공성 타겟: {len(df_valid)}개 생존!")

    def correct_water_stability(row):
        metal = str(row['Metal'])
        topo = str(row['Topology'])
        has_oms = row['OMS']
        
        robust_metals = ['Zr', 'Ce', 'Ti', 'Hf']
        robust_topos = ['fcu', 'scu', 'sod', 'rob', 'csq', 'ftw']
        
        # OMS가 있으나 초강력 금속이 아니면 수분에 무너짐
        if has_oms and metal not in robust_metals:
            return 'True Weak (DAC Target)'
            
        # 강력한 금속이거나 튼튼한 방어 위상(토폴로지)을 가졌다면
        if metal in robust_metals or any(t in topo for t in robust_topos):
            return 'True Robust (Flue Gas Target)' 
        else:
            return 'True Weak (DAC Target)'

    df_valid['Water_Correction'] = df_valid.apply(correct_water_stability, axis=1)

    # =================================================================
    # 5. 아웃라이어 방어 및 스코어링 (Log & Winsorization)
    # =================================================================
    df_valid['Log_K_H'] = np.log10(df_valid['CO2_Affinity'] + 1e-9)
    pv_cap = df_valid['PV'].quantile(0.95)
    df_valid['Clipped_PV'] = df_valid['PV'].clip(upper=pv_cap)
    
    df_valid['K_H_Norm'] = (df_valid['Log_K_H'] - df_valid['Log_K_H'].min()) / (df_valid['Log_K_H'].max() - df_valid['Log_K_H'].min() + 1e-9) * 100
    df_valid['PV_Norm'] = (df_valid['Clipped_PV'] - df_valid['Clipped_PV'].min()) / (df_valid['Clipped_PV'].max() - df_valid['Clipped_PV'].min() + 1e-9) * 100

    def calculate_scores(row):
        kh = row['K_H_Norm']
        pv = row['PV_Norm']
        has_oms = row['OMS']
        
        flue_score = (kh * 0.5) + (pv * 0.5)
        dac_score = (kh * 0.8) + (pv * 0.2)
        if has_oms: dac_score *= 1.3 
            
        return pd.Series([flue_score, dac_score])

    df_valid[['FlueGas_Score', 'DAC_Score']] = df_valid.apply(calculate_scores, axis=1)

    # =================================================================
    # 6. 초고해상도 산점도 함수 (Affinity vs Selectivity)
    # =================================================================
    def draw_strict_frontier(df, title, filename):
        if len(df) == 0: return

        plt.figure(figsize=(14, 10))
        
        # 내습성 그룹에 따른 색상, OMS에 따른 마커 모양 지정
        palette = {'True Robust (Flue Gas Target)': '#2ca02c', 'True Weak (DAC Target)': '#d62728'}
        markers = {True: '^', False: 'o'} # OMS: True(세모), False(동그라미)
        
        sns.scatterplot(
            data=df, 
            x='CO2_Affinity', 
            y='Selectivity', 
            hue='Water_Correction', 
            style='OMS',
            palette=palette,
            markers=markers,  
            s=300,            
            alpha=0.85,      
            edgecolor='black',
            linewidth=1.2
        )

        # Log Scale 적용 (기존 연구자님 세팅 유지)
        plt.xscale('log')
        plt.yscale('log')
        
        # 주요 1티어 타겟 텍스트 라벨링
        top_targets = pd.concat([
            df[df['Water_Correction'].str.contains('Robust')].nlargest(3, 'FlueGas_Score'),
            df[df['Water_Correction'].str.contains('Weak')].nlargest(3, 'DAC_Score')
        ]).drop_duplicates(subset=['Short_ID'])

        for _, row in top_targets.iterrows():
            plt.annotate(row['Short_ID'], 
                         (row['CO2_Affinity'], row['Selectivity']),
                         xytext=(8, 8), textcoords='offset points',
                         fontsize=12, fontweight='bold')

        plt.title(title, fontsize=22, fontweight='bold', pad=18)
        plt.xlabel('CO$_2$ Affinity (Widom $K_H$) -> Higher Capture Force', fontsize=16)
        plt.ylabel('Selectivity (CO$_2$ / N$_2$) -> Higher Purity', fontsize=16)
        plt.grid(True, which="both", ls="--", alpha=0.4)
        plt.tick_params(labelsize=14)

        plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left', borderaxespad=0., fontsize=13, title='Material Properties', title_fontsize=15)
        plt.tight_layout()
        plt.savefig(filename, dpi=600, bbox_inches='tight')
        plt.show()
        plt.close()

    # 출력 실행
    print("📈 평가 스코어 기반 프론티어 맵 출력 중...")
    draw_strict_frontier(df_valid, 'MOF CCUS Frontier (Robust 2.0): Affinity vs Selectivity', 'Robust2.0_Frontier.png')
    print("🏁 모든 작업이 성공적으로 완료되었습니다!")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np
import ast
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# 1. 폰트 및 마이너스 기호 방어 (그래프 깨짐 방지)
mpl.rcParams['font.family'] = 'sans-serif'
mpl.rcParams['axes.unicode_minus'] = False 

# =================================================================
# 2. [Base 데이터 로드]
# =================================================================
try:
    # ⚠️ 연구자님의 로컬 파일 경로/이름에 맞게 수정하세요.
    n2_sieving = pd.read_csv("N2_Sieving_Group_63_MOFs.csv")
    high_flux = pd.read_csv("High_Flux_Bottleneck_52_MOFs.csv")
    base_df = pd.concat([high_flux, n2_sieving], ignore_index=True)
except FileNotFoundError:
    print("⚠️ CSV 파일을 찾을 수 없습니다. 경로를 확인해주세요.")
    base_df = pd.DataFrame()

if not base_df.empty:
    # =================================================================
    # 3. 필수 데이터 파싱 (물리량, 기하학, 화학 속성, OMS 불리언 변환)
    # =================================================================
    def get_short_id(id_str):
        try:
            d = ast.literal_eval(id_str) if isinstance(id_str, str) else id_str
            mofid = d.get('mofid-v1', str(id_str))
            if ';' in mofid: return mofid.split(';')[1][:6].upper()
        except: pass
        return 'XXXXXX'

    def parse_features(row):
        try:
            # 친화력 및 선택도 (Widom 기반)
            gemc = ast.literal_eval(row['GEMC_data']) if isinstance(row['GEMC_data'], str) else row['GEMC_data']
            widom = gemc.get('Widom', [0, 0])
            widom_kh = widom[0] if len(widom) > 0 else 0
            selectivity = (widom[0] / widom[1]) if len(widom) >= 2 and widom[1] > 0 else 1.0
            
            # 물리적 한계 지표 (Zeo++)
            zeopp = ast.literal_eval(row['Zeopp']) if isinstance(row['Zeopp'], str) else row['Zeopp']
            dim = zeopp.get('dimension', 0)
            pld = zeopp.get('PLD', 0)
            vf = zeopp.get('VF', 0)
            pv = zeopp.get('PV', 0)
            
            # 화학/기하학 지표 (Metal, Topology, OMS)
            metal_dict = ast.literal_eval(row['metal']) if pd.notna(row['metal']) and isinstance(row['metal'], str) else {}
            metal_type = metal_dict.get('metal_type', 'unknown')
            node_stab = str(row.get('node_stability', 'unknown')).lower()
            
            # [수정 완결] 'Yes'/'No' 문자열을 파이썬이 인식하는 완벽한 True/False로 강제 변환
            oms_raw = metal_dict.get('has_OMS', False)
            oms_flag = True if str(oms_raw).lower() in ['yes', 'true', '1', 't'] else False
            
            return pd.Series([widom_kh, selectivity, dim, pld, vf, pv, metal_type, node_stab, oms_flag])
        except:
            return pd.Series([0, 1.0, 0, 0, 0, 0, 'unknown', 'unknown', False])

    print("🔄 Base 데이터 기준 물리/화학량 파싱 중...")
    base_df['Short_ID'] = base_df['id'].apply(get_short_id)
    base_df[['CO2_Affinity', 'Selectivity', 'Dimension', 'PLD', 'VF', 'PV', 'Metal', 'Topology', 'OMS']] = base_df.apply(parse_features, axis=1)

    # 친화력이 0 이하인 결측치/오류 데이터 제거
    base_df = base_df[base_df['CO2_Affinity'] > 0]

    # =================================================================
    # 🌟 4. 위양성 하드 필터링 & 토폴로지 내습성 교정 (Robust 2.0)
    # =================================================================
    # 1D/2D 밀집상 및 가스 확산 불가 병목 제거
    df_valid = base_df[(base_df['Dimension'] == 3) & (base_df['PLD'] >= 3.4) & (base_df['VF'] >= 0.2)].copy()
    df_valid = df_valid.drop_duplicates(subset=['Short_ID'])
    print(f"🛡️ 1D/2D 병목 제거 후 진성 3D 다공성 타겟: {len(df_valid)}개 생존!")

    def correct_water_stability(row):
        metal = str(row['Metal'])
        topo = str(row['Topology'])
        has_oms = row['OMS']
        
        # 내습성을 보장하는 강력한 금속 노드 및 기하학적 차폐 위상(Topology)
        robust_metals = ['Zr', 'Ce', 'Ti', 'Hf']
        robust_topos = ['fcu', 'scu', 'sod', 'rob', 'csq', 'ftw']
        
        # OMS가 있으나 초강력 금속이 아니면 수분에 무조건 붕괴됨
        if has_oms and metal not in robust_metals:
            return 'True Weak (DAC Target)'
            
        # 강력한 금속이거나 튼튼한 방어 위상을 가졌다면
        if metal in robust_metals or any(t in topo for t in robust_topos):
            return 'True Robust (Flue Gas Target)' 
        else:
            return 'True Weak (DAC Target)'

    df_valid['Water_Correction'] = df_valid.apply(correct_water_stability, axis=1)

    # =================================================================
    # 5. 아웃라이어 방어 및 스코어링 (Log & Winsorization)
    # =================================================================
    # K_H는 지수적으로 작용하므로 로그 스케일링을 통해 왜곡 방지
    df_valid['Log_K_H'] = np.log10(df_valid['CO2_Affinity'] + 1e-9)
    # 거대 기공 아웃라이어를 막기 위해 상위 95% 값으로 천장(Cap) 설정
    pv_cap = df_valid['PV'].quantile(0.95)
    df_valid['Clipped_PV'] = df_valid['PV'].clip(upper=pv_cap)
    
    # 0~100 스케일 Min-Max 정규화
    df_valid['K_H_Norm'] = (df_valid['Log_K_H'] - df_valid['Log_K_H'].min()) / (df_valid['Log_K_H'].max() - df_valid['Log_K_H'].min() + 1e-9) * 100
    df_valid['PV_Norm'] = (df_valid['Clipped_PV'] - df_valid['Clipped_PV'].min()) / (df_valid['Clipped_PV'].max() - df_valid['Clipped_PV'].min() + 1e-9) * 100

    def calculate_scores(row):
        kh = row['K_H_Norm']
        pv = row['PV_Norm']
        has_oms = row['OMS']
        
        # Flue Gas: CO2 15% 농도. 친화력(5) : 용량(5)
        flue_score = (kh * 0.5) + (pv * 0.5)
        # DAC: CO2 400ppm 극한 저압. 친화력(8) : 용량(2)
        dac_score = (kh * 0.8) + (pv * 0.2)
        
        # OMS가 존재하면 저압(DAC) 환경에서 루이스 산-염기 상호작용으로 폭발적 성능 향상 (1.3배 가중치)
        if has_oms: 
            dac_score *= 1.3 
            
        return pd.Series([flue_score, dac_score])

    df_valid[['FlueGas_Score', 'DAC_Score']] = df_valid.apply(calculate_scores, axis=1)
    df_valid.to_csv("Final_MOF_Evaluations_Robust2.0.csv", index=False)
    print("✅ 데이터 처리 완료. 'Final_MOF_Evaluations_Robust2.0.csv'로 저장되었습니다.")

    # =================================================================
    # 6. 초고해상도 산점도 함수 (Affinity vs Selectivity)
    # =================================================================
    def draw_strict_frontier(df, title, filename):
        if len(df) == 0: return

        plt.figure(figsize=(14, 10))
        sns.set_theme(style="whitegrid")
        
        # 내습성 그룹 색상 및 OMS 상태 마커 모양 지정
        palette = {'True Robust (Flue Gas Target)': '#2ca02c', 'True Weak (DAC Target)': '#d62728'}
        
        # [수정 완결] Seaborn 마커 매핑 오류를 방지하기 위한 이중 안전망 설정
        markers = {True: '^', False: 'o', 'Yes': '^', 'No': 'o', 'unknown': 's'}
        
        sns.scatterplot(
            data=df, 
            x='CO2_Affinity', 
            y='Selectivity', 
            hue='Water_Correction', 
            style='OMS',
            palette=palette,
            markers=markers,  
            s=300,            
            alpha=0.85,      
            edgecolor='black',
            linewidth=1.2
        )

        # Log Scale 적용 (열역학적 스케일 표현)
        plt.xscale('log')
        plt.yscale('log')
        
        # 각 타겟 그룹별 최상위 물질 텍스트 라벨링 (점수 기준 Top 3 추출)
        top_targets = pd.concat([
            df[df['Water_Correction'].str.contains('Robust')].nlargest(3, 'FlueGas_Score'),
            df[df['Water_Correction'].str.contains('Weak')].nlargest(3, 'DAC_Score')
        ]).drop_duplicates(subset=['Short_ID'])

        for _, row in top_targets.iterrows():
            plt.annotate(row['Short_ID'], 
                         (row['CO2_Affinity'], row['Selectivity']),
                         xytext=(8, 8), textcoords='offset points',
                         fontsize=12, fontweight='bold')

        plt.title(title, fontsize=22, fontweight='bold', pad=18)
        plt.xlabel('CO$_2$ Affinity (Widom $K_H$) -> Higher Capture Force', fontsize=16)
        plt.ylabel('Selectivity (CO$_2$ / N$_2$) -> Higher Purity', fontsize=16)
        
        # 그래프 그리드 및 축 폰트 설정
        plt.grid(True, which="both", ls="--", alpha=0.4)
        plt.tick_params(labelsize=14)

        # 범례 위치 조정
        plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left', borderaxespad=0., fontsize=13, title='Material Properties', title_fontsize=15)
        plt.tight_layout()
        
        # 그래프 저장 및 출력
        plt.savefig(filename, dpi=600, bbox_inches='tight')
        plt.show()
        plt.close()

    # 출력 실행
    print("📈 평가 스코어 기반 프론티어 맵 그리기 중...")
    draw_strict_frontier(df_valid, 'MOF CCUS Frontier (Robust 2.0): Affinity vs Selectivity', 'Robust2.0_Frontier.png')
    print("🏁 모든 작업이 성공적으로 완료되었습니다!")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np
import ast
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# 1. 폰트 및 마이너스 기호 방어
mpl.rcParams['font.family'] = 'sans-serif'
mpl.rcParams['axes.unicode_minus'] = False 

# =================================================================
# 2. 데이터 로드 및 파싱 (앞선 완결 로직과 동일)
# =================================================================
try:
    n2_sieving = pd.read_csv("N2_Sieving_Group_63_MOFs.csv")
    high_flux = pd.read_csv("High_Flux_Bottleneck_52_MOFs.csv")
    base_df = pd.concat([high_flux, n2_sieving], ignore_index=True)
except FileNotFoundError:
    print("⚠️ CSV 파일을 찾을 수 없습니다.")
    base_df = pd.DataFrame()

if not base_df.empty:
    def get_short_id(id_str):
        try:
            d = ast.literal_eval(id_str) if isinstance(id_str, str) else id_str
            mofid = d.get('mofid-v1', str(id_str))
            if ';' in mofid: return mofid.split(';')[1][:6].upper()
        except: pass
        return 'XXXXXX'

    def parse_features(row):
        try:
            gemc = ast.literal_eval(row['GEMC_data']) if isinstance(row['GEMC_data'], str) else row['GEMC_data']
            widom = gemc.get('Widom', [0, 0])
            widom_kh = widom[0] if len(widom) > 0 else 0
            selectivity = (widom[0] / widom[1]) if len(widom) >= 2 and widom[1] > 0 else 1.0
            
            zeopp = ast.literal_eval(row['Zeopp']) if isinstance(row['Zeopp'], str) else row['Zeopp']
            dim = zeopp.get('dimension', 0)
            pld = zeopp.get('PLD', 0)
            vf = zeopp.get('VF', 0)
            pv = zeopp.get('PV', 0)
            
            metal_dict = ast.literal_eval(row['metal']) if pd.notna(row['metal']) and isinstance(row['metal'], str) else {}
            metal_type = metal_dict.get('metal_type', 'unknown')
            node_stab = str(row.get('node_stability', 'unknown')).lower()
            
            oms_raw = metal_dict.get('has_OMS', False)
            oms_flag = True if str(oms_raw).lower() in ['yes', 'true', '1', 't'] else False
            
            return pd.Series([widom_kh, selectivity, dim, pld, vf, pv, metal_type, node_stab, oms_flag])
        except:
            return pd.Series([0, 1.0, 0, 0, 0, 0, 'unknown', 'unknown', False])

    base_df['Short_ID'] = base_df['id'].apply(get_short_id)
    base_df[['CO2_Affinity', 'Selectivity', 'Dimension', 'PLD', 'VF', 'PV', 'Metal', 'Topology', 'OMS']] = base_df.apply(parse_features, axis=1)
    base_df = base_df[base_df['CO2_Affinity'] > 0]

    # =================================================================
    # 3. 위양성 차단 및 토폴로지 교정
    # =================================================================
    df_valid = base_df[(base_df['Dimension'] == 3) & (base_df['PLD'] >= 3.4) & (base_df['VF'] >= 0.2)].copy()
    df_valid = df_valid.drop_duplicates(subset=['Short_ID'])

    def correct_water_stability(row):
        metal = str(row['Metal'])
        topo = str(row['Topology'])
        has_oms = row['OMS']
        
        robust_metals = ['Zr', 'Ce', 'Ti', 'Hf']
        robust_topos = ['fcu', 'scu', 'sod', 'rob', 'csq', 'ftw']
        
        if has_oms and metal not in robust_metals: return 'True Weak (DAC Target)'
        if metal in robust_metals or any(t in topo for t in robust_topos): return 'True Robust (Flue Gas Target)' 
        else: return 'True Weak (DAC Target)'

    df_valid['Water_Correction'] = df_valid.apply(correct_water_stability, axis=1)

    # 아웃라이어 방어 및 점수화
    df_valid['Log_K_H'] = np.log10(df_valid['CO2_Affinity'] + 1e-9)
    df_valid['Clipped_PV'] = df_valid['PV'].clip(upper=df_valid['PV'].quantile(0.95))
    df_valid['K_H_Norm'] = (df_valid['Log_K_H'] - df_valid['Log_K_H'].min()) / (df_valid['Log_K_H'].max() - df_valid['Log_K_H'].min() + 1e-9) * 100
    df_valid['PV_Norm'] = (df_valid['Clipped_PV'] - df_valid['Clipped_PV'].min()) / (df_valid['Clipped_PV'].max() - df_valid['Clipped_PV'].min() + 1e-9) * 100

    def calculate_scores(row):
        kh, pv, has_oms = row['K_H_Norm'], row['PV_Norm'], row['OMS']
        flue_score = (kh * 0.5) + (pv * 0.5)
        dac_score = (kh * 0.8) + (pv * 0.2)
        if has_oms: dac_score *= 1.3 
        return pd.Series([flue_score, dac_score])

    df_valid[['FlueGas_Score', 'DAC_Score']] = df_valid.apply(calculate_scores, axis=1)

    # =================================================================
    # 🌟 4. [신규] 금속 매핑 및 듀얼 프론티어 출력 함수
    # =================================================================
    # 금속별 일관된 색상을 위해 전역 팔레트 생성
    unique_metals = sorted([m for m in df_valid['Metal'].unique() if m != 'unknown'])
    metal_palette = dict(zip(unique_metals, sns.color_palette("tab10", n_colors=len(unique_metals))))
    metal_palette['unknown'] = '#B0B0B0' # 알 수 없는 금속은 회색
    
    # 두 그래프의 축 스케일을 완벽히 통일 (시각적 비교를 위함)
    global_xlim = (df_valid['CO2_Affinity'].min() * 0.5, df_valid['CO2_Affinity'].max() * 2.0)
    global_ylim = (df_valid['Selectivity'].min() * 0.5, df_valid['Selectivity'].max() * 2.0)

    def draw_split_frontier(df, title, score_col, filename):
        if len(df) == 0: return
        
        plt.figure(figsize=(12, 9))
        sns.set_theme(style="whitegrid")
        
        markers = {True: '^', False: 'o', 'Yes': '^', 'No': 'o', 'unknown': 's'}
        
        sns.scatterplot(
            data=df, x='CO2_Affinity', y='Selectivity', 
            hue='Metal', style='OMS', palette=metal_palette,
            markers=markers, s=350, alpha=0.85, edgecolor='black', linewidth=1.2
        )

        plt.xscale('log')
        plt.yscale('log')
        plt.xlim(global_xlim)
        plt.ylim(global_ylim)
        
        # 랭킹 Top 3 물질 라벨링
        top_targets = df.nlargest(3, score_col)
        for _, row in top_targets.iterrows():
            plt.annotate(row['Short_ID'], 
                         (row['CO2_Affinity'], row['Selectivity']),
                         xytext=(8, 8), textcoords='offset points',
                         fontsize=13, fontweight='bold')

        plt.title(title, fontsize=20, fontweight='bold', pad=18)
        plt.xlabel('CO$_2$ Affinity (Widom $K_H$)', fontsize=15)
        plt.ylabel('Selectivity (CO$_2$ / N$_2$)', fontsize=15)
        
        plt.grid(True, which="both", ls="--", alpha=0.4)
        plt.tick_params(labelsize=13)
        plt.legend(
            bbox_to_anchor=(1.04, 1), 
            loc='upper left', 
            borderaxespad=0., 
            fontsize=12, 
            title='Metal & OMS', 
            title_fontsize=14,
            labelspacing=1.5,      # 항목 간 상하 간격을 1.5배로 넓힘
            handletextpad=1.2,     # 도형과 텍스트 사이의 좌우 간격을 넓힘
            borderpad=1.0          # 범례 테두리 안쪽 여백 확대
        )
        plt.tight_layout()
        plt.savefig(filename, dpi=600, bbox_inches='tight')
        plt.show()
        plt.close()

    # 데이터 분리 및 두 개의 그래프 각각 출력
    df_dac = df_valid[df_valid['Water_Correction'] == 'True Weak (DAC Target)']
    df_flue = df_valid[df_valid['Water_Correction'] == 'True Robust (Flue Gas Target)']

    print("📈 1/2. DAC 코어 타겟 프론티어 맵 출력 중...")
    draw_split_frontier(df_dac, 'MOF Frontier 1: DAC Targets (Core Materials)', 'DAC_Score', 'Frontier_DAC_Metal.png')
    
    print("📈 2/2. Flue Gas 직투입 타겟 프론티어 맵 출력 중...")
    draw_split_frontier(df_flue, 'MOF Frontier 2: Flue Gas Targets (Robust Materials)', 'FlueGas_Score', 'Frontier_FlueGas_Metal.png')

    print("🏁 모든 그래프 출력이 성공적으로 완료되었습니다!")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np
import ast
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# 1. 폰트 및 마이너스 기호 방어
mpl.rcParams['font.family'] = 'sans-serif'
mpl.rcParams['axes.unicode_minus'] = False 

# =================================================================
# 2. 데이터 로드 및 파싱 (OMS 완벽 처리)
# =================================================================
try:
    n2_sieving = pd.read_csv("N2_Sieving_Group_63_MOFs.csv")
    high_flux = pd.read_csv("High_Flux_Bottleneck_52_MOFs.csv")
    base_df = pd.concat([high_flux, n2_sieving], ignore_index=True)
except FileNotFoundError:
    print("⚠️ CSV 파일을 찾을 수 없습니다.")
    base_df = pd.DataFrame()

if not base_df.empty:
    def get_short_id(id_str):
        try:
            d = ast.literal_eval(id_str) if isinstance(id_str, str) else id_str
            mofid = d.get('mofid-v1', str(id_str))
            if ';' in mofid: return mofid.split(';')[1][:6].upper()
        except: pass
        return 'XXXXXX'

    def parse_features(row):
        try:
            gemc = ast.literal_eval(row['GEMC_data']) if isinstance(row['GEMC_data'], str) else row['GEMC_data']
            widom = gemc.get('Widom', [0, 0])
            widom_kh = widom[0] if len(widom) > 0 else 0
            selectivity = (widom[0] / widom[1]) if len(widom) >= 2 and widom[1] > 0 else 1.0
            
            zeopp = ast.literal_eval(row['Zeopp']) if isinstance(row['Zeopp'], str) else row['Zeopp']
            dim = zeopp.get('dimension', 0)
            pld = zeopp.get('PLD', 0)
            vf = zeopp.get('VF', 0)
            pv = zeopp.get('PV', 0)
            
            metal_dict = ast.literal_eval(row['metal']) if pd.notna(row['metal']) and isinstance(row['metal'], str) else {}
            metal_type = metal_dict.get('metal_type', 'unknown')
            node_stab = str(row.get('node_stability', 'unknown')).lower()
            
            # [안전장치] 문자열을 완벽한 불리언으로 변환
            oms_raw = metal_dict.get('has_OMS', False)
            oms_flag = True if str(oms_raw).lower() in ['yes', 'true', '1', 't'] else False
            
            return pd.Series([widom_kh, selectivity, dim, pld, vf, pv, metal_type, node_stab, oms_flag])
        except:
            return pd.Series([0, 1.0, 0, 0, 0, 0, 'unknown', 'unknown', False])

    base_df['Short_ID'] = base_df['id'].apply(get_short_id)
    base_df[['CO2_Affinity', 'Selectivity', 'Dimension', 'PLD', 'VF', 'PV', 'Metal', 'Topology', 'OMS']] = base_df.apply(parse_features, axis=1)
    base_df = base_df[base_df['CO2_Affinity'] > 0]

    # =================================================================
    # 3. 필터링, 교정 및 점수화
    # =================================================================
    df_valid = base_df[(base_df['Dimension'] == 3) & (base_df['PLD'] >= 3.4) & (base_df['VF'] >= 0.2)].copy()
    df_valid = df_valid.drop_duplicates(subset=['Short_ID'])

    # [수정] 4분할을 위해 내습성(Stability) 속성만 먼저 분리
    def classify_stability(row):
        metal = str(row['Metal'])
        topo = str(row['Topology'])
        has_oms = row['OMS']
        robust_metals = ['Zr', 'Ce', 'Ti', 'Hf']
        robust_topos = ['fcu', 'scu', 'sod', 'rob', 'csq', 'ftw']
        
        if has_oms and metal not in robust_metals: return 'Core'
        if metal in robust_metals or any(t in topo for t in robust_topos): return 'Robust'
        return 'Core'

    df_valid['Stability'] = df_valid.apply(classify_stability, axis=1)

    # 스코어링 (로그 및 윈조라이징 적용)
    df_valid['Log_K_H'] = np.log10(df_valid['CO2_Affinity'] + 1e-9)
    df_valid['Clipped_PV'] = df_valid['PV'].clip(upper=df_valid['PV'].quantile(0.95))
    df_valid['K_H_Norm'] = (df_valid['Log_K_H'] - df_valid['Log_K_H'].min()) / (df_valid['Log_K_H'].max() - df_valid['Log_K_H'].min() + 1e-9) * 100
    df_valid['PV_Norm'] = (df_valid['Clipped_PV'] - df_valid['Clipped_PV'].min()) / (df_valid['Clipped_PV'].max() - df_valid['Clipped_PV'].min() + 1e-9) * 100

    def calculate_scores(row):
        kh, pv, has_oms = row['K_H_Norm'], row['PV_Norm'], row['OMS']
        flue_score = (kh * 0.5) + (pv * 0.5)
        dac_score = (kh * 0.8) + (pv * 0.2)
        if has_oms: dac_score *= 1.3 
        return pd.Series([flue_score, dac_score])

    df_valid[['FlueGas_Score', 'DAC_Score']] = df_valid.apply(calculate_scores, axis=1)
    
    # [수정] 더 높은 점수를 받은 공정을 해당 물질의 '주력 타겟(Best Process)'으로 판정
    df_valid['Best_Process'] = np.where(df_valid['DAC_Score'] >= df_valid['FlueGas_Score'], 'DAC', 'Flue Gas')
    
    # 최종 4그룹 생성: Core-DAC, Core-Flue Gas, Robust-DAC, Robust-Flue Gas
    df_valid['Category'] = df_valid['Stability'] + " - " + df_valid['Best_Process']

    # =================================================================
    # 🌟 4. [신규] 전역 색상 통일 & 4대 그룹 개별 그래프 출력
    # =================================================================
    # 금속(Metal)별 고정 팔레트 생성 (그래프가 바뀌어도 색상은 절대 변하지 않음)
    unique_metals = sorted([m for m in df_valid['Metal'].unique() if m != 'unknown'])
    global_palette = dict(zip(unique_metals, sns.color_palette("tab10", n_colors=len(unique_metals))))
    global_palette['unknown'] = '#B0B0B0' 
    
    # 축 스케일 완벽 고정
    global_xlim = (df_valid['CO2_Affinity'].min() * 0.5, df_valid['CO2_Affinity'].max() * 2.0)
    global_ylim = (df_valid['Selectivity'].min() * 0.5, df_valid['Selectivity'].max() * 2.0)

    def draw_quadrant_frontier(df, title, filename):
        if len(df) == 0: 
            print(f"⚠️ [{title}]에 해당하는 물질이 없어 출력을 건너뜁니다.")
            return
        
        plt.figure(figsize=(12, 9))
        sns.set_theme(style="whitegrid")
        markers = {True: '^', False: 'o', 'Yes': '^', 'No': 'o', 'unknown': 's'}
        
        sns.scatterplot(
            data=df, x='CO2_Affinity', y='Selectivity', 
            hue='Metal', style='OMS', palette=global_palette, hue_order=unique_metals,
            markers=markers, s=350, alpha=0.85, edgecolor='black', linewidth=1.2
        )

        plt.xscale('log')
        plt.yscale('log')
        plt.xlim(global_xlim)
        plt.ylim(global_ylim)
        
        # 각 그래프에서 가장 우수한 물질 3개 이름 표시
        top_targets = df.nlargest(3, 'CO2_Affinity') if 'DAC' in title else df.nlargest(3, 'Selectivity')
        for _, row in top_targets.iterrows():
            plt.annotate(row['Short_ID'], (row['CO2_Affinity'], row['Selectivity']),
                         xytext=(8, 8), textcoords='offset points', fontsize=13, fontweight='bold')

        plt.title(title, fontsize=20, fontweight='bold', pad=18)
        plt.xlabel('CO$_2$ Affinity (Widom $K_H$)', fontsize=15)
        plt.ylabel('Selectivity (CO$_2$ / N$_2$)', fontsize=15)
        
        plt.grid(True, which="both", ls="--", alpha=0.4)
        plt.tick_params(labelsize=13)
        
        # [수정] 범례 간격 넓게 고정
        plt.legend(bbox_to_anchor=(1.04, 1), loc='upper left', title='Metal & OMS', 
                   title_fontsize=14, fontsize=12, labelspacing=1.5, handletextpad=1.2)
        
        plt.tight_layout()
        plt.savefig(filename, dpi=600, bbox_inches='tight')
        plt.show()
        plt.close()

    # 4개 그룹별 그래프 출력
    categories = ['Core - DAC', 'Core - Flue Gas', 'Robust - DAC', 'Robust - Flue Gas']
    for idx, cat in enumerate(categories, 1):
        print(f"📈 {idx}/4. [{cat}] 타겟 프론티어 맵 출력 중...")
        sub_df = df_valid[df_valid['Category'] == cat]
        draw_quadrant_frontier(sub_df, f"MOF Frontier: {cat} Targets", f"Frontier_{cat.replace(' ', '')}.png")

    print("🏁 4대 그룹 독립 그래프 출력이 성공적으로 완료되었습니다!")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np
import ast
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# 1. 폰트 및 마이너스 기호 방어
mpl.rcParams['font.family'] = 'sans-serif'
mpl.rcParams['axes.unicode_minus'] = False 

# =================================================================
# 2. 데이터 로드 및 필수 파싱
# =================================================================
try:
    n2_sieving = pd.read_csv("N2_Sieving_Group_63_MOFs.csv")
    high_flux = pd.read_csv("High_Flux_Bottleneck_52_MOFs.csv")
    base_df = pd.concat([high_flux, n2_sieving], ignore_index=True)
except FileNotFoundError:
    print("⚠️ CSV 파일을 찾을 수 없습니다. (빈 데이터프레임으로 테스트 진행)")
    base_df = pd.DataFrame()

if not base_df.empty:
    def get_short_id(id_str):
        try:
            d = ast.literal_eval(id_str) if isinstance(id_str, str) else id_str
            mofid = d.get('mofid-v1', str(id_str))
            if ';' in mofid: return mofid.split(';')[1][:6].upper()
        except: pass
        return 'XXXXXX'

    def parse_features(row):
        try:
            gemc = ast.literal_eval(row['GEMC_data']) if isinstance(row['GEMC_data'], str) else row['GEMC_data']
            widom = gemc.get('Widom', [0, 0])
            widom_kh = widom[0] if len(widom) > 0 else 0
            selectivity = (widom[0] / widom[1]) if len(widom) >= 2 and widom[1] > 0 else 1.0
            
            zeopp = ast.literal_eval(row['Zeopp']) if isinstance(row['Zeopp'], str) else row['Zeopp']
            dim = zeopp.get('dimension', 0)
            pld = zeopp.get('PLD', 0)
            vf = zeopp.get('VF', 0)
            pv = zeopp.get('PV', 0)
            
            metal_dict = ast.literal_eval(row['metal']) if pd.notna(row['metal']) and isinstance(row['metal'], str) else {}
            metal_type = metal_dict.get('metal_type', 'unknown')
            node_stab = str(row.get('node_stability', 'unknown')).lower()
            
            # OMS 불리언 완벽 파싱
            oms_raw = metal_dict.get('has_OMS', False)
            oms_flag = True if str(oms_raw).lower() in ['yes', 'true', '1', 't'] else False
            
            return pd.Series([widom_kh, selectivity, dim, pld, vf, pv, metal_type, node_stab, oms_flag])
        except:
            return pd.Series([0, 1.0, 0, 0, 0, 0, 'unknown', 'unknown', False])

    base_df['Short_ID'] = base_df['id'].apply(get_short_id)
    base_df[['CO2_Affinity', 'Selectivity', 'Dimension', 'PLD', 'VF', 'PV', 'Metal', 'Topology', 'OMS']] = base_df.apply(parse_features, axis=1)
    base_df = base_df[base_df['CO2_Affinity'] > 0]

    # =================================================================
    # 3. 필터링 및 내습성(Stability) 단일 기준 분리
    # =================================================================
    df_valid = base_df[(base_df['Dimension'] == 3) & (base_df['PLD'] >= 3.4) & (base_df['VF'] >= 0.2)].copy()
    df_valid = df_valid.drop_duplicates(subset=['Short_ID'])

    def classify_stability(row):
        metal = str(row['Metal'])
        topo = str(row['Topology'])
        has_oms = row['OMS']
        robust_metals = ['Zr', 'Ce', 'Ti', 'Hf']
        robust_topos = ['fcu', 'scu', 'sod', 'rob', 'csq', 'ftw']
        
        # 임의의 'Best Process' 컷오프를 삭제하고 오직 내습성(Robust/Core)으로만 구별합니다.
        if has_oms and metal not in robust_metals: return 'Core'
        if metal in robust_metals or any(t in topo for t in robust_topos): return 'Robust'
        return 'Core'

    df_valid['Stability'] = df_valid.apply(classify_stability, axis=1)

    # 스코어링 로직
    df_valid['Log_K_H'] = np.log10(df_valid['CO2_Affinity'] + 1e-9)
    df_valid['Clipped_PV'] = df_valid['PV'].clip(upper=df_valid['PV'].quantile(0.95))
    df_valid['K_H_Norm'] = (df_valid['Log_K_H'] - df_valid['Log_K_H'].min()) / (df_valid['Log_K_H'].max() - df_valid['Log_K_H'].min() + 1e-9) * 100
    df_valid['PV_Norm'] = (df_valid['Clipped_PV'] - df_valid['Clipped_PV'].min()) / (df_valid['Clipped_PV'].max() - df_valid['Clipped_PV'].min() + 1e-9) * 100

    def calculate_scores(row):
        kh, pv, has_oms = row['K_H_Norm'], row['PV_Norm'], row['OMS']
        flue_score = (kh * 0.5) + (pv * 0.5)
        dac_score = (kh * 0.8) + (pv * 0.2)
        if has_oms: dac_score *= 1.3 
        return pd.Series([flue_score, dac_score])

    df_valid[['FlueGas_Score', 'DAC_Score']] = df_valid.apply(calculate_scores, axis=1)
    
    # [신규] 성능 점수표 CSV 저장 로직
    export_cols = ['Short_ID', 'Metal', 'Topology', 'OMS', 'Stability', 'CO2_Affinity', 'Selectivity', 'PV', 'FlueGas_Score', 'DAC_Score']
    df_export = df_valid[export_cols].sort_values(by='DAC_Score', ascending=False)
    df_export.to_csv("MOF_Comprehensive_Performance_Scores.csv", index=False, encoding='utf-8-sig')
    print("✅ 전체 성능 평가표가 'MOF_Comprehensive_Performance_Scores.csv'로 저장되었습니다.")

    # =================================================================
    # 4. 4대 그룹 그래프 개별 출력 (전역 색상, 겹침 방지 범례 반영)
    # =================================================================
    unique_metals = sorted([m for m in df_valid['Metal'].unique() if m != 'unknown'])
    global_palette = dict(zip(unique_metals, sns.color_palette("tab10", n_colors=len(unique_metals))))
    global_palette['unknown'] = '#B0B0B0' 
    
    global_xlim = (df_valid['CO2_Affinity'].min() * 0.5, df_valid['CO2_Affinity'].max() * 2.0)
    global_ylim = (df_valid['Selectivity'].min() * 0.5, df_valid['Selectivity'].max() * 2.0)

    def draw_quadrant_frontier(df, stability_type, process_type, score_col, filename):
        if len(df) == 0: return
        
        plt.figure(figsize=(12, 9))
        sns.set_theme(style="whitegrid")
        markers = {True: '^', False: 'o', 'Yes': '^', 'No': 'o', 'unknown': 's'}
        
        sns.scatterplot(
            data=df, x='CO2_Affinity', y='Selectivity', 
            hue='Metal', style='OMS', palette=global_palette, hue_order=unique_metals,
            markers=markers, s=350, alpha=0.85, edgecolor='black', linewidth=1.2
        )

        plt.xscale('log')
        plt.yscale('log')
        plt.xlim(global_xlim)
        plt.ylim(global_ylim)
        
        # 현재 평가 중인 공정(Process) 점수를 기준으로 가장 우수한 3개 물질에 라벨링
        top_targets = df.nlargest(3, score_col)
        for _, row in top_targets.iterrows():
            plt.annotate(row['Short_ID'], (row['CO2_Affinity'], row['Selectivity']),
                         xytext=(8, 8), textcoords='offset points', fontsize=13, fontweight='bold')

        title = f"MOF Frontier: {stability_type} Materials evaluated for {process_type}"
        plt.title(title, fontsize=20, fontweight='bold', pad=18)
        plt.xlabel('CO$_2$ Affinity (Widom $K_H$)', fontsize=15)
        plt.ylabel('Selectivity (CO$_2$ / N$_2$)', fontsize=15)
        plt.grid(True, which="both", ls="--", alpha=0.4)
        plt.tick_params(labelsize=13)
        
        # [수정 완결] 범례 겹침 방지를 위한 넉넉한 여백 설정
        plt.legend(bbox_to_anchor=(1.04, 1), loc='upper left', title='Metal & OMS', 
                   title_fontsize=14, fontsize=12, labelspacing=1.5, handletextpad=1.2, borderpad=1.0)
        
        plt.tight_layout()
        plt.savefig(filename, dpi=600, bbox_inches='tight')
        plt.show()
        plt.close()

    # 4가지 평가 조합(그래프) 출력
    plot_configs = [
        ('Core', 'DAC', 'DAC_Score', 'Frontier_Core_DAC.png'),
        ('Core', 'Flue Gas', 'FlueGas_Score', 'Frontier_Core_FlueGas.png'),
        ('Robust', 'DAC', 'DAC_Score', 'Frontier_Robust_DAC.png'),
        ('Robust', 'Flue Gas', 'FlueGas_Score', 'Frontier_Robust_FlueGas.png')
    ]
    
    for idx, (stab, proc, score_col, fname) in enumerate(plot_configs, 1):
        print(f"📈 {idx}/4. [{stab} - {proc}] 타겟 프론티어 맵 출력 중...")
        sub_df = df_valid[df_valid['Stability'] == stab]
        draw_quadrant_frontier(sub_df, stab, proc, score_col, fname)

    print("🏁 4대 그룹 개별 그래프 및 성능 CSV 추출이 완벽하게 완료되었습니다!")